# HazardousTrucks Hyperparameter Optimization (HPO) Notebook
Notebook ini mengimplementasikan pencarian hyperparameter terbaik pada model Deteksi Objek (YOLO / RT-DETR) menggunakan 4 Algoritma Optimasi Metaheuristik:
1. **Particle Swarm Optimization (PSO)**
2. **Grey Wolf Optimizer (GWO)**
3. **Whale Optimization Algorithm (WOA)**
4. **Ant Colony Optimization (ACO)**

### Validitas Ilmiah dan Rekonstruksi Fold
Agar hasil perbandingan dengan baseline valid secara akademis, notebook ini **merekonstruksi data split K-Fold** yang sama persis dengan eksperimen baseline referensi (berdasarkan log nama berkas dari baseline).

### Cara Menjalankan
1. Tentukan nama model pada variabel `MODEL_NAME` di sel konfigurasi (misalnya: `'yolov8n.pt'`, `'yolov9t.pt'`, `'yolov10n.pt'`, `'yolo11n.pt'`, `'yolo26n.pt'`, atau `'rtdetr-l.pt'`).
2. Jalankan seluruh sel. Jika menggunakan Google Colab atau Kaggle, pastikan akselerator GPU diaktifkan.

In [ ]:
# @title 1. Install Dependencies & Impor Pustaka
!pip install roboflow ultralytics scikit-learn matplotlib seaborn pandas numpy pyyaml tqdm

import os
import json
import yaml
import shutil
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from tqdm import tqdm
from roboflow import Roboflow
from ultralytics import YOLO, RTDETR

print("Pustaka berhasil diimpor.")

In [ ]:
# @title 2. Konfigurasi Parameter Eksperimen
# Pilih model yang ingin dilatih di akun ini:
# "yolov8n.pt", "yolov9t.pt", "yolov10n.pt", "yolo11n.pt", "yolo26n.pt", "rtdetr-l.pt"
MODEL_NAME = "yolov8n.pt"

# Konfigurasi HPO
NUM_AGENTS = 5        # Disamakan untuk semua model agar adil
NUM_ITERATIONS = 5    # Disamakan untuk semua model agar adil
HPO_EPOCHS = 25       # Jumlah epoch per evaluasi agen HPO
HPO_FOLDS = [0]       # Evaluasi HPO pada Fold 0 untuk menghemat waktu komputasi
FINAL_EPOCHS = 50     # Jumlah epoch untuk training final (sesuai baseline)

# Roboflow Credentials (sesuai konfigurasi dasar)
ROBOFLOW_API_KEY = "NiIS17WWRAdd5FcsTuDj"
ROBOFLOW_WORKSPACE = "test-v9r03"
ROBOFLOW_PROJECT = "hazardoustrucks"
ROBOFLOW_VERSION = 2

# Direktori Output
OUTPUT_DIR = "./hpo_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Deteksi GPU/CPU secara otomatis agar ramah Kaggle/Colab/Lokal
import torch
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"Target Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"HPO Config: {NUM_AGENTS} agents, {NUM_ITERATIONS} iterations, {HPO_EPOCHS} epochs over folds {HPO_FOLDS}")

In [ ]:
# @title 3. Download & Bersihkan Dataset dari Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("coco")

dataset_dir = os.path.abspath("./HazardousTrucks-2")

# Pembersihan dataset (menghapus kelas id 0 'none' dan file truckSafe-70 sesuai alur prapemrosesan)
json_path = f"{dataset_dir}/train/_annotations.coco.json"
image_to_delete = "truckSafe-70-_jpg.rf.a2c93a5b12fcc07fbe0b7a8d7594104d.jpg"
image_path = f"{dataset_dir}/train/{image_to_delete}"

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Hapus kategori id: 0
    data['categories'] = [cat for cat in data['categories'] if cat['id'] != 0]

    # Hapus file dari gambar metadata
    image_id_to_remove = None
    new_images = []
    for img in data['images']:
        if img['file_name'] == image_to_delete:
            image_id_to_remove = img['id']
        else:
            new_images.append(img)
    data['images'] = new_images

    # Hapus anotasinya
    data['annotations'] = [
        ann for ann in data['annotations']
        if ann['image_id'] != image_id_to_remove and ann['category_id'] != 0
    ]

    with open(json_path, 'w') as f:
        json.dump(data, f, indent=4)
    print("File JSON berhasil dibersihkan.")

if os.path.exists(image_path):
    os.remove(image_path)
    print(f"File gambar '{image_to_delete}' berhasil dihapus dari direktori.")

In [ ]:
# @title 4. Injeksi Data Folds Rekonstruksi
# Dictionary nama file gambar per fold untuk merekonstruksi split data baseline agar 100% sama
RECONSTRUCTED_FOLDS = {0: {'train': ['truckUnsafe-1-_jpg.rf.66c30a4c2d51092237f994a461307404.jpg', 'truckSafe-85-_jpg.rf.2f4965dbddf3770cbc1021fcbfa735d2.jpg', 'truckUnsafe-160-_jpg.rf.08ee2d61efd03a95fc5e91677cabee5e.jpg', 'truckUnsafe-7-_jpg.rf.c0d7b79f7935ed57afd0ecb8cdedefb6.jpg', 'truckSafe-59-_jpg.rf.4d6618413c4ca50d80c7d6d80901f8bf.jpg', 'truckUnsafe-10-_jpg.rf.d78d52add836967b3808f43fd6cecb8f.jpg', 'truckUnsafe-193-_jpg.rf.6c0c40e33ffa9be8acab53248760516b.jpg', 'truckUnsafe-197-_jpg.rf.0b4368e6f43d8d46673149674ec4b528.jpg', 'truckUnsafe-205-_jpg.rf.e7ec53759f5b458230488d7e44a49aa1.jpg', 'truckSafe-110-_jpg.rf.f5db6b1c6a2a533ee002291a5f10aeff.jpg', 'truckSafe-150-_jpg.rf.1ed50e629f3d2e845d7cd23d4eb96335.jpg', 'truckSafe-2-_webp.rf.e828b2375655806bb92294d8525dfa63.jpg', 'truckSafe-9-_webp.rf.ec8194f35ab0607566d472c4be9dd874.jpg', 'truckUnsafe-33-_jpg.rf.de3050699444a5b892e0a8c017e86079.jpg', 'truckUnsafe-159-_jpg.rf.5bb7ecbd26dd51e833a8dfd01ee86d10.jpg', 'truckSafe-34-_jpg.rf.f695860cb2d067e30dce85dfef5018cc.jpg', 'truckUnsafe-176-_jpg.rf.9d134c36da2b927a9a28a3f99a819446.jpg', 'truckUnsafe-66-_jpg.rf.27776d206953c93e79c8934473e8565a.jpg', 'truckSafe-134-_jpg.rf.bf97520f126239daa9f788726a60bcb5.jpg', 'truckSafe-45-_jpg.rf.f851c3844c8d13234a67d027ee685500.jpg', 'truckUnsafe-29-_jpg.rf.0136f08138f167488b2b4eeaecacfa99.jpg', 'truckUnsafe-95-_jpg.rf.112c640125660cb2b77411962d0abad4.jpg', 'truckSafe-44-_jpg.rf.8a970f8bf631260f3c96a82cc097d6e8.jpg', 'truckSafe-177-_jpg.rf.7e6424dc96bc2b50c3b05cb72b65ce8b.jpg', 'truckSafe-32-_jpg.rf.de9045b2500ea54d720f3e76a9305433.jpg', 'truckUnsafe-194-_jpg.rf.07119965499d07a9e671f515db2959a5.jpg', 'truckUnsafe-143-_jpg.rf.79d14da101782672022e049e1bdeee89.jpg', 'truckSafe-69-_jpg.rf.85937ed142b1cc9728bb8c8f2d9785e6.jpg', 'truckSafe-178-_jpg.rf.4bed5ae6d363dea263d1dbb66cce89f5.jpg', 'truckUnsafe-177-_jpg.rf.5d4f267583529dca089b41e9f7e97883.jpg', 'truckUnsafe-38-_jpg.rf.48e1108253f0374e5a089513a90e3ff0.jpg', 'truckSafe-35-_jpg.rf.6210c1789743001e32363c0ea38a15e5.jpg', 'truckSafe-96-_jpg.rf.181271efd70cc303a7f94f9e1a01bdb6.jpg', 'truckSafe-60-_jpg.rf.5bf0d644a9329f054b125c78f6db78de.jpg', 'truckUnsafe-129-_jpg.rf.dd4a5fe7b54d4ac7825291e4611313fd.jpg', 'truckSafe-30-_jpg.rf.e5ca2dcd251c8c9395023416c350df6a.jpg', 'truckUnsafe-56-_jpg.rf.dced56bae27edc9d6b1c458653cdd2de.jpg', 'truckSafe-28-_jpg.rf.f7d74f66420f4fe61a6f47ed7e8afd04.jpg', 'truckSafe-12-_jpg.rf.39eb53587cbba23d7368073ae290f8ad.jpg', 'truckSafe-21-_jpg.rf.b599ac3334760a3f82d388857814f39e.jpg', 'truckUnsafe-115-_jpg.rf.9877e3d97ac61c51d93ec8ea7233c80b.jpg', 'truckSafe-148-_jpg.rf.fb5880a0ed45d36edaf38617112e89bf.jpg', 'truckSafe-97-_jpg.rf.d0098bd00a7779e4865439140a6380c1.jpg', 'truckSafe-117-_jpg.rf.f903f0bd99d3b9f73b12dda652c88fb8.jpg', 'truckSafe-25-_jpg.rf.cd090f76ee4de2de30ef91b9fe629c57.jpg', 'truckUnsafe-119-_jpg.rf.672aa2017377b983d01ca9b670ca0e91.jpg', 'truckSafe-181-_jpg.rf.ae0f5f346b1f48b2b28ee67320750f15.jpg', 'truckSafe-8-_jpg.rf.d9de58166f3fc2a27cb9fb70a14f5832.jpg', 'truckUnsafe-36-_jpg.rf.5ed78ccd6dbdaea3187841d0607dfa21.jpg', 'truckSafe-27-_jpg.rf.f6bb32d9ec7a45b384a973ed17090b73.jpg', 'truckUnsafe-44-_jpg.rf.e8dcce3e0ce162ab3c6bd8e4f7eaf5d0.jpg', 'truckUnsafe-85-_jpg.rf.1979c63ca47ceaa8533ccd773fa63214.jpg', 'truckSafe-130-_jpg.rf.b05ad19ba0d647e19a9553b76e2e598d.jpg', 'truckUnsafe-92-_jpg.rf.c0bc0344d40691da7b49ded98c8614f0.jpg', 'truckUnsafe-147-_jpg.rf.a245cb4334e0df8c14e6c9d25d9252f0.jpg', 'truckUnsafe-153-_jpg.rf.7019a8000dd041fc1d6f8d7fe1d8c3c5.jpg', 'truckSafe-65-_jpg.rf.a69a4e6a56701f9244892aab2c69aa5e.jpg', 'truckSafe-51-_jpg.rf.baf14a2919019fdd13b08ed4168a998a.jpg', 'truckSafe-162-_jpg.rf.1061926c8dbf08afb1a4d7eeab58b4c1.jpg', 'truckUnsafe-77-_jpg.rf.d99926b669c1222f8f3b5668ce9d9e7d.jpg', 'truckUnsafe-158-_jpg.rf.2b4978d0d7c31782247ebe044b20260a.jpg', 'truckSafe-29-_jpg.rf.5160f33be04265d12ced4efc8c13d202.jpg', 'truckSafe-19-_jpg.rf.46fc002423b49f86c30c68bd93b4bea2.jpg', 'truckUnsafe-28-_jpg.rf.f03baff0ec0f1661ad9b4dba594a741b.jpg', 'truckSafe-105-_jpg.rf.9867433ae340497a1dff7c7799dbbc4f.jpg', 'truckSafe-26-_jpg.rf.54985421d17fd0a04e187bfabd621ec2.jpg', 'truckUnsafe-163-_jpg.rf.85d049ab8af8b2c86310d5ee9002537d.jpg', 'truckUnsafe-179-_jpg.rf.b0f40909cd7092c96c6a7c0bf22f9a2f.jpg', 'truckSafe-24-_jpg.rf.aae58a00b277e292aa6ebe6a709720c1.jpg', 'truckUnsafe-126-_jpg.rf.0963a81c618f910e514f53fe9626da38.jpg', 'truckUnsafe-104-_jpg.rf.050ff76c09644829e14b6eaf10728d0b.jpg', 'truckUnsafe-21-_jpg.rf.3a060c0d3b53704bd922d3d5a465fc51.jpg', 'truckSafe-73-_jpg.rf.ea654fb67610c7379028bc356f6eaddb.jpg', 'truckSafe-109-_jpg.rf.769b22963f7c9ef3f89d7b69489d2d44.jpg', 'truckSafe-188-_jpg.rf.2be4f01e41c66cbe250bb1f83a45d77c.jpg', 'truckUnsafe-166-_jpg.rf.54016f8d1644b1eae8a6b6f69d590534.jpg', 'truckSafe-127-_jpg.rf.5ff817b579d8b08ae45a1ad16bdcc9ff.jpg', 'truckSafe-145-_jpg.rf.14db44130e285bc48832b4a268154942.jpg', 'truckUnsafe-121-_jpg.rf.ada93505365462e8720693f545f24aa6.jpg', 'truckUnsafe-55-_jpg.rf.b79f03ebb0feb12365e3be86a65b1ea8.jpg', 'truckUnsafe-47-_jpg.rf.4c887da5c757e9d5dfb2dfe245bcb1a1.jpg', 'truckUnsafe-97-_jpg.rf.062184dd5c4304a13ca457c25763a725.jpg', 'truckUnsafe-93-_jpg.rf.5185ffc4e708caf4450d6b50e637f53a.jpg', 'truckSafe-157-_jpg.rf.03771434f2554193b0fbbf476595752d.jpg', 'truckUnsafe-98-_jpg.rf.36d3f90d22be913327d5235c3e197c97.jpg', 'truckUnsafe-24-_jpg.rf.d6a5b64e5c3084a2620a5314a1afa9de.jpg', 'truckUnsafe-114-_jpg.rf.ea89c824deef9111ba53df1e3b358fc6.jpg', 'truckUnsafe-67-_jpg.rf.7ec55fc8a43a9050056ccbaf8be3cca8.jpg', 'truckSafe-38-_jpg.rf.4c37a79e117bf366f74f4f89292e31bb.jpg', 'truckUnsafe-144-_jpg.rf.b1c7133774e19ce0a8037921cfe39646.jpg', 'truckUnsafe-41-_jpg.rf.24167e65d8001dbcec8995250a395740.jpg', 'truckSafe-193-_jpg.rf.0cc0ef8c2b93513d8de1f5a9bbffc43c.jpg', 'truckSafe-106-_jpg.rf.afebeba9f3fa4fcb9573a2c57a524c14.jpg', 'truckUnsafe-43-_jpg.rf.d220226c519c51d7096fdfd4180366fc.jpg', 'truckUnsafe-49-_jpg.rf.8c40763992228923494b3b88975c49f3.jpg', 'truckSafe-5-_jpg.rf.aff0ea198b86b963b316a564abab4c7e.jpg', 'truckUnsafe-48-_jpg.rf.628681efd2a41b93d3ead8d21cb04a3e.jpg', 'truckUnsafe-54-_jpg.rf.e33ad9e7f96d185270f38398fd442c85.jpg', 'truckUnsafe-146-_jpg.rf.1b1dcb3b4b35776ab87be6bbfecfa8d8.jpg', 'truckSafe-100-_jpg.rf.9e19d4331f246a02ab81257bd84e783f.jpg', 'truckSafe-185-_jpg.rf.d48bb5c402f48d46dd5ba2b0645bb54f.jpg', 'truckUnsafe-27-_jpg.rf.b67e4bd8a18953b44880689e93a9f75d.jpg', 'truckUnsafe-199-_jpg.rf.e611104b8652ea2ae107c2f05e7742a2.jpg', 'truckUnsafe-11-_jpg.rf.f0591c57aff5fbe9d96c17b33776d576.jpg', 'truckSafe-3-_webp.rf.ae5b6a2e4cb447cb3eab97d973e07cd4.jpg', 'truckSafe-161-_jpg.rf.828c830a5836bf8462a29b72e3efafa9.jpg', 'truckSafe-133-_jpg.rf.b20fbe959839064c74b03254da73feb4.jpg', 'truckSafe-175-_jpg.rf.1a4bed9b5c41c9ed36b8f15e76375fe4.jpg', 'truckUnsafe-84-_jpg.rf.ac6fbdce8437790b644771cdd9268fdf.jpg', 'truckSafe-66-_jpg.rf.7ccbfd700c1761ff0d798562b9baf2fe.jpg', 'truckUnsafe-91-_jpg.rf.c3e0b0affa5fd5ebd79d8ac93d570d29.jpg', 'truckSafe-7-_webp.rf.316dd3dfb61fcd5424dc6d4ef6ff1d9d.jpg', 'truckUnsafe-37-_jpg.rf.0e48a6d560ca471a29e53aea79a73062.jpg', 'truckSafe-122-_jpg.rf.f1327dc90994c69507adcca64208b256.jpg', 'truckSafe-61-_jpg.rf.e31c686802cfbac5d44558ee1365a2e0.jpg', 'truckUnsafe-64-_jpg.rf.9af27645e83ae4264c19d830674615c4.jpg', 'truckUnsafe-94-_jpg.rf.24071cadc7abd856bbd85a2d237f6379.jpg', 'truckUnsafe-192-_jpg.rf.95ff1338941d488d2a2bdd0ff7fa338e.jpg', 'truckUnsafe-142-_jpg.rf.135d9a65569b31f916bf316e7f151995.jpg', 'truckSafe-169-_jpg.rf.87a8c50d791b94ad907ba8afe7e9d8df.jpg', 'truckUnsafe-165-_jpg.rf.af4191e9509fc76ef6e4298f268a8e8c.jpg', 'truckUnsafe-118-_jpg.rf.b71532e64d2ae7e38092c83aaedc44ea.jpg', 'truckUnsafe-4-_jpg.rf.ea6a0a852f9a431b0e6b9b74cbf2cf99.jpg', 'truckUnsafe-96-_jpg.rf.b9afb7a259a409e4df70272513e5ec19.jpg', 'truckSafe-2-_jpg.rf.c8ceda007375848fe364d9fb95acad70.jpg', 'truckUnsafe-87-_jpg.rf.9580a4daf820d1aba9855dd51536e723.jpg', 'truckSafe-46-_jpg.rf.a38f8664325ecbc40a7e0a4729822caf.jpg', 'truckUnsafe-167-_jpg.rf.326f6ceac9547b80d1cf6513b2e299d8.jpg', 'truckUnsafe-90-_jpg.rf.a4a481d609241403cd343a0497ad1c42.jpg', 'truckSafe-141-_jpg.rf.1d1663f3393709d1ac68a7bab2a5dcc5.jpg', 'truckSafe-84-_jpg.rf.f12222161bc89e93ba1841fe28e6f916.jpg', 'truckSafe-174-_jpg.rf.1f16f54d94437219b942177e4bc0db67.jpg', 'truckSafe-4-_webp.rf.1398d3d92f30abc2102efab6d45f8e7d.jpg', 'truckSafe-7-_jpg.rf.9f79036ac23a53c55a71bbb0dbf8e110.jpg', 'truckUnsafe-157-_jpg.rf.ff8c910ef1bce7bf7eb7695e7e7e2395.jpg', 'truckSafe-158-_jpg.rf.71b34d0312441967683f0c2c1080ac55.jpg', 'truckUnsafe-18-_jpg.rf.a7df91f4a191d49cfb52d239273b9f3b.jpg', 'truckUnsafe-79-_jpg.rf.388712ca31e3c2d9d8fc7134301a9eb5.jpg', 'truckSafe-152-_jpg.rf.850d0eb944f345859e327fe227e2ffd5.jpg', 'truckUnsafe-12-_jpg.rf.8b882577219da7c2a85456de63e31ab1.jpg', 'truckUnsafe-100-_jpg.rf.a34138f18e494e1e2c36308440b028f3.jpg', 'truckUnsafe-111-_jpg.rf.818a3eca4a4cfe36948dcd81c135feb3.jpg', 'truckSafe-166-_jpg.rf.10f57ff7822fac232f026f0d1997be81.jpg', 'truckSafe-114-_jpg.rf.9467fcc3e3d833851f7afc2acd05adca.jpg', 'truckUnsafe-173-_jpg.rf.35096135cdaadfea73dc3dd3a145fc08.jpg', 'truckUnsafe-42-_jpg.rf.6b6e9da1327d7357a6ecd0a70191a597.jpg', 'truckSafe-164-_jpg.rf.fa9aa8d5bb4229652b71f190d1f4d25f.jpg', 'truckUnsafe-201-_jpg.rf.271a90b8de5b6026cde49a6ecb489fa9.jpg', 'truckUnsafe-105-_jpg.rf.d15a089e8cce2eb10cc9215e3b6223c0.jpg', 'truckUnsafe-99-_jpg.rf.5e7d063e8fc2fac56cf4b98005c2a03f.jpg', 'truckUnsafe-103-_jpg.rf.081710aa938cdb75b0e684e3a75f8818.jpg', 'truckSafe-156-_jpg.rf.73c9dd229c5811afa503f92c0ff20424.jpg', 'truckSafe-183-_jpg.rf.eb2027b28dc77eff4c56e3025cbc2653.jpg', 'truckUnsafe-71-_jpg.rf.a604f89995f09072cd98b2bf5eff7664.jpg', 'truckUnsafe-108-_jpg.rf.2ccd21321b603b761d1d0ff0882eaa90.jpg', 'truckUnsafe-151-_jpg.rf.97c5beaf7e14f92154ac594ac10d8ff5.jpg', 'truckUnsafe-191-_jpg.rf.95da8d25b2f7d4ba407f2da19072ce62.jpg', 'truckUnsafe-198-_jpg.rf.5b99f84247c9fc028d168c9ad4f66ec8.jpg', 'truckUnsafe-26-_jpg.rf.7320e4935f5ddfb7ff8f313bc620b5a8.jpg', 'truckSafe-176-_jpg.rf.9193d80068af2848538b375a9ab821e0.jpg', 'truckUnsafe-162-_jpg.rf.1480f80ee9853b314e72c38864c81a3a.jpg', 'truckUnsafe-86-_jpg.rf.d08bb0606fa8a30be8f46a43c11bdc4a.jpg', 'truckSafe-184-_jpg.rf.43245b2aa995677c14325289e8fd1c5b.jpg', 'truckUnsafe-2-_jpg.rf.d8a033434f0bb74a6c3cbadfb339258b.jpg', 'truckSafe-82-_jpg.rf.0c32f3892190cf4557c83d707b16e74b.jpg', 'truckUnsafe-19-_jpg.rf.b579de1709760947387a5d0a192f1801.jpg', 'truckUnsafe-172-_jpg.rf.9205866b23df7f774bf50554f3a669d4.jpg', 'truckUnsafe-53-_jpg.rf.4176b3f2f3881205d285678f6ea71a17.jpg', 'truckUnsafe-137-_jpg.rf.c66321af191a7352b5bd5c07633058dd.jpg', 'truckSafe-173-_jpg.rf.014f1884ca80ad770ca0bfba19fdca49.jpg', 'truckSafe-153-_jpg.rf.84535b234f54b4a811a9fc3630edd9d8.jpg', 'truckUnsafe-140-_jpg.rf.701dad58d868e3c83399023aa1a00765.jpg', 'truckSafe-104-_jpg.rf.d1c05a12238769503eb3d4d131fcbc9e.jpg', 'truckUnsafe-58-_jpg.rf.e3118e0dfb34d6f897354e8661ead737.jpg', 'truckSafe-87-_jpg.rf.fa900c0d3f7e4c9a7667da0a03362765.jpg', 'truckUnsafe-74-_jpg.rf.9460147f5eccc8238ec6794cdcf2c0bd.jpg', 'truckSafe-13-_jpg.rf.05c4b7d4f35bec861860469c6a4897cd.jpg', 'truckUnsafe-130-_jpg.rf.ec114dd5101a292503a6b96cb7918d3d.jpg', 'truckSafe-90-_jpg.rf.59afc6a6554cbdbff546165bbd1d74a2.jpg', 'truckUnsafe-9-_jpg.rf.2f24b55c2436a8c830f604176db8e7bf.jpg', 'truckSafe-37-_jpg.rf.ad92d1aeb3b1c3f08910c1822ce15259.jpg', 'truckUnsafe-1-_webp.rf.83d14e4ad5e491866b755205e8ed1a84.jpg', 'truckSafe-99-_jpg.rf.d6fdd3a54a647a87cf406cd166efa175.jpg', 'truckSafe-17-_jpg.rf.3b732e87d7e125fc8b227f6578adf59e.jpg', 'truckUnsafe-123-_jpg.rf.e5151da7b48173b50cbd5fdcf270460b.jpg', 'truckUnsafe-161-_jpg.rf.f6946afcf62d422f2f859b13ceda1e33.jpg', 'truckSafe-81-_jpg.rf.9d94f916ffef79645f1d090afd393983.jpg', 'truckUnsafe-133-_jpg.rf.3e694cb4e574c24f9569949ce61b39ba.jpg', 'truckSafe-189-_jpg.rf.32e5ded963fdd6234812af2d3cc0f32a.jpg', 'truckUnsafe-135-_jpg.rf.ebb702987d717323caf9023bc74dc21e.jpg', 'truckSafe-191-_jpg.rf.6af016a6449226cdc9e500cc560b58b7.jpg', 'truckUnsafe-141-_jpg.rf.825ba1db5e1614f0a7a1f30845d02b5a.jpg', 'truckUnsafe-128-_jpg.rf.c2068342dd5985126c6d1c29251fd7b2.jpg', 'truckUnsafe-125-_jpg.rf.36ce28ef8b9d9f0ce74335e7797e30ea.jpg', 'truckSafe-103-_jpg.rf.565796b94aa3b9a650551dad5638dd62.jpg', 'truckUnsafe-180-_jpg.rf.bfe074781a458e41ca7bb78ba589d84a.jpg', 'truckSafe-86-_jpg.rf.4ecc7aac28ace212e96d7556749f8c6a.jpg', 'truckSafe-76-_jpg.rf.3982d37199d66499fad89e44876df5a8.jpg', 'truckUnsafe-13-_jpg.rf.e315ed30edd8462b3103071d935b5636.jpg', 'truckUnsafe-102-_jpg.rf.ba3b895eec73c56576c420107c71fc30.jpg', 'truckSafe-58-_jpg.rf.6d7c4081902acae3464a2f527ce1a2b9.jpg', 'truckSafe-33-_jpg.rf.a577ed75fe979e9284eb7c9bf6382e70.jpg', 'truckSafe-107-_jpg.rf.f2d11dc26ef91ab7a39a1b5493c38d30.jpg', 'truckUnsafe-78-_jpg.rf.18ef332f3ab814b6cc78f0dc1a94fc55.jpg', 'truckSafe-64-_jpg.rf.133330c5158fc2e55a7dc2d7feca10b5.jpg', 'truckUnsafe-107-_jpg.rf.babd965d5bc5ae3bffe9c00b0251d2b3.jpg', 'truckSafe-146-_jpg.rf.650600abdaf8b7e0a4cf3c3d6d0482f7.jpg', 'truckSafe-16-_jpg.rf.dd82e0008d1ca38f19a9015d9c1d904d.jpg', 'truckUnsafe-164-_jpg.rf.4bf99daa9fbd4b994d966286be2f7b78.jpg', 'truckSafe-116-_jpg.rf.4eec97da079f01e7f44e22b2203fee89.jpg', 'truckUnsafe-112-_jpg.rf.36928ae59d723b0cd298b437976aceac.jpg', 'truckUnsafe-51-_jpg.rf.b76605bff08c32482c482cab3d355357.jpg', 'truckSafe-10-_jpg.rf.8a9ccf6b54dad10d7d6008315ac72650.jpg', 'truckUnsafe-89-_jpg.rf.a30019a08e5a6bf467932a216fc3f622.jpg', 'truckSafe-40-_jpg.rf.77ec15e052b68d0980bc2e69a14bc27d.jpg', 'truckSafe-3-_jpg.rf.75b6061d720508ec78e98e6ecb6fc39a.jpg', 'truckSafe-118-_jpg.rf.95d277cb7ae1ef4c3680def0797626ae.jpg', 'truckSafe-80-_jpg.rf.983ac820621eb75446c37990b8739001.jpg', 'truckSafe-43-_jpg.rf.8e4d9cdcb79e28695037e6f147092b1b.jpg', 'truckUnsafe-186-_jpg.rf.8a207c733043e64808014528e310d89d.jpg', 'truckUnsafe-122-_jpg.rf.fe6f35b0c37aad5ac962f4189221e0e6.jpg', 'truckUnsafe-106-_jpg.rf.be4ef15ef9ab2e2ce369596c4ae9ad31.jpg', 'truckUnsafe-182-_jpg.rf.f07e5fb6163965727248439dc676c083.jpg', 'truckUnsafe-14-_jpg.rf.67b4993e6194f1ef8a2c62bc7bce52f4.jpg', 'truckSafe-129-_jpg.rf.2cd2112916d88bbdbca81f6c3856b746.jpg', 'truckUnsafe-132-_jpg.rf.6d7b5d74ebbf2a0768884828e594f6b6.jpg', 'truckSafe-6-_webp.rf.a9c48df2e9f7efcfeb3af3d5fb3122c6.jpg', 'truckSafe-49-_jpg.rf.a8aca39572383374c24ba49242b55343.jpg', 'truckSafe-102-_jpg.rf.7753b5b01a12f79096edaaf996ddbc6d.jpg', 'truckUnsafe-5-_jpg.rf.24986876cec09070289cb7de410a4ee2.jpg', 'truckUnsafe-75-_jpg.rf.712a44e5bdde0cfdfcc23136f784634d.jpg', 'truckSafe-137-_jpg.rf.9f2180c1d5d9004edfda5aa71550d460.jpg', 'truckSafe-57-_jpg.rf.884e1204d78fb5ee677f683e8f3d38e0.jpg', 'truckSafe-140-_jpg.rf.f9c34aa7675f6de5982107f8edeb86c3.jpg', 'truckSafe-124-_jpg.rf.f720158b37bb2809f3cd9a829ec31d22.jpg', 'truckSafe-62-_jpg.rf.10ab461a5116c816bd0b5c8421eddbb5.jpg', 'truckUnsafe-124-_jpg.rf.4954171d878114814dc9cfff995f6c39.jpg', 'truckSafe-154-_jpg.rf.edc7e1f835cc24ddd2564d8124cc3bb4.jpg', 'truckSafe-67-_jpg.rf.5d0c427be03cb6cdf0f45411361b502e.jpg', 'truckSafe-91-_jpg.rf.a512bb3290043c460cbfae7113c731dd.jpg', 'truckSafe-71-_jpg.rf.3fc30fe5f23685a0dd8cd5ee87886689.jpg', 'truckSafe-42-_jpg.rf.c0ec22b050facbe92d64613902252be2.jpg', 'truckUnsafe-183-_jpg.rf.94580a51961ee1289b34017fc3b3d38b.jpg', 'truckUnsafe-2-_webp.rf.d0dc6a759c3d04ca8d49451c36c021f3.jpg', 'truckSafe-78-_jpg.rf.81d818a87769c8d811532f8f2bd923a2.jpg', 'truckUnsafe-39-_jpg.rf.a9da71bff29f03d2151a906de369388d.jpg', 'truckUnsafe-113-_jpg.rf.c43fd8559af2e24e15eb90a471d17cef.jpg', 'truckUnsafe-204-_jpg.rf.9493419167eded56b9c2b7872d6e3274.jpg', 'truckSafe-93-_jpg.rf.2ae480e95b8b2cc7fc43609f9cc4a303.jpg', 'truckSafe-187-_jpg.rf.7eb0d289edadf67321837721ea1c6e5d.jpg', 'truckSafe-54-_jpg.rf.c2cbdf3ac479b6c927a15d3edc21e937.jpg', 'truckUnsafe-203-_jpg.rf.6f6dea5a207484867bca181d9966c81d.jpg', 'truckUnsafe-202-_jpg.rf.2b3c827bff086afefc1cca63d7b9b85f.jpg', 'truckUnsafe-101-_jpg.rf.eef663035a8721e04c556ce2ed207eba.jpg', 'truckUnsafe-175-_jpg.rf.49a83ef0f4582ed7b87c59820f9be87b.jpg', 'truckUnsafe-72-_jpg.rf.4c92a307d646a5b26e27f41a155dfd00.jpg', 'truckUnsafe-188-_jpg.rf.a84bf42690643feedcdb4319c728e825.jpg', 'truckUnsafe-131-_jpg.rf.90e0129fad38b50648f76f78418b3777.jpg', 'truckUnsafe-68-_jpg.rf.372701c8570c60192cff003497247da2.jpg', 'truckUnsafe-82-_jpg.rf.151a75d74a7794160fc510beb7bcd783.jpg', 'truckUnsafe-178-_jpg.rf.3ac5c179e6c2c7a5cbea74e3478b2ad9.jpg', 'truckUnsafe-189-_jpg.rf.8e57c964c07ff8549187692fc9c1b022.jpg', 'truckUnsafe-152-_jpg.rf.67bf16f806a35feb3259467eb213f943.jpg', 'truckSafe-165-_jpg.rf.cad213dc04105fca4c50c3ce1ecba6ca.jpg', 'truckUnsafe-61-_jpg.rf.5c973d77f5a568d776af6f3748548a03.jpg', 'truckUnsafe-138-_jpg.rf.e479f04b6e8a66ff08b8e29013e9265c.jpg', 'truckUnsafe-57-_jpg.rf.191df9af0064b469773cd3a0f70f9229.jpg', 'truckUnsafe-69-_jpg.rf.c5e1dacbed4cbc6f1afc31a7e4a551f5.jpg', 'truckSafe-112-_jpg.rf.8e746d7a4b4c801f8a1fc3f4f6477831.jpg', 'truckSafe-23-_jpg.rf.24301a8f573a7207760e2bf6517c61f4.jpg', 'truckSafe-68-_jpg.rf.78e006c591eb26eb8fcea748bfc9c3c3.jpg', 'truckUnsafe-25-_jpg.rf.cceb12e6df5d0f35bb90c4901596f4e2.jpg', 'truckSafe-159-_jpg.rf.81a0274667bc6f3acfa11c4552f819fd.jpg', 'truckUnsafe-190-_jpg.rf.6cb0f5bd29831a7842202d0723627bdf.jpg', 'truckSafe-50-_jpg.rf.e05140a6f2e41a4552cb81c87f22f3e1.jpg', 'truckSafe-9-_jpg.rf.71ca59ea13482b86225a8c1ba67a0429.jpg', 'truckUnsafe-40-_jpg.rf.cd6b17f9f652fd121f3a6a8e15eb654d.jpg', 'truckSafe-168-_jpg.rf.2c7646b81b3efb9ceed8bf2b997bad03.jpg', 'truckSafe-136-_jpg.rf.48c908bff9c2a624086b514e59b13ffc.jpg', 'truckUnsafe-83-_jpg.rf.0f9f64f54581b130c50cf540a820b669.jpg', 'truckUnsafe-31-_jpg.rf.4aec26eb17cc019b648d0a92ea9944ad.jpg', 'truckSafe-94-_jpg.rf.5cfdfe8842439763d21e889ba7a6ad61.jpg', 'truckUnsafe-168-_jpg.rf.c2f29f4b129a5454bb08d0a3d7b99544.jpg', 'truckUnsafe-76-_jpg.rf.7a77c02aa45758eaa911b02dbaed9e32.jpg', 'truckUnsafe-149-_jpg.rf.b126d7b734bb20b91b2bfd099818e2c9.jpg', 'truckSafe-128-_jpg.rf.26dd042ba0eeb0597a577d37d652a5c4.jpg', 'truckUnsafe-60-_jpg.rf.1f9281128a662c54690eb6a8688fa98f.jpg', 'truckUnsafe-8-_jpg.rf.a27f93be283dca606b152c96bbdd40e3.jpg', 'truckSafe-31-_jpg.rf.fb1ede627cf672ef46b645be0fc2ece4.jpg', 'truckSafe-72-_jpg.rf.b8faea27058cf7ff0ae7ea0e753ce3f1.jpg', 'truckSafe-47-_jpg.rf.570b68b697d7ecfde919c8749f0e8002.jpg', 'truckUnsafe-20-_jpg.rf.b478bd1204714b4287d62ef7877a45b4.jpg', 'truckUnsafe-22-_jpg.rf.180e2087cfd23ffcb8b04c313232b3ab.jpg', 'truckSafe-138-_jpg.rf.ae0c93eb4ec6b9e06a93eaf873e92b9d.jpg', 'truckSafe-5-_webp.rf.3fe61e0f0022da182f4de2f6bc3b2a58.jpg', 'truckUnsafe-136-_jpg.rf.26c2544af1f9957a430f6aee3ba13c03.jpg', 'truckSafe-1-_png.rf.c7e9cd572c91cb21b8c1d4ac101ad79a.jpg', 'truckUnsafe-200-_jpg.rf.1a13ce61158e8958b223c822802e3080.jpg', 'truckUnsafe-62-_jpg.rf.f60b501172d3815263491816aa762aa3.jpg', 'truckSafe-192-_jpg.rf.3a61ee8db33f0c3c97f0357962031b60.jpg', 'truckSafe-143-_jpg.rf.213bd5fb3648e0e1e5df30ad98b9e604.jpg', 'truckSafe-135-_jpg.rf.41737206cdde399534d020bbd47fe49c.jpg', 'truckSafe-56-_jpg.rf.883d44105fcd428f68f2531965e11162.jpg', 'truckSafe-171-_jpg.rf.070592362ff110c0b78c454193c41984.jpg', 'truckSafe-132-_jpg.rf.82a84a11180f4f71daf9dd68b0b456f7.jpg', 'truckSafe-79-_jpg.rf.abd81ee14a2be9c4006363b5ce38b7af.jpg', 'truckSafe-89-_jpg.rf.67f18184939682f9c5007fbecbdb9dbe.jpg', 'truckSafe-74-_jpg.rf.364e4fab490377d740d1707a08e8081a.jpg', 'truckSafe-151-_jpg.rf.a14cc0a9d69663026441bc5b19feee5e.jpg', 'truckSafe-147-_jpg.rf.7fa6061c05d1c0500eb4c4f8649820eb.jpg', 'truckUnsafe-32-_jpg.rf.3866133a1dc169fb9df501a57309836b.jpg', 'truckSafe-14-_jpg.rf.e0567568b5c62e5978a66a8aed08c3d1.jpg', 'truckSafe-160-_jpg.rf.5af3e09ff37e810b861fc84cbec1117c.jpg', 'truckUnsafe-73-_jpg.rf.b6f977a418e50b3600b5a97f202844dd.jpg', 'truckSafe-11-_jpg.rf.6c71b100f961c69d222281d938ebc131.jpg', 'truckSafe-77-_jpg.rf.6e11dd3de52a99335b1eccbfde3346f1.jpg', 'truckUnsafe-50-_jpg.rf.8a7f0178404d39dc81e94945a9c00771.jpg', 'truckSafe-1-_webp.rf.41ca3c859c10f738185f9c8e603ff11a.jpg', 'truckUnsafe-63-_jpg.rf.1802510f46c25d9f829cd2ba6eeb58d1.jpg', 'truckUnsafe-139-_jpg.rf.98829e0a502800918cf4573505c29747.jpg', 'truckSafe-115-_jpg.rf.ce07d75065483e165228016805608b58.jpg', 'truckSafe-4-_jpg.rf.ca4ce57a90296c7c370030b00fc6d5f1.jpg', 'truckSafe-92-_jpg.rf.8512ce0b4cbe793ab5fc77c7f344e471.jpg', 'truckUnsafe-184-_jpg.rf.860b75d169ff3565604fd17e13ba3a67.jpg', 'truckSafe-18-_jpg.rf.c35a172942205125cdfaf669f878d799.jpg'], 'val': ['truckSafe-139-_jpg.rf.bfbca44af0e528d799053e3092c79a78.jpg', 'truckUnsafe-145-_jpg.rf.41a408aec03313d78e7f8de686329990.jpg', 'truckUnsafe-45-_jpg.rf.9b951cffe7d878b553b03cd797cfb60e.jpg', 'truckSafe-167-_jpg.rf.653ffdb4b979776db1631b64bd6f32f1.jpg', 'truckUnsafe-81-_jpg.rf.0976c988b44c10754b54d4a058b57904.jpg', 'truckUnsafe-196-_jpg.rf.fdf00be085bacd5306c84e14b241ff0e.jpg', 'truckSafe-15-_jpg.rf.010c2c81a5448848cdc9722cd8674185.jpg', 'truckSafe-48-_jpg.rf.abd7a87b72f2147f85c75c553f74a437.jpg', 'truckUnsafe-148-_jpg.rf.5051fa0fa94065ee4142e28d8e71cb8b.jpg', 'truckSafe-142-_jpg.rf.49aca5c7329af95d1bb6651e8b7c55aa.jpg', 'truckUnsafe-3-_jpg.rf.61b5183f6788797dd9cc609f6dd1d167.jpg', 'truckUnsafe-16-_jpg.rf.da5db16ebd567ecaedf50b54ebedf5e1.jpg', 'truckUnsafe-34-_jpg.rf.6759a7c4f90e6c32f4aadabd71555a3a.jpg', 'truckUnsafe-109-_jpg.rf.a05f8541c4fd8c30f3dab96f798a9603.jpg', 'truckUnsafe-80-_jpg.rf.fe4f1f99d634d1f2103561d39e0edb12.jpg', 'truckSafe-39-_jpg.rf.5b26385e493d70aa4285a404653a88b5.jpg', 'truckSafe-8-_webp.rf.818239fc2a975dfd17f7733341b86f4b.jpg', 'truckUnsafe-127-_jpg.rf.2ad3044b802aed3efcc8b7bed371d70b.jpg', 'truckSafe-186-_jpg.rf.3029317de36f7e0a368b2f6d2501b8c3.jpg', 'truckSafe-52-_jpg.rf.c96d653bbff8df77bff82f9c6bb777f1.jpg', 'truckSafe-131-_jpg.rf.0c1f5be05d28edf0b553b4bd8e7158e6.jpg', 'truckUnsafe-3-_webp.rf.c674dff6c062307d014c597698bd36b1.jpg', 'truckSafe-163-_jpg.rf.7ea0aaebcbc3bbdaf27c8c330f9ccff6.jpg', 'truckSafe-75-_jpg.rf.ceaee62937387e471cc55fd76a7da4a7.jpg', 'truckUnsafe-52-_jpg.rf.3623d6ef83d47e24e61ba611ddb7f227.jpg', 'truckSafe-88-_jpg.rf.d2c3cadff6f47d1b726af8259132ac61.jpg', 'truckSafe-41-_jpg.rf.e44c8a931548a9d8f7450a43206cc764.jpg', 'truckUnsafe-23-_jpg.rf.08192ec5d1763d69b44c5560e29f6a0f.jpg', 'truckSafe-170-_jpg.rf.558b296a74c9d5f63e3161f2af8cdbc4.jpg', 'truckUnsafe-171-_jpg.rf.133139ceb1d2b99742873b8d2040ef23.jpg', 'truckUnsafe-174-_jpg.rf.64a1c55d4e43522b493ad19a3cdb5aef.jpg', 'truckUnsafe-17-_jpg.rf.18bfbfc041068b2c06bcebd48947e43d.jpg', 'truckUnsafe-70-_jpg.rf.89c6768acd46c5e4e9993f162711c003.jpg', 'truckUnsafe-6-_jpg.rf.6f20ea65517ee4f1903be31c540a051d.jpg', 'truckSafe-182-_jpg.rf.3f86479d7c598edb7be0688883e6112c.jpg', 'truckUnsafe-156-_jpg.rf.a523b36fd59be5a14650b9e277c29ed8.jpg', 'truckSafe-55-_jpg.rf.19368f9adcd1e5572a0e87f661319e3c.jpg', 'truckSafe-121-_jpg.rf.72059f36368fbde4d53c677f44065d30.jpg', 'truckSafe-6-_jpg.rf.39935e83fc92779a8e98b096aa62e92e.jpg', 'truckSafe-36-_jpg.rf.72afcabf92688c96fcfc2e27b93d99b0.jpg', 'truckSafe-101-_jpg.rf.40dc566e05fce20d06038b0ce3fe0ecf.jpg', 'truckUnsafe-187-_jpg.rf.0092685b8e65fdc2418250d631105749.jpg', 'truckUnsafe-46-_jpg.rf.ec77d5373365cb6d9cece324197f39b5.jpg', 'truckSafe-194-_jpg.rf.c1b179c6fcd4c0375fd6982e2f1e5a35.jpg', 'truckSafe-180-_jpg.rf.30727e4a07fe21babf5174c1ece296d7.jpg', 'truckSafe-179-_jpg.rf.828db7378c3802d44a575e1c44c7fa44.jpg', 'truckSafe-63-_jpg.rf.b031c09836d48b8946504ba68c000c20.jpg', 'truckUnsafe-195-_jpg.rf.be3e1027ed833259568cb52c78835a86.jpg', 'truckSafe-83-_jpg.rf.0eae6bde531e04b9a4a3bc3e07edd74c.jpg', 'truckSafe-172-_jpg.rf.c5e19482d2a3f12c8fbde13b42143ea0.jpg', 'truckUnsafe-154-_jpg.rf.572c40fd0390108e8d7cdbe0f94abaca.jpg', 'truckUnsafe-134-_jpg.rf.2b64fd235fee765bffcabc349adcb811.jpg', 'truckSafe-53-_jpg.rf.debc72c4c389011f41a94a44f69b8c42.jpg', 'truckUnsafe-181-_jpg.rf.7b902744c06191b3af371ed6c22b956c.jpg', 'truckSafe-1-_jpg.rf.c0c82f13fe564a63107241f74f0a9e3d.jpg', 'truckUnsafe-110-_jpg.rf.d0de5edb5dc0dbe52c7bfe1c0908fa0f.jpg', 'truckUnsafe-15-_jpg.rf.cc5e44fad855c19f9a65b507108efb7f.jpg', 'truckSafe-20-_jpg.rf.63a052b5b2e3bdff34aea844bcf94289.jpg', 'truckSafe-144-_jpg.rf.d2c5ddc26f85b6a3e8e7b8f26b97aceb.jpg', 'truckSafe-123-_jpg.rf.321f8187a8b607a05a0563f15a93e79f.jpg', 'truckUnsafe-185-_jpg.rf.6b2b9d024f6a35b55747d508137260a2.jpg', 'truckSafe-120-_jpg.rf.2958ef3918c556ea5f01f9b4c23e0da1.jpg', 'truckSafe-22-_jpg.rf.60c5bc122fa965840765ef1d6d1da2cd.jpg', 'truckUnsafe-35-_jpg.rf.3a140385e21297bebed441337548e8ee.jpg', 'truckSafe-113-_jpg.rf.d0971be4fc07d6bb7bf33d2d3e2d3064.jpg', 'truckUnsafe-59-_jpg.rf.b4f8d8cc6de495bacbf4708459963268.jpg', 'truckUnsafe-150-_jpg.rf.8e208de58316788754df7138624cd61e.jpg', 'truckUnsafe-116-_jpg.rf.94578f4ab6415c9fab8865a52798a70c.jpg', 'truckUnsafe-88-_jpg.rf.41c380a1b28aae61d17167da35c4de92.jpg', 'truckUnsafe-117-_jpg.rf.1cacd5f61ce9c6f7cec1af7584b6f903.jpg', 'truckSafe-190-_jpg.rf.2d8164dbd7e5cb51d97b32a13c379271.jpg', 'truckUnsafe-155-_jpg.rf.8a89427c494956e2ac55eba8460549ae.jpg', 'truckSafe-108-_jpg.rf.de0ca0561bf258860f5da9855e02e579.jpg', 'truckUnsafe-30-_jpg.rf.942a0c3d8244420914d6f474881897fa.jpg', 'truckSafe-95-_jpg.rf.b32a883f8955c72646c481c4b7e27cb5.jpg', 'truckSafe-155-_jpg.rf.8c619944eb41eb510aa6220ff4c601d2.jpg', 'truckUnsafe-65-_jpg.rf.fcfd34958d37e457faf6cff33c941b90.jpg', 'truckSafe-98-_jpg.rf.7880d9b1a5a7d5b5473cdba5666fed42.jpg', 'truckSafe-126-_jpg.rf.00ba94172a9babfe1285305d9eebcc11.jpg', 'truckUnsafe-120-_jpg.rf.a106ba89cd477455d34e6a0f5ca463e8.jpg', 'truckSafe-125-_jpg.rf.19dfe38428480391f3e4222801185034.jpg', 'truckSafe-149-_jpg.rf.aa650f61662aff9a25bd905d171b7154.jpg']}, 1: {'train': ['truckSafe-139-_jpg.rf.bfbca44af0e528d799053e3092c79a78.jpg', 'truckUnsafe-1-_jpg.rf.66c30a4c2d51092237f994a461307404.jpg', 'truckSafe-85-_jpg.rf.2f4965dbddf3770cbc1021fcbfa735d2.jpg', 'truckUnsafe-7-_jpg.rf.c0d7b79f7935ed57afd0ecb8cdedefb6.jpg', 'truckUnsafe-145-_jpg.rf.41a408aec03313d78e7f8de686329990.jpg', 'truckUnsafe-193-_jpg.rf.6c0c40e33ffa9be8acab53248760516b.jpg', 'truckUnsafe-45-_jpg.rf.9b951cffe7d878b553b03cd797cfb60e.jpg', 'truckUnsafe-205-_jpg.rf.e7ec53759f5b458230488d7e44a49aa1.jpg', 'truckSafe-110-_jpg.rf.f5db6b1c6a2a533ee002291a5f10aeff.jpg', 'truckSafe-150-_jpg.rf.1ed50e629f3d2e845d7cd23d4eb96335.jpg', 'truckSafe-2-_webp.rf.e828b2375655806bb92294d8525dfa63.jpg', 'truckSafe-167-_jpg.rf.653ffdb4b979776db1631b64bd6f32f1.jpg', 'truckUnsafe-176-_jpg.rf.9d134c36da2b927a9a28a3f99a819446.jpg', 'truckUnsafe-66-_jpg.rf.27776d206953c93e79c8934473e8565a.jpg', 'truckUnsafe-81-_jpg.rf.0976c988b44c10754b54d4a058b57904.jpg', 'truckSafe-134-_jpg.rf.bf97520f126239daa9f788726a60bcb5.jpg', 'truckUnsafe-196-_jpg.rf.fdf00be085bacd5306c84e14b241ff0e.jpg', 'truckUnsafe-29-_jpg.rf.0136f08138f167488b2b4eeaecacfa99.jpg', 'truckUnsafe-95-_jpg.rf.112c640125660cb2b77411962d0abad4.jpg', 'truckSafe-44-_jpg.rf.8a970f8bf631260f3c96a82cc097d6e8.jpg', 'truckSafe-177-_jpg.rf.7e6424dc96bc2b50c3b05cb72b65ce8b.jpg', 'truckSafe-15-_jpg.rf.010c2c81a5448848cdc9722cd8674185.jpg', 'truckSafe-48-_jpg.rf.abd7a87b72f2147f85c75c553f74a437.jpg', 'truckSafe-32-_jpg.rf.de9045b2500ea54d720f3e76a9305433.jpg', 'truckUnsafe-148-_jpg.rf.5051fa0fa94065ee4142e28d8e71cb8b.jpg', 'truckUnsafe-194-_jpg.rf.07119965499d07a9e671f515db2959a5.jpg', 'truckUnsafe-143-_jpg.rf.79d14da101782672022e049e1bdeee89.jpg', 'truckSafe-178-_jpg.rf.4bed5ae6d363dea263d1dbb66cce89f5.jpg', 'truckUnsafe-177-_jpg.rf.5d4f267583529dca089b41e9f7e97883.jpg', 'truckSafe-142-_jpg.rf.49aca5c7329af95d1bb6651e8b7c55aa.jpg', 'truckUnsafe-38-_jpg.rf.48e1108253f0374e5a089513a90e3ff0.jpg', 'truckSafe-35-_jpg.rf.6210c1789743001e32363c0ea38a15e5.jpg', 'truckUnsafe-3-_jpg.rf.61b5183f6788797dd9cc609f6dd1d167.jpg', 'truckSafe-96-_jpg.rf.181271efd70cc303a7f94f9e1a01bdb6.jpg', 'truckSafe-60-_jpg.rf.5bf0d644a9329f054b125c78f6db78de.jpg', 'truckUnsafe-16-_jpg.rf.da5db16ebd567ecaedf50b54ebedf5e1.jpg', 'truckUnsafe-34-_jpg.rf.6759a7c4f90e6c32f4aadabd71555a3a.jpg', 'truckUnsafe-129-_jpg.rf.dd4a5fe7b54d4ac7825291e4611313fd.jpg', 'truckSafe-30-_jpg.rf.e5ca2dcd251c8c9395023416c350df6a.jpg', 'truckUnsafe-56-_jpg.rf.dced56bae27edc9d6b1c458653cdd2de.jpg', 'truckSafe-28-_jpg.rf.f7d74f66420f4fe61a6f47ed7e8afd04.jpg', 'truckSafe-12-_jpg.rf.39eb53587cbba23d7368073ae290f8ad.jpg', 'truckSafe-21-_jpg.rf.b599ac3334760a3f82d388857814f39e.jpg', 'truckUnsafe-115-_jpg.rf.9877e3d97ac61c51d93ec8ea7233c80b.jpg', 'truckSafe-148-_jpg.rf.fb5880a0ed45d36edaf38617112e89bf.jpg', 'truckUnsafe-109-_jpg.rf.a05f8541c4fd8c30f3dab96f798a9603.jpg', 'truckUnsafe-80-_jpg.rf.fe4f1f99d634d1f2103561d39e0edb12.jpg', 'truckSafe-39-_jpg.rf.5b26385e493d70aa4285a404653a88b5.jpg', 'truckSafe-97-_jpg.rf.d0098bd00a7779e4865439140a6380c1.jpg', 'truckUnsafe-119-_jpg.rf.672aa2017377b983d01ca9b670ca0e91.jpg', 'truckSafe-181-_jpg.rf.ae0f5f346b1f48b2b28ee67320750f15.jpg', 'truckUnsafe-36-_jpg.rf.5ed78ccd6dbdaea3187841d0607dfa21.jpg', 'truckSafe-27-_jpg.rf.f6bb32d9ec7a45b384a973ed17090b73.jpg', 'truckUnsafe-85-_jpg.rf.1979c63ca47ceaa8533ccd773fa63214.jpg', 'truckSafe-130-_jpg.rf.b05ad19ba0d647e19a9553b76e2e598d.jpg', 'truckUnsafe-92-_jpg.rf.c0bc0344d40691da7b49ded98c8614f0.jpg', 'truckSafe-8-_webp.rf.818239fc2a975dfd17f7733341b86f4b.jpg', 'truckUnsafe-147-_jpg.rf.a245cb4334e0df8c14e6c9d25d9252f0.jpg', 'truckUnsafe-127-_jpg.rf.2ad3044b802aed3efcc8b7bed371d70b.jpg', 'truckSafe-186-_jpg.rf.3029317de36f7e0a368b2f6d2501b8c3.jpg', 'truckUnsafe-153-_jpg.rf.7019a8000dd041fc1d6f8d7fe1d8c3c5.jpg', 'truckSafe-52-_jpg.rf.c96d653bbff8df77bff82f9c6bb777f1.jpg', 'truckSafe-131-_jpg.rf.0c1f5be05d28edf0b553b4bd8e7158e6.jpg', 'truckUnsafe-3-_webp.rf.c674dff6c062307d014c597698bd36b1.jpg', 'truckSafe-162-_jpg.rf.1061926c8dbf08afb1a4d7eeab58b4c1.jpg', 'truckSafe-163-_jpg.rf.7ea0aaebcbc3bbdaf27c8c330f9ccff6.jpg', 'truckUnsafe-158-_jpg.rf.2b4978d0d7c31782247ebe044b20260a.jpg', 'truckSafe-75-_jpg.rf.ceaee62937387e471cc55fd76a7da4a7.jpg', 'truckSafe-29-_jpg.rf.5160f33be04265d12ced4efc8c13d202.jpg', 'truckSafe-19-_jpg.rf.46fc002423b49f86c30c68bd93b4bea2.jpg', 'truckUnsafe-28-_jpg.rf.f03baff0ec0f1661ad9b4dba594a741b.jpg', 'truckSafe-105-_jpg.rf.9867433ae340497a1dff7c7799dbbc4f.jpg', 'truckSafe-26-_jpg.rf.54985421d17fd0a04e187bfabd621ec2.jpg', 'truckUnsafe-52-_jpg.rf.3623d6ef83d47e24e61ba611ddb7f227.jpg', 'truckUnsafe-163-_jpg.rf.85d049ab8af8b2c86310d5ee9002537d.jpg', 'truckUnsafe-179-_jpg.rf.b0f40909cd7092c96c6a7c0bf22f9a2f.jpg', 'truckSafe-88-_jpg.rf.d2c3cadff6f47d1b726af8259132ac61.jpg', 'truckUnsafe-126-_jpg.rf.0963a81c618f910e514f53fe9626da38.jpg', 'truckUnsafe-104-_jpg.rf.050ff76c09644829e14b6eaf10728d0b.jpg', 'truckUnsafe-21-_jpg.rf.3a060c0d3b53704bd922d3d5a465fc51.jpg', 'truckSafe-73-_jpg.rf.ea654fb67610c7379028bc356f6eaddb.jpg', 'truckSafe-109-_jpg.rf.769b22963f7c9ef3f89d7b69489d2d44.jpg', 'truckSafe-188-_jpg.rf.2be4f01e41c66cbe250bb1f83a45d77c.jpg', 'truckSafe-41-_jpg.rf.e44c8a931548a9d8f7450a43206cc764.jpg', 'truckUnsafe-166-_jpg.rf.54016f8d1644b1eae8a6b6f69d590534.jpg', 'truckUnsafe-23-_jpg.rf.08192ec5d1763d69b44c5560e29f6a0f.jpg', 'truckSafe-145-_jpg.rf.14db44130e285bc48832b4a268154942.jpg', 'truckUnsafe-121-_jpg.rf.ada93505365462e8720693f545f24aa6.jpg', 'truckUnsafe-55-_jpg.rf.b79f03ebb0feb12365e3be86a65b1ea8.jpg', 'truckUnsafe-98-_jpg.rf.36d3f90d22be913327d5235c3e197c97.jpg', 'truckSafe-170-_jpg.rf.558b296a74c9d5f63e3161f2af8cdbc4.jpg', 'truckUnsafe-114-_jpg.rf.ea89c824deef9111ba53df1e3b358fc6.jpg', 'truckUnsafe-171-_jpg.rf.133139ceb1d2b99742873b8d2040ef23.jpg', 'truckUnsafe-67-_jpg.rf.7ec55fc8a43a9050056ccbaf8be3cca8.jpg', 'truckUnsafe-41-_jpg.rf.24167e65d8001dbcec8995250a395740.jpg', 'truckSafe-193-_jpg.rf.0cc0ef8c2b93513d8de1f5a9bbffc43c.jpg', 'truckSafe-106-_jpg.rf.afebeba9f3fa4fcb9573a2c57a524c14.jpg', 'truckUnsafe-43-_jpg.rf.d220226c519c51d7096fdfd4180366fc.jpg', 'truckUnsafe-174-_jpg.rf.64a1c55d4e43522b493ad19a3cdb5aef.jpg', 'truckUnsafe-49-_jpg.rf.8c40763992228923494b3b88975c49f3.jpg', 'truckUnsafe-17-_jpg.rf.18bfbfc041068b2c06bcebd48947e43d.jpg', 'truckSafe-5-_jpg.rf.aff0ea198b86b963b316a564abab4c7e.jpg', 'truckUnsafe-48-_jpg.rf.628681efd2a41b93d3ead8d21cb04a3e.jpg', 'truckUnsafe-54-_jpg.rf.e33ad9e7f96d185270f38398fd442c85.jpg', 'truckUnsafe-146-_jpg.rf.1b1dcb3b4b35776ab87be6bbfecfa8d8.jpg', 'truckUnsafe-70-_jpg.rf.89c6768acd46c5e4e9993f162711c003.jpg', 'truckSafe-185-_jpg.rf.d48bb5c402f48d46dd5ba2b0645bb54f.jpg', 'truckUnsafe-27-_jpg.rf.b67e4bd8a18953b44880689e93a9f75d.jpg', 'truckUnsafe-199-_jpg.rf.e611104b8652ea2ae107c2f05e7742a2.jpg', 'truckUnsafe-11-_jpg.rf.f0591c57aff5fbe9d96c17b33776d576.jpg', 'truckUnsafe-6-_jpg.rf.6f20ea65517ee4f1903be31c540a051d.jpg', 'truckSafe-3-_webp.rf.ae5b6a2e4cb447cb3eab97d973e07cd4.jpg', 'truckSafe-182-_jpg.rf.3f86479d7c598edb7be0688883e6112c.jpg', 'truckUnsafe-156-_jpg.rf.a523b36fd59be5a14650b9e277c29ed8.jpg', 'truckSafe-133-_jpg.rf.b20fbe959839064c74b03254da73feb4.jpg', 'truckSafe-175-_jpg.rf.1a4bed9b5c41c9ed36b8f15e76375fe4.jpg', 'truckUnsafe-84-_jpg.rf.ac6fbdce8437790b644771cdd9268fdf.jpg', 'truckSafe-55-_jpg.rf.19368f9adcd1e5572a0e87f661319e3c.jpg', 'truckSafe-66-_jpg.rf.7ccbfd700c1761ff0d798562b9baf2fe.jpg', 'truckUnsafe-91-_jpg.rf.c3e0b0affa5fd5ebd79d8ac93d570d29.jpg', 'truckUnsafe-37-_jpg.rf.0e48a6d560ca471a29e53aea79a73062.jpg', 'truckSafe-61-_jpg.rf.e31c686802cfbac5d44558ee1365a2e0.jpg', 'truckSafe-121-_jpg.rf.72059f36368fbde4d53c677f44065d30.jpg', 'truckUnsafe-94-_jpg.rf.24071cadc7abd856bbd85a2d237f6379.jpg', 'truckSafe-6-_jpg.rf.39935e83fc92779a8e98b096aa62e92e.jpg', 'truckUnsafe-192-_jpg.rf.95ff1338941d488d2a2bdd0ff7fa338e.jpg', 'truckUnsafe-165-_jpg.rf.af4191e9509fc76ef6e4298f268a8e8c.jpg', 'truckUnsafe-118-_jpg.rf.b71532e64d2ae7e38092c83aaedc44ea.jpg', 'truckUnsafe-4-_jpg.rf.ea6a0a852f9a431b0e6b9b74cbf2cf99.jpg', 'truckUnsafe-96-_jpg.rf.b9afb7a259a409e4df70272513e5ec19.jpg', 'truckSafe-2-_jpg.rf.c8ceda007375848fe364d9fb95acad70.jpg', 'truckUnsafe-87-_jpg.rf.9580a4daf820d1aba9855dd51536e723.jpg', 'truckUnsafe-167-_jpg.rf.326f6ceac9547b80d1cf6513b2e299d8.jpg', 'truckUnsafe-90-_jpg.rf.a4a481d609241403cd343a0497ad1c42.jpg', 'truckSafe-36-_jpg.rf.72afcabf92688c96fcfc2e27b93d99b0.jpg', 'truckSafe-141-_jpg.rf.1d1663f3393709d1ac68a7bab2a5dcc5.jpg', 'truckSafe-84-_jpg.rf.f12222161bc89e93ba1841fe28e6f916.jpg', 'truckSafe-174-_jpg.rf.1f16f54d94437219b942177e4bc0db67.jpg', 'truckSafe-101-_jpg.rf.40dc566e05fce20d06038b0ce3fe0ecf.jpg', 'truckUnsafe-187-_jpg.rf.0092685b8e65fdc2418250d631105749.jpg', 'truckSafe-4-_webp.rf.1398d3d92f30abc2102efab6d45f8e7d.jpg', 'truckUnsafe-46-_jpg.rf.ec77d5373365cb6d9cece324197f39b5.jpg', 'truckSafe-158-_jpg.rf.71b34d0312441967683f0c2c1080ac55.jpg', 'truckUnsafe-18-_jpg.rf.a7df91f4a191d49cfb52d239273b9f3b.jpg', 'truckSafe-194-_jpg.rf.c1b179c6fcd4c0375fd6982e2f1e5a35.jpg', 'truckSafe-152-_jpg.rf.850d0eb944f345859e327fe227e2ffd5.jpg', 'truckUnsafe-12-_jpg.rf.8b882577219da7c2a85456de63e31ab1.jpg', 'truckSafe-180-_jpg.rf.30727e4a07fe21babf5174c1ece296d7.jpg', 'truckUnsafe-100-_jpg.rf.a34138f18e494e1e2c36308440b028f3.jpg', 'truckUnsafe-111-_jpg.rf.818a3eca4a4cfe36948dcd81c135feb3.jpg', 'truckSafe-166-_jpg.rf.10f57ff7822fac232f026f0d1997be81.jpg', 'truckSafe-114-_jpg.rf.9467fcc3e3d833851f7afc2acd05adca.jpg', 'truckUnsafe-173-_jpg.rf.35096135cdaadfea73dc3dd3a145fc08.jpg', 'truckUnsafe-42-_jpg.rf.6b6e9da1327d7357a6ecd0a70191a597.jpg', 'truckSafe-164-_jpg.rf.fa9aa8d5bb4229652b71f190d1f4d25f.jpg', 'truckSafe-179-_jpg.rf.828db7378c3802d44a575e1c44c7fa44.jpg', 'truckUnsafe-201-_jpg.rf.271a90b8de5b6026cde49a6ecb489fa9.jpg', 'truckSafe-63-_jpg.rf.b031c09836d48b8946504ba68c000c20.jpg', 'truckUnsafe-103-_jpg.rf.081710aa938cdb75b0e684e3a75f8818.jpg', 'truckSafe-183-_jpg.rf.eb2027b28dc77eff4c56e3025cbc2653.jpg', 'truckUnsafe-71-_jpg.rf.a604f89995f09072cd98b2bf5eff7664.jpg', 'truckUnsafe-108-_jpg.rf.2ccd21321b603b761d1d0ff0882eaa90.jpg', 'truckUnsafe-151-_jpg.rf.97c5beaf7e14f92154ac594ac10d8ff5.jpg', 'truckUnsafe-191-_jpg.rf.95da8d25b2f7d4ba407f2da19072ce62.jpg', 'truckUnsafe-198-_jpg.rf.5b99f84247c9fc028d168c9ad4f66ec8.jpg', 'truckUnsafe-26-_jpg.rf.7320e4935f5ddfb7ff8f313bc620b5a8.jpg', 'truckSafe-176-_jpg.rf.9193d80068af2848538b375a9ab821e0.jpg', 'truckUnsafe-162-_jpg.rf.1480f80ee9853b314e72c38864c81a3a.jpg', 'truckUnsafe-195-_jpg.rf.be3e1027ed833259568cb52c78835a86.jpg', 'truckSafe-82-_jpg.rf.0c32f3892190cf4557c83d707b16e74b.jpg', 'truckUnsafe-19-_jpg.rf.b579de1709760947387a5d0a192f1801.jpg', 'truckUnsafe-172-_jpg.rf.9205866b23df7f774bf50554f3a669d4.jpg', 'truckUnsafe-53-_jpg.rf.4176b3f2f3881205d285678f6ea71a17.jpg', 'truckUnsafe-137-_jpg.rf.c66321af191a7352b5bd5c07633058dd.jpg', 'truckSafe-173-_jpg.rf.014f1884ca80ad770ca0bfba19fdca49.jpg', 'truckSafe-83-_jpg.rf.0eae6bde531e04b9a4a3bc3e07edd74c.jpg', 'truckSafe-104-_jpg.rf.d1c05a12238769503eb3d4d131fcbc9e.jpg', 'truckUnsafe-58-_jpg.rf.e3118e0dfb34d6f897354e8661ead737.jpg', 'truckUnsafe-74-_jpg.rf.9460147f5eccc8238ec6794cdcf2c0bd.jpg', 'truckSafe-172-_jpg.rf.c5e19482d2a3f12c8fbde13b42143ea0.jpg', 'truckSafe-13-_jpg.rf.05c4b7d4f35bec861860469c6a4897cd.jpg', 'truckSafe-37-_jpg.rf.ad92d1aeb3b1c3f08910c1822ce15259.jpg', 'truckUnsafe-154-_jpg.rf.572c40fd0390108e8d7cdbe0f94abaca.jpg', 'truckUnsafe-1-_webp.rf.83d14e4ad5e491866b755205e8ed1a84.jpg', 'truckSafe-99-_jpg.rf.d6fdd3a54a647a87cf406cd166efa175.jpg', 'truckUnsafe-134-_jpg.rf.2b64fd235fee765bffcabc349adcb811.jpg', 'truckSafe-17-_jpg.rf.3b732e87d7e125fc8b227f6578adf59e.jpg', 'truckUnsafe-123-_jpg.rf.e5151da7b48173b50cbd5fdcf270460b.jpg', 'truckUnsafe-133-_jpg.rf.3e694cb4e574c24f9569949ce61b39ba.jpg', 'truckSafe-189-_jpg.rf.32e5ded963fdd6234812af2d3cc0f32a.jpg', 'truckUnsafe-135-_jpg.rf.ebb702987d717323caf9023bc74dc21e.jpg', 'truckSafe-191-_jpg.rf.6af016a6449226cdc9e500cc560b58b7.jpg', 'truckUnsafe-141-_jpg.rf.825ba1db5e1614f0a7a1f30845d02b5a.jpg', 'truckSafe-53-_jpg.rf.debc72c4c389011f41a94a44f69b8c42.jpg', 'truckUnsafe-125-_jpg.rf.36ce28ef8b9d9f0ce74335e7797e30ea.jpg', 'truckSafe-86-_jpg.rf.4ecc7aac28ace212e96d7556749f8c6a.jpg', 'truckUnsafe-13-_jpg.rf.e315ed30edd8462b3103071d935b5636.jpg', 'truckUnsafe-102-_jpg.rf.ba3b895eec73c56576c420107c71fc30.jpg', 'truckUnsafe-181-_jpg.rf.7b902744c06191b3af371ed6c22b956c.jpg', 'truckSafe-58-_jpg.rf.6d7c4081902acae3464a2f527ce1a2b9.jpg', 'truckSafe-33-_jpg.rf.a577ed75fe979e9284eb7c9bf6382e70.jpg', 'truckSafe-107-_jpg.rf.f2d11dc26ef91ab7a39a1b5493c38d30.jpg', 'truckUnsafe-78-_jpg.rf.18ef332f3ab814b6cc78f0dc1a94fc55.jpg', 'truckSafe-1-_jpg.rf.c0c82f13fe564a63107241f74f0a9e3d.jpg', 'truckSafe-64-_jpg.rf.133330c5158fc2e55a7dc2d7feca10b5.jpg', 'truckUnsafe-107-_jpg.rf.babd965d5bc5ae3bffe9c00b0251d2b3.jpg', 'truckSafe-16-_jpg.rf.dd82e0008d1ca38f19a9015d9c1d904d.jpg', 'truckUnsafe-164-_jpg.rf.4bf99daa9fbd4b994d966286be2f7b78.jpg', 'truckSafe-116-_jpg.rf.4eec97da079f01e7f44e22b2203fee89.jpg', 'truckUnsafe-110-_jpg.rf.d0de5edb5dc0dbe52c7bfe1c0908fa0f.jpg', 'truckUnsafe-15-_jpg.rf.cc5e44fad855c19f9a65b507108efb7f.jpg', 'truckUnsafe-112-_jpg.rf.36928ae59d723b0cd298b437976aceac.jpg', 'truckSafe-20-_jpg.rf.63a052b5b2e3bdff34aea844bcf94289.jpg', 'truckUnsafe-51-_jpg.rf.b76605bff08c32482c482cab3d355357.jpg', 'truckSafe-10-_jpg.rf.8a9ccf6b54dad10d7d6008315ac72650.jpg', 'truckUnsafe-89-_jpg.rf.a30019a08e5a6bf467932a216fc3f622.jpg', 'truckSafe-3-_jpg.rf.75b6061d720508ec78e98e6ecb6fc39a.jpg', 'truckSafe-118-_jpg.rf.95d277cb7ae1ef4c3680def0797626ae.jpg', 'truckSafe-80-_jpg.rf.983ac820621eb75446c37990b8739001.jpg', 'truckSafe-43-_jpg.rf.8e4d9cdcb79e28695037e6f147092b1b.jpg', 'truckSafe-144-_jpg.rf.d2c5ddc26f85b6a3e8e7b8f26b97aceb.jpg', 'truckUnsafe-122-_jpg.rf.fe6f35b0c37aad5ac962f4189221e0e6.jpg', 'truckSafe-123-_jpg.rf.321f8187a8b607a05a0563f15a93e79f.jpg', 'truckUnsafe-106-_jpg.rf.be4ef15ef9ab2e2ce369596c4ae9ad31.jpg', 'truckUnsafe-182-_jpg.rf.f07e5fb6163965727248439dc676c083.jpg', 'truckSafe-129-_jpg.rf.2cd2112916d88bbdbca81f6c3856b746.jpg', 'truckUnsafe-185-_jpg.rf.6b2b9d024f6a35b55747d508137260a2.jpg', 'truckSafe-120-_jpg.rf.2958ef3918c556ea5f01f9b4c23e0da1.jpg', 'truckUnsafe-132-_jpg.rf.6d7b5d74ebbf2a0768884828e594f6b6.jpg', 'truckSafe-6-_webp.rf.a9c48df2e9f7efcfeb3af3d5fb3122c6.jpg', 'truckSafe-49-_jpg.rf.a8aca39572383374c24ba49242b55343.jpg', 'truckUnsafe-5-_jpg.rf.24986876cec09070289cb7de410a4ee2.jpg', 'truckUnsafe-75-_jpg.rf.712a44e5bdde0cfdfcc23136f784634d.jpg', 'truckSafe-137-_jpg.rf.9f2180c1d5d9004edfda5aa71550d460.jpg', 'truckSafe-57-_jpg.rf.884e1204d78fb5ee677f683e8f3d38e0.jpg', 'truckSafe-140-_jpg.rf.f9c34aa7675f6de5982107f8edeb86c3.jpg', 'truckSafe-124-_jpg.rf.f720158b37bb2809f3cd9a829ec31d22.jpg', 'truckSafe-22-_jpg.rf.60c5bc122fa965840765ef1d6d1da2cd.jpg', 'truckSafe-154-_jpg.rf.edc7e1f835cc24ddd2564d8124cc3bb4.jpg', 'truckSafe-91-_jpg.rf.a512bb3290043c460cbfae7113c731dd.jpg', 'truckSafe-71-_jpg.rf.3fc30fe5f23685a0dd8cd5ee87886689.jpg', 'truckSafe-42-_jpg.rf.c0ec22b050facbe92d64613902252be2.jpg', 'truckUnsafe-2-_webp.rf.d0dc6a759c3d04ca8d49451c36c021f3.jpg', 'truckUnsafe-35-_jpg.rf.3a140385e21297bebed441337548e8ee.jpg', 'truckSafe-78-_jpg.rf.81d818a87769c8d811532f8f2bd923a2.jpg', 'truckUnsafe-39-_jpg.rf.a9da71bff29f03d2151a906de369388d.jpg', 'truckUnsafe-113-_jpg.rf.c43fd8559af2e24e15eb90a471d17cef.jpg', 'truckSafe-113-_jpg.rf.d0971be4fc07d6bb7bf33d2d3e2d3064.jpg', 'truckUnsafe-204-_jpg.rf.9493419167eded56b9c2b7872d6e3274.jpg', 'truckSafe-93-_jpg.rf.2ae480e95b8b2cc7fc43609f9cc4a303.jpg', 'truckUnsafe-59-_jpg.rf.b4f8d8cc6de495bacbf4708459963268.jpg', 'truckSafe-187-_jpg.rf.7eb0d289edadf67321837721ea1c6e5d.jpg', 'truckSafe-54-_jpg.rf.c2cbdf3ac479b6c927a15d3edc21e937.jpg', 'truckUnsafe-202-_jpg.rf.2b3c827bff086afefc1cca63d7b9b85f.jpg', 'truckUnsafe-101-_jpg.rf.eef663035a8721e04c556ce2ed207eba.jpg', 'truckUnsafe-72-_jpg.rf.4c92a307d646a5b26e27f41a155dfd00.jpg', 'truckUnsafe-131-_jpg.rf.90e0129fad38b50648f76f78418b3777.jpg', 'truckUnsafe-68-_jpg.rf.372701c8570c60192cff003497247da2.jpg', 'truckUnsafe-82-_jpg.rf.151a75d74a7794160fc510beb7bcd783.jpg', 'truckUnsafe-150-_jpg.rf.8e208de58316788754df7138624cd61e.jpg', 'truckUnsafe-178-_jpg.rf.3ac5c179e6c2c7a5cbea74e3478b2ad9.jpg', 'truckUnsafe-189-_jpg.rf.8e57c964c07ff8549187692fc9c1b022.jpg', 'truckUnsafe-152-_jpg.rf.67bf16f806a35feb3259467eb213f943.jpg', 'truckSafe-165-_jpg.rf.cad213dc04105fca4c50c3ce1ecba6ca.jpg', 'truckUnsafe-116-_jpg.rf.94578f4ab6415c9fab8865a52798a70c.jpg', 'truckUnsafe-88-_jpg.rf.41c380a1b28aae61d17167da35c4de92.jpg', 'truckUnsafe-69-_jpg.rf.c5e1dacbed4cbc6f1afc31a7e4a551f5.jpg', 'truckSafe-112-_jpg.rf.8e746d7a4b4c801f8a1fc3f4f6477831.jpg', 'truckUnsafe-117-_jpg.rf.1cacd5f61ce9c6f7cec1af7584b6f903.jpg', 'truckSafe-23-_jpg.rf.24301a8f573a7207760e2bf6517c61f4.jpg', 'truckSafe-68-_jpg.rf.78e006c591eb26eb8fcea748bfc9c3c3.jpg', 'truckSafe-190-_jpg.rf.2d8164dbd7e5cb51d97b32a13c379271.jpg', 'truckUnsafe-25-_jpg.rf.cceb12e6df5d0f35bb90c4901596f4e2.jpg', 'truckSafe-159-_jpg.rf.81a0274667bc6f3acfa11c4552f819fd.jpg', 'truckUnsafe-190-_jpg.rf.6cb0f5bd29831a7842202d0723627bdf.jpg', 'truckSafe-50-_jpg.rf.e05140a6f2e41a4552cb81c87f22f3e1.jpg', 'truckSafe-9-_jpg.rf.71ca59ea13482b86225a8c1ba67a0429.jpg', 'truckUnsafe-40-_jpg.rf.cd6b17f9f652fd121f3a6a8e15eb654d.jpg', 'truckSafe-168-_jpg.rf.2c7646b81b3efb9ceed8bf2b997bad03.jpg', 'truckSafe-136-_jpg.rf.48c908bff9c2a624086b514e59b13ffc.jpg', 'truckUnsafe-83-_jpg.rf.0f9f64f54581b130c50cf540a820b669.jpg', 'truckUnsafe-31-_jpg.rf.4aec26eb17cc019b648d0a92ea9944ad.jpg', 'truckSafe-94-_jpg.rf.5cfdfe8842439763d21e889ba7a6ad61.jpg', 'truckUnsafe-168-_jpg.rf.c2f29f4b129a5454bb08d0a3d7b99544.jpg', 'truckUnsafe-76-_jpg.rf.7a77c02aa45758eaa911b02dbaed9e32.jpg', 'truckUnsafe-155-_jpg.rf.8a89427c494956e2ac55eba8460549ae.jpg', 'truckUnsafe-60-_jpg.rf.1f9281128a662c54690eb6a8688fa98f.jpg', 'truckUnsafe-8-_jpg.rf.a27f93be283dca606b152c96bbdd40e3.jpg', 'truckSafe-31-_jpg.rf.fb1ede627cf672ef46b645be0fc2ece4.jpg', 'truckSafe-47-_jpg.rf.570b68b697d7ecfde919c8749f0e8002.jpg', 'truckUnsafe-20-_jpg.rf.b478bd1204714b4287d62ef7877a45b4.jpg', 'truckUnsafe-22-_jpg.rf.180e2087cfd23ffcb8b04c313232b3ab.jpg', 'truckSafe-108-_jpg.rf.de0ca0561bf258860f5da9855e02e579.jpg', 'truckSafe-138-_jpg.rf.ae0c93eb4ec6b9e06a93eaf873e92b9d.jpg', 'truckUnsafe-30-_jpg.rf.942a0c3d8244420914d6f474881897fa.jpg', 'truckSafe-5-_webp.rf.3fe61e0f0022da182f4de2f6bc3b2a58.jpg', 'truckUnsafe-136-_jpg.rf.26c2544af1f9957a430f6aee3ba13c03.jpg', 'truckSafe-95-_jpg.rf.b32a883f8955c72646c481c4b7e27cb5.jpg', 'truckSafe-1-_png.rf.c7e9cd572c91cb21b8c1d4ac101ad79a.jpg', 'truckUnsafe-200-_jpg.rf.1a13ce61158e8958b223c822802e3080.jpg', 'truckUnsafe-62-_jpg.rf.f60b501172d3815263491816aa762aa3.jpg', 'truckSafe-192-_jpg.rf.3a61ee8db33f0c3c97f0357962031b60.jpg', 'truckSafe-143-_jpg.rf.213bd5fb3648e0e1e5df30ad98b9e604.jpg', 'truckSafe-155-_jpg.rf.8c619944eb41eb510aa6220ff4c601d2.jpg', 'truckUnsafe-65-_jpg.rf.fcfd34958d37e457faf6cff33c941b90.jpg', 'truckSafe-135-_jpg.rf.41737206cdde399534d020bbd47fe49c.jpg', 'truckSafe-98-_jpg.rf.7880d9b1a5a7d5b5473cdba5666fed42.jpg', 'truckSafe-79-_jpg.rf.abd81ee14a2be9c4006363b5ce38b7af.jpg', 'truckSafe-151-_jpg.rf.a14cc0a9d69663026441bc5b19feee5e.jpg', 'truckSafe-147-_jpg.rf.7fa6061c05d1c0500eb4c4f8649820eb.jpg', 'truckUnsafe-32-_jpg.rf.3866133a1dc169fb9df501a57309836b.jpg', 'truckSafe-126-_jpg.rf.00ba94172a9babfe1285305d9eebcc11.jpg', 'truckSafe-14-_jpg.rf.e0567568b5c62e5978a66a8aed08c3d1.jpg', 'truckSafe-160-_jpg.rf.5af3e09ff37e810b861fc84cbec1117c.jpg', 'truckSafe-77-_jpg.rf.6e11dd3de52a99335b1eccbfde3346f1.jpg', 'truckUnsafe-120-_jpg.rf.a106ba89cd477455d34e6a0f5ca463e8.jpg', 'truckSafe-1-_webp.rf.41ca3c859c10f738185f9c8e603ff11a.jpg', 'truckUnsafe-139-_jpg.rf.98829e0a502800918cf4573505c29747.jpg', 'truckSafe-115-_jpg.rf.ce07d75065483e165228016805608b58.jpg', 'truckSafe-4-_jpg.rf.ca4ce57a90296c7c370030b00fc6d5f1.jpg', 'truckSafe-125-_jpg.rf.19dfe38428480391f3e4222801185034.jpg', 'truckSafe-149-_jpg.rf.aa650f61662aff9a25bd905d171b7154.jpg', 'truckSafe-92-_jpg.rf.8512ce0b4cbe793ab5fc77c7f344e471.jpg', 'truckUnsafe-184-_jpg.rf.860b75d169ff3565604fd17e13ba3a67.jpg', 'truckSafe-18-_jpg.rf.c35a172942205125cdfaf669f878d799.jpg'], 'val': ['truckUnsafe-160-_jpg.rf.08ee2d61efd03a95fc5e91677cabee5e.jpg', 'truckSafe-59-_jpg.rf.4d6618413c4ca50d80c7d6d80901f8bf.jpg', 'truckUnsafe-10-_jpg.rf.d78d52add836967b3808f43fd6cecb8f.jpg', 'truckUnsafe-197-_jpg.rf.0b4368e6f43d8d46673149674ec4b528.jpg', 'truckSafe-9-_webp.rf.ec8194f35ab0607566d472c4be9dd874.jpg', 'truckUnsafe-33-_jpg.rf.de3050699444a5b892e0a8c017e86079.jpg', 'truckUnsafe-159-_jpg.rf.5bb7ecbd26dd51e833a8dfd01ee86d10.jpg', 'truckSafe-34-_jpg.rf.f695860cb2d067e30dce85dfef5018cc.jpg', 'truckSafe-45-_jpg.rf.f851c3844c8d13234a67d027ee685500.jpg', 'truckSafe-69-_jpg.rf.85937ed142b1cc9728bb8c8f2d9785e6.jpg', 'truckSafe-117-_jpg.rf.f903f0bd99d3b9f73b12dda652c88fb8.jpg', 'truckSafe-25-_jpg.rf.cd090f76ee4de2de30ef91b9fe629c57.jpg', 'truckSafe-8-_jpg.rf.d9de58166f3fc2a27cb9fb70a14f5832.jpg', 'truckUnsafe-44-_jpg.rf.e8dcce3e0ce162ab3c6bd8e4f7eaf5d0.jpg', 'truckSafe-65-_jpg.rf.a69a4e6a56701f9244892aab2c69aa5e.jpg', 'truckSafe-51-_jpg.rf.baf14a2919019fdd13b08ed4168a998a.jpg', 'truckUnsafe-77-_jpg.rf.d99926b669c1222f8f3b5668ce9d9e7d.jpg', 'truckSafe-24-_jpg.rf.aae58a00b277e292aa6ebe6a709720c1.jpg', 'truckSafe-127-_jpg.rf.5ff817b579d8b08ae45a1ad16bdcc9ff.jpg', 'truckUnsafe-47-_jpg.rf.4c887da5c757e9d5dfb2dfe245bcb1a1.jpg', 'truckUnsafe-97-_jpg.rf.062184dd5c4304a13ca457c25763a725.jpg', 'truckUnsafe-93-_jpg.rf.5185ffc4e708caf4450d6b50e637f53a.jpg', 'truckSafe-157-_jpg.rf.03771434f2554193b0fbbf476595752d.jpg', 'truckUnsafe-24-_jpg.rf.d6a5b64e5c3084a2620a5314a1afa9de.jpg', 'truckSafe-38-_jpg.rf.4c37a79e117bf366f74f4f89292e31bb.jpg', 'truckUnsafe-144-_jpg.rf.b1c7133774e19ce0a8037921cfe39646.jpg', 'truckSafe-100-_jpg.rf.9e19d4331f246a02ab81257bd84e783f.jpg', 'truckSafe-161-_jpg.rf.828c830a5836bf8462a29b72e3efafa9.jpg', 'truckSafe-7-_webp.rf.316dd3dfb61fcd5424dc6d4ef6ff1d9d.jpg', 'truckSafe-122-_jpg.rf.f1327dc90994c69507adcca64208b256.jpg', 'truckUnsafe-64-_jpg.rf.9af27645e83ae4264c19d830674615c4.jpg', 'truckUnsafe-142-_jpg.rf.135d9a65569b31f916bf316e7f151995.jpg', 'truckSafe-169-_jpg.rf.87a8c50d791b94ad907ba8afe7e9d8df.jpg', 'truckSafe-46-_jpg.rf.a38f8664325ecbc40a7e0a4729822caf.jpg', 'truckSafe-7-_jpg.rf.9f79036ac23a53c55a71bbb0dbf8e110.jpg', 'truckUnsafe-157-_jpg.rf.ff8c910ef1bce7bf7eb7695e7e7e2395.jpg', 'truckUnsafe-79-_jpg.rf.388712ca31e3c2d9d8fc7134301a9eb5.jpg', 'truckUnsafe-105-_jpg.rf.d15a089e8cce2eb10cc9215e3b6223c0.jpg', 'truckUnsafe-99-_jpg.rf.5e7d063e8fc2fac56cf4b98005c2a03f.jpg', 'truckSafe-156-_jpg.rf.73c9dd229c5811afa503f92c0ff20424.jpg', 'truckUnsafe-86-_jpg.rf.d08bb0606fa8a30be8f46a43c11bdc4a.jpg', 'truckSafe-184-_jpg.rf.43245b2aa995677c14325289e8fd1c5b.jpg', 'truckUnsafe-2-_jpg.rf.d8a033434f0bb74a6c3cbadfb339258b.jpg', 'truckSafe-153-_jpg.rf.84535b234f54b4a811a9fc3630edd9d8.jpg', 'truckUnsafe-140-_jpg.rf.701dad58d868e3c83399023aa1a00765.jpg', 'truckSafe-87-_jpg.rf.fa900c0d3f7e4c9a7667da0a03362765.jpg', 'truckUnsafe-130-_jpg.rf.ec114dd5101a292503a6b96cb7918d3d.jpg', 'truckSafe-90-_jpg.rf.59afc6a6554cbdbff546165bbd1d74a2.jpg', 'truckUnsafe-9-_jpg.rf.2f24b55c2436a8c830f604176db8e7bf.jpg', 'truckUnsafe-161-_jpg.rf.f6946afcf62d422f2f859b13ceda1e33.jpg', 'truckSafe-81-_jpg.rf.9d94f916ffef79645f1d090afd393983.jpg', 'truckUnsafe-128-_jpg.rf.c2068342dd5985126c6d1c29251fd7b2.jpg', 'truckSafe-103-_jpg.rf.565796b94aa3b9a650551dad5638dd62.jpg', 'truckUnsafe-180-_jpg.rf.bfe074781a458e41ca7bb78ba589d84a.jpg', 'truckSafe-76-_jpg.rf.3982d37199d66499fad89e44876df5a8.jpg', 'truckSafe-146-_jpg.rf.650600abdaf8b7e0a4cf3c3d6d0482f7.jpg', 'truckSafe-40-_jpg.rf.77ec15e052b68d0980bc2e69a14bc27d.jpg', 'truckUnsafe-186-_jpg.rf.8a207c733043e64808014528e310d89d.jpg', 'truckUnsafe-14-_jpg.rf.67b4993e6194f1ef8a2c62bc7bce52f4.jpg', 'truckSafe-102-_jpg.rf.7753b5b01a12f79096edaaf996ddbc6d.jpg', 'truckSafe-62-_jpg.rf.10ab461a5116c816bd0b5c8421eddbb5.jpg', 'truckUnsafe-124-_jpg.rf.4954171d878114814dc9cfff995f6c39.jpg', 'truckSafe-67-_jpg.rf.5d0c427be03cb6cdf0f45411361b502e.jpg', 'truckUnsafe-183-_jpg.rf.94580a51961ee1289b34017fc3b3d38b.jpg', 'truckUnsafe-203-_jpg.rf.6f6dea5a207484867bca181d9966c81d.jpg', 'truckUnsafe-175-_jpg.rf.49a83ef0f4582ed7b87c59820f9be87b.jpg', 'truckUnsafe-188-_jpg.rf.a84bf42690643feedcdb4319c728e825.jpg', 'truckUnsafe-61-_jpg.rf.5c973d77f5a568d776af6f3748548a03.jpg', 'truckUnsafe-138-_jpg.rf.e479f04b6e8a66ff08b8e29013e9265c.jpg', 'truckUnsafe-57-_jpg.rf.191df9af0064b469773cd3a0f70f9229.jpg', 'truckUnsafe-149-_jpg.rf.b126d7b734bb20b91b2bfd099818e2c9.jpg', 'truckSafe-128-_jpg.rf.26dd042ba0eeb0597a577d37d652a5c4.jpg', 'truckSafe-72-_jpg.rf.b8faea27058cf7ff0ae7ea0e753ce3f1.jpg', 'truckSafe-56-_jpg.rf.883d44105fcd428f68f2531965e11162.jpg', 'truckSafe-171-_jpg.rf.070592362ff110c0b78c454193c41984.jpg', 'truckSafe-132-_jpg.rf.82a84a11180f4f71daf9dd68b0b456f7.jpg', 'truckSafe-89-_jpg.rf.67f18184939682f9c5007fbecbdb9dbe.jpg', 'truckSafe-74-_jpg.rf.364e4fab490377d740d1707a08e8081a.jpg', 'truckUnsafe-73-_jpg.rf.b6f977a418e50b3600b5a97f202844dd.jpg', 'truckSafe-11-_jpg.rf.6c71b100f961c69d222281d938ebc131.jpg', 'truckUnsafe-50-_jpg.rf.8a7f0178404d39dc81e94945a9c00771.jpg', 'truckUnsafe-63-_jpg.rf.1802510f46c25d9f829cd2ba6eeb58d1.jpg']}, 2: {'train': ['truckSafe-139-_jpg.rf.bfbca44af0e528d799053e3092c79a78.jpg', 'truckUnsafe-1-_jpg.rf.66c30a4c2d51092237f994a461307404.jpg', 'truckUnsafe-160-_jpg.rf.08ee2d61efd03a95fc5e91677cabee5e.jpg', 'truckUnsafe-7-_jpg.rf.c0d7b79f7935ed57afd0ecb8cdedefb6.jpg', 'truckUnsafe-145-_jpg.rf.41a408aec03313d78e7f8de686329990.jpg', 'truckSafe-59-_jpg.rf.4d6618413c4ca50d80c7d6d80901f8bf.jpg', 'truckUnsafe-10-_jpg.rf.d78d52add836967b3808f43fd6cecb8f.jpg', 'truckUnsafe-193-_jpg.rf.6c0c40e33ffa9be8acab53248760516b.jpg', 'truckUnsafe-45-_jpg.rf.9b951cffe7d878b553b03cd797cfb60e.jpg', 'truckUnsafe-197-_jpg.rf.0b4368e6f43d8d46673149674ec4b528.jpg', 'truckSafe-110-_jpg.rf.f5db6b1c6a2a533ee002291a5f10aeff.jpg', 'truckSafe-150-_jpg.rf.1ed50e629f3d2e845d7cd23d4eb96335.jpg', 'truckSafe-2-_webp.rf.e828b2375655806bb92294d8525dfa63.jpg', 'truckSafe-167-_jpg.rf.653ffdb4b979776db1631b64bd6f32f1.jpg', 'truckSafe-9-_webp.rf.ec8194f35ab0607566d472c4be9dd874.jpg', 'truckUnsafe-33-_jpg.rf.de3050699444a5b892e0a8c017e86079.jpg', 'truckUnsafe-159-_jpg.rf.5bb7ecbd26dd51e833a8dfd01ee86d10.jpg', 'truckSafe-34-_jpg.rf.f695860cb2d067e30dce85dfef5018cc.jpg', 'truckUnsafe-176-_jpg.rf.9d134c36da2b927a9a28a3f99a819446.jpg', 'truckUnsafe-66-_jpg.rf.27776d206953c93e79c8934473e8565a.jpg', 'truckUnsafe-81-_jpg.rf.0976c988b44c10754b54d4a058b57904.jpg', 'truckSafe-45-_jpg.rf.f851c3844c8d13234a67d027ee685500.jpg', 'truckUnsafe-196-_jpg.rf.fdf00be085bacd5306c84e14b241ff0e.jpg', 'truckUnsafe-95-_jpg.rf.112c640125660cb2b77411962d0abad4.jpg', 'truckSafe-44-_jpg.rf.8a970f8bf631260f3c96a82cc097d6e8.jpg', 'truckSafe-177-_jpg.rf.7e6424dc96bc2b50c3b05cb72b65ce8b.jpg', 'truckSafe-15-_jpg.rf.010c2c81a5448848cdc9722cd8674185.jpg', 'truckSafe-48-_jpg.rf.abd7a87b72f2147f85c75c553f74a437.jpg', 'truckSafe-32-_jpg.rf.de9045b2500ea54d720f3e76a9305433.jpg', 'truckUnsafe-148-_jpg.rf.5051fa0fa94065ee4142e28d8e71cb8b.jpg', 'truckUnsafe-194-_jpg.rf.07119965499d07a9e671f515db2959a5.jpg', 'truckUnsafe-143-_jpg.rf.79d14da101782672022e049e1bdeee89.jpg', 'truckSafe-69-_jpg.rf.85937ed142b1cc9728bb8c8f2d9785e6.jpg', 'truckSafe-142-_jpg.rf.49aca5c7329af95d1bb6651e8b7c55aa.jpg', 'truckUnsafe-38-_jpg.rf.48e1108253f0374e5a089513a90e3ff0.jpg', 'truckSafe-35-_jpg.rf.6210c1789743001e32363c0ea38a15e5.jpg', 'truckUnsafe-3-_jpg.rf.61b5183f6788797dd9cc609f6dd1d167.jpg', 'truckSafe-96-_jpg.rf.181271efd70cc303a7f94f9e1a01bdb6.jpg', 'truckSafe-60-_jpg.rf.5bf0d644a9329f054b125c78f6db78de.jpg', 'truckUnsafe-16-_jpg.rf.da5db16ebd567ecaedf50b54ebedf5e1.jpg', 'truckUnsafe-34-_jpg.rf.6759a7c4f90e6c32f4aadabd71555a3a.jpg', 'truckUnsafe-129-_jpg.rf.dd4a5fe7b54d4ac7825291e4611313fd.jpg', 'truckSafe-30-_jpg.rf.e5ca2dcd251c8c9395023416c350df6a.jpg', 'truckUnsafe-56-_jpg.rf.dced56bae27edc9d6b1c458653cdd2de.jpg', 'truckSafe-28-_jpg.rf.f7d74f66420f4fe61a6f47ed7e8afd04.jpg', 'truckSafe-12-_jpg.rf.39eb53587cbba23d7368073ae290f8ad.jpg', 'truckSafe-21-_jpg.rf.b599ac3334760a3f82d388857814f39e.jpg', 'truckUnsafe-115-_jpg.rf.9877e3d97ac61c51d93ec8ea7233c80b.jpg', 'truckSafe-148-_jpg.rf.fb5880a0ed45d36edaf38617112e89bf.jpg', 'truckUnsafe-109-_jpg.rf.a05f8541c4fd8c30f3dab96f798a9603.jpg', 'truckUnsafe-80-_jpg.rf.fe4f1f99d634d1f2103561d39e0edb12.jpg', 'truckSafe-39-_jpg.rf.5b26385e493d70aa4285a404653a88b5.jpg', 'truckSafe-97-_jpg.rf.d0098bd00a7779e4865439140a6380c1.jpg', 'truckSafe-117-_jpg.rf.f903f0bd99d3b9f73b12dda652c88fb8.jpg', 'truckSafe-25-_jpg.rf.cd090f76ee4de2de30ef91b9fe629c57.jpg', 'truckUnsafe-119-_jpg.rf.672aa2017377b983d01ca9b670ca0e91.jpg', 'truckSafe-181-_jpg.rf.ae0f5f346b1f48b2b28ee67320750f15.jpg', 'truckSafe-8-_jpg.rf.d9de58166f3fc2a27cb9fb70a14f5832.jpg', 'truckUnsafe-36-_jpg.rf.5ed78ccd6dbdaea3187841d0607dfa21.jpg', 'truckSafe-27-_jpg.rf.f6bb32d9ec7a45b384a973ed17090b73.jpg', 'truckUnsafe-44-_jpg.rf.e8dcce3e0ce162ab3c6bd8e4f7eaf5d0.jpg', 'truckSafe-8-_webp.rf.818239fc2a975dfd17f7733341b86f4b.jpg', 'truckUnsafe-147-_jpg.rf.a245cb4334e0df8c14e6c9d25d9252f0.jpg', 'truckUnsafe-127-_jpg.rf.2ad3044b802aed3efcc8b7bed371d70b.jpg', 'truckSafe-186-_jpg.rf.3029317de36f7e0a368b2f6d2501b8c3.jpg', 'truckUnsafe-153-_jpg.rf.7019a8000dd041fc1d6f8d7fe1d8c3c5.jpg', 'truckSafe-65-_jpg.rf.a69a4e6a56701f9244892aab2c69aa5e.jpg', 'truckSafe-52-_jpg.rf.c96d653bbff8df77bff82f9c6bb777f1.jpg', 'truckSafe-131-_jpg.rf.0c1f5be05d28edf0b553b4bd8e7158e6.jpg', 'truckUnsafe-3-_webp.rf.c674dff6c062307d014c597698bd36b1.jpg', 'truckSafe-51-_jpg.rf.baf14a2919019fdd13b08ed4168a998a.jpg', 'truckSafe-162-_jpg.rf.1061926c8dbf08afb1a4d7eeab58b4c1.jpg', 'truckUnsafe-77-_jpg.rf.d99926b669c1222f8f3b5668ce9d9e7d.jpg', 'truckSafe-163-_jpg.rf.7ea0aaebcbc3bbdaf27c8c330f9ccff6.jpg', 'truckUnsafe-158-_jpg.rf.2b4978d0d7c31782247ebe044b20260a.jpg', 'truckSafe-75-_jpg.rf.ceaee62937387e471cc55fd76a7da4a7.jpg', 'truckSafe-29-_jpg.rf.5160f33be04265d12ced4efc8c13d202.jpg', 'truckUnsafe-28-_jpg.rf.f03baff0ec0f1661ad9b4dba594a741b.jpg', 'truckSafe-105-_jpg.rf.9867433ae340497a1dff7c7799dbbc4f.jpg', 'truckUnsafe-52-_jpg.rf.3623d6ef83d47e24e61ba611ddb7f227.jpg', 'truckUnsafe-163-_jpg.rf.85d049ab8af8b2c86310d5ee9002537d.jpg', 'truckSafe-24-_jpg.rf.aae58a00b277e292aa6ebe6a709720c1.jpg', 'truckSafe-88-_jpg.rf.d2c3cadff6f47d1b726af8259132ac61.jpg', 'truckUnsafe-126-_jpg.rf.0963a81c618f910e514f53fe9626da38.jpg', 'truckSafe-73-_jpg.rf.ea654fb67610c7379028bc356f6eaddb.jpg', 'truckSafe-109-_jpg.rf.769b22963f7c9ef3f89d7b69489d2d44.jpg', 'truckSafe-188-_jpg.rf.2be4f01e41c66cbe250bb1f83a45d77c.jpg', 'truckSafe-41-_jpg.rf.e44c8a931548a9d8f7450a43206cc764.jpg', 'truckUnsafe-166-_jpg.rf.54016f8d1644b1eae8a6b6f69d590534.jpg', 'truckSafe-127-_jpg.rf.5ff817b579d8b08ae45a1ad16bdcc9ff.jpg', 'truckUnsafe-23-_jpg.rf.08192ec5d1763d69b44c5560e29f6a0f.jpg', 'truckSafe-145-_jpg.rf.14db44130e285bc48832b4a268154942.jpg', 'truckUnsafe-121-_jpg.rf.ada93505365462e8720693f545f24aa6.jpg', 'truckUnsafe-55-_jpg.rf.b79f03ebb0feb12365e3be86a65b1ea8.jpg', 'truckUnsafe-47-_jpg.rf.4c887da5c757e9d5dfb2dfe245bcb1a1.jpg', 'truckUnsafe-97-_jpg.rf.062184dd5c4304a13ca457c25763a725.jpg', 'truckUnsafe-93-_jpg.rf.5185ffc4e708caf4450d6b50e637f53a.jpg', 'truckSafe-157-_jpg.rf.03771434f2554193b0fbbf476595752d.jpg', 'truckUnsafe-24-_jpg.rf.d6a5b64e5c3084a2620a5314a1afa9de.jpg', 'truckSafe-170-_jpg.rf.558b296a74c9d5f63e3161f2af8cdbc4.jpg', 'truckUnsafe-171-_jpg.rf.133139ceb1d2b99742873b8d2040ef23.jpg', 'truckSafe-38-_jpg.rf.4c37a79e117bf366f74f4f89292e31bb.jpg', 'truckUnsafe-144-_jpg.rf.b1c7133774e19ce0a8037921cfe39646.jpg', 'truckSafe-193-_jpg.rf.0cc0ef8c2b93513d8de1f5a9bbffc43c.jpg', 'truckUnsafe-174-_jpg.rf.64a1c55d4e43522b493ad19a3cdb5aef.jpg', 'truckUnsafe-17-_jpg.rf.18bfbfc041068b2c06bcebd48947e43d.jpg', 'truckSafe-5-_jpg.rf.aff0ea198b86b963b316a564abab4c7e.jpg', 'truckUnsafe-48-_jpg.rf.628681efd2a41b93d3ead8d21cb04a3e.jpg', 'truckUnsafe-146-_jpg.rf.1b1dcb3b4b35776ab87be6bbfecfa8d8.jpg', 'truckSafe-100-_jpg.rf.9e19d4331f246a02ab81257bd84e783f.jpg', 'truckUnsafe-70-_jpg.rf.89c6768acd46c5e4e9993f162711c003.jpg', 'truckSafe-185-_jpg.rf.d48bb5c402f48d46dd5ba2b0645bb54f.jpg', 'truckUnsafe-27-_jpg.rf.b67e4bd8a18953b44880689e93a9f75d.jpg', 'truckUnsafe-199-_jpg.rf.e611104b8652ea2ae107c2f05e7742a2.jpg', 'truckUnsafe-6-_jpg.rf.6f20ea65517ee4f1903be31c540a051d.jpg', 'truckSafe-3-_webp.rf.ae5b6a2e4cb447cb3eab97d973e07cd4.jpg', 'truckSafe-161-_jpg.rf.828c830a5836bf8462a29b72e3efafa9.jpg', 'truckSafe-182-_jpg.rf.3f86479d7c598edb7be0688883e6112c.jpg', 'truckUnsafe-156-_jpg.rf.a523b36fd59be5a14650b9e277c29ed8.jpg', 'truckSafe-133-_jpg.rf.b20fbe959839064c74b03254da73feb4.jpg', 'truckSafe-55-_jpg.rf.19368f9adcd1e5572a0e87f661319e3c.jpg', 'truckSafe-7-_webp.rf.316dd3dfb61fcd5424dc6d4ef6ff1d9d.jpg', 'truckUnsafe-37-_jpg.rf.0e48a6d560ca471a29e53aea79a73062.jpg', 'truckSafe-122-_jpg.rf.f1327dc90994c69507adcca64208b256.jpg', 'truckSafe-61-_jpg.rf.e31c686802cfbac5d44558ee1365a2e0.jpg', 'truckUnsafe-64-_jpg.rf.9af27645e83ae4264c19d830674615c4.jpg', 'truckSafe-121-_jpg.rf.72059f36368fbde4d53c677f44065d30.jpg', 'truckSafe-6-_jpg.rf.39935e83fc92779a8e98b096aa62e92e.jpg', 'truckUnsafe-192-_jpg.rf.95ff1338941d488d2a2bdd0ff7fa338e.jpg', 'truckUnsafe-142-_jpg.rf.135d9a65569b31f916bf316e7f151995.jpg', 'truckSafe-169-_jpg.rf.87a8c50d791b94ad907ba8afe7e9d8df.jpg', 'truckUnsafe-165-_jpg.rf.af4191e9509fc76ef6e4298f268a8e8c.jpg', 'truckUnsafe-118-_jpg.rf.b71532e64d2ae7e38092c83aaedc44ea.jpg', 'truckUnsafe-4-_jpg.rf.ea6a0a852f9a431b0e6b9b74cbf2cf99.jpg', 'truckUnsafe-96-_jpg.rf.b9afb7a259a409e4df70272513e5ec19.jpg', 'truckSafe-46-_jpg.rf.a38f8664325ecbc40a7e0a4729822caf.jpg', 'truckUnsafe-167-_jpg.rf.326f6ceac9547b80d1cf6513b2e299d8.jpg', 'truckSafe-36-_jpg.rf.72afcabf92688c96fcfc2e27b93d99b0.jpg', 'truckSafe-141-_jpg.rf.1d1663f3393709d1ac68a7bab2a5dcc5.jpg', 'truckSafe-84-_jpg.rf.f12222161bc89e93ba1841fe28e6f916.jpg', 'truckSafe-174-_jpg.rf.1f16f54d94437219b942177e4bc0db67.jpg', 'truckSafe-101-_jpg.rf.40dc566e05fce20d06038b0ce3fe0ecf.jpg', 'truckUnsafe-187-_jpg.rf.0092685b8e65fdc2418250d631105749.jpg', 'truckSafe-4-_webp.rf.1398d3d92f30abc2102efab6d45f8e7d.jpg', 'truckSafe-7-_jpg.rf.9f79036ac23a53c55a71bbb0dbf8e110.jpg', 'truckUnsafe-157-_jpg.rf.ff8c910ef1bce7bf7eb7695e7e7e2395.jpg', 'truckUnsafe-46-_jpg.rf.ec77d5373365cb6d9cece324197f39b5.jpg', 'truckSafe-158-_jpg.rf.71b34d0312441967683f0c2c1080ac55.jpg', 'truckSafe-194-_jpg.rf.c1b179c6fcd4c0375fd6982e2f1e5a35.jpg', 'truckUnsafe-79-_jpg.rf.388712ca31e3c2d9d8fc7134301a9eb5.jpg', 'truckSafe-180-_jpg.rf.30727e4a07fe21babf5174c1ece296d7.jpg', 'truckUnsafe-111-_jpg.rf.818a3eca4a4cfe36948dcd81c135feb3.jpg', 'truckSafe-166-_jpg.rf.10f57ff7822fac232f026f0d1997be81.jpg', 'truckSafe-114-_jpg.rf.9467fcc3e3d833851f7afc2acd05adca.jpg', 'truckUnsafe-173-_jpg.rf.35096135cdaadfea73dc3dd3a145fc08.jpg', 'truckUnsafe-42-_jpg.rf.6b6e9da1327d7357a6ecd0a70191a597.jpg', 'truckSafe-164-_jpg.rf.fa9aa8d5bb4229652b71f190d1f4d25f.jpg', 'truckSafe-179-_jpg.rf.828db7378c3802d44a575e1c44c7fa44.jpg', 'truckUnsafe-105-_jpg.rf.d15a089e8cce2eb10cc9215e3b6223c0.jpg', 'truckSafe-63-_jpg.rf.b031c09836d48b8946504ba68c000c20.jpg', 'truckUnsafe-99-_jpg.rf.5e7d063e8fc2fac56cf4b98005c2a03f.jpg', 'truckSafe-156-_jpg.rf.73c9dd229c5811afa503f92c0ff20424.jpg', 'truckUnsafe-71-_jpg.rf.a604f89995f09072cd98b2bf5eff7664.jpg', 'truckUnsafe-108-_jpg.rf.2ccd21321b603b761d1d0ff0882eaa90.jpg', 'truckUnsafe-26-_jpg.rf.7320e4935f5ddfb7ff8f313bc620b5a8.jpg', 'truckSafe-176-_jpg.rf.9193d80068af2848538b375a9ab821e0.jpg', 'truckUnsafe-162-_jpg.rf.1480f80ee9853b314e72c38864c81a3a.jpg', 'truckUnsafe-86-_jpg.rf.d08bb0606fa8a30be8f46a43c11bdc4a.jpg', 'truckUnsafe-195-_jpg.rf.be3e1027ed833259568cb52c78835a86.jpg', 'truckSafe-184-_jpg.rf.43245b2aa995677c14325289e8fd1c5b.jpg', 'truckUnsafe-2-_jpg.rf.d8a033434f0bb74a6c3cbadfb339258b.jpg', 'truckSafe-82-_jpg.rf.0c32f3892190cf4557c83d707b16e74b.jpg', 'truckUnsafe-19-_jpg.rf.b579de1709760947387a5d0a192f1801.jpg', 'truckUnsafe-172-_jpg.rf.9205866b23df7f774bf50554f3a669d4.jpg', 'truckUnsafe-53-_jpg.rf.4176b3f2f3881205d285678f6ea71a17.jpg', 'truckUnsafe-137-_jpg.rf.c66321af191a7352b5bd5c07633058dd.jpg', 'truckSafe-173-_jpg.rf.014f1884ca80ad770ca0bfba19fdca49.jpg', 'truckSafe-83-_jpg.rf.0eae6bde531e04b9a4a3bc3e07edd74c.jpg', 'truckSafe-153-_jpg.rf.84535b234f54b4a811a9fc3630edd9d8.jpg', 'truckUnsafe-140-_jpg.rf.701dad58d868e3c83399023aa1a00765.jpg', 'truckSafe-104-_jpg.rf.d1c05a12238769503eb3d4d131fcbc9e.jpg', 'truckSafe-87-_jpg.rf.fa900c0d3f7e4c9a7667da0a03362765.jpg', 'truckSafe-172-_jpg.rf.c5e19482d2a3f12c8fbde13b42143ea0.jpg', 'truckSafe-13-_jpg.rf.05c4b7d4f35bec861860469c6a4897cd.jpg', 'truckUnsafe-130-_jpg.rf.ec114dd5101a292503a6b96cb7918d3d.jpg', 'truckSafe-90-_jpg.rf.59afc6a6554cbdbff546165bbd1d74a2.jpg', 'truckUnsafe-9-_jpg.rf.2f24b55c2436a8c830f604176db8e7bf.jpg', 'truckSafe-37-_jpg.rf.ad92d1aeb3b1c3f08910c1822ce15259.jpg', 'truckUnsafe-154-_jpg.rf.572c40fd0390108e8d7cdbe0f94abaca.jpg', 'truckUnsafe-134-_jpg.rf.2b64fd235fee765bffcabc349adcb811.jpg', 'truckSafe-17-_jpg.rf.3b732e87d7e125fc8b227f6578adf59e.jpg', 'truckUnsafe-123-_jpg.rf.e5151da7b48173b50cbd5fdcf270460b.jpg', 'truckUnsafe-161-_jpg.rf.f6946afcf62d422f2f859b13ceda1e33.jpg', 'truckSafe-81-_jpg.rf.9d94f916ffef79645f1d090afd393983.jpg', 'truckSafe-189-_jpg.rf.32e5ded963fdd6234812af2d3cc0f32a.jpg', 'truckUnsafe-135-_jpg.rf.ebb702987d717323caf9023bc74dc21e.jpg', 'truckSafe-191-_jpg.rf.6af016a6449226cdc9e500cc560b58b7.jpg', 'truckUnsafe-141-_jpg.rf.825ba1db5e1614f0a7a1f30845d02b5a.jpg', 'truckUnsafe-128-_jpg.rf.c2068342dd5985126c6d1c29251fd7b2.jpg', 'truckSafe-53-_jpg.rf.debc72c4c389011f41a94a44f69b8c42.jpg', 'truckUnsafe-125-_jpg.rf.36ce28ef8b9d9f0ce74335e7797e30ea.jpg', 'truckSafe-103-_jpg.rf.565796b94aa3b9a650551dad5638dd62.jpg', 'truckUnsafe-180-_jpg.rf.bfe074781a458e41ca7bb78ba589d84a.jpg', 'truckSafe-76-_jpg.rf.3982d37199d66499fad89e44876df5a8.jpg', 'truckUnsafe-13-_jpg.rf.e315ed30edd8462b3103071d935b5636.jpg', 'truckUnsafe-102-_jpg.rf.ba3b895eec73c56576c420107c71fc30.jpg', 'truckUnsafe-181-_jpg.rf.7b902744c06191b3af371ed6c22b956c.jpg', 'truckSafe-58-_jpg.rf.6d7c4081902acae3464a2f527ce1a2b9.jpg', 'truckSafe-33-_jpg.rf.a577ed75fe979e9284eb7c9bf6382e70.jpg', 'truckSafe-107-_jpg.rf.f2d11dc26ef91ab7a39a1b5493c38d30.jpg', 'truckUnsafe-78-_jpg.rf.18ef332f3ab814b6cc78f0dc1a94fc55.jpg', 'truckSafe-1-_jpg.rf.c0c82f13fe564a63107241f74f0a9e3d.jpg', 'truckSafe-64-_jpg.rf.133330c5158fc2e55a7dc2d7feca10b5.jpg', 'truckUnsafe-107-_jpg.rf.babd965d5bc5ae3bffe9c00b0251d2b3.jpg', 'truckSafe-146-_jpg.rf.650600abdaf8b7e0a4cf3c3d6d0482f7.jpg', 'truckUnsafe-164-_jpg.rf.4bf99daa9fbd4b994d966286be2f7b78.jpg', 'truckSafe-116-_jpg.rf.4eec97da079f01e7f44e22b2203fee89.jpg', 'truckUnsafe-110-_jpg.rf.d0de5edb5dc0dbe52c7bfe1c0908fa0f.jpg', 'truckUnsafe-15-_jpg.rf.cc5e44fad855c19f9a65b507108efb7f.jpg', 'truckUnsafe-112-_jpg.rf.36928ae59d723b0cd298b437976aceac.jpg', 'truckSafe-20-_jpg.rf.63a052b5b2e3bdff34aea844bcf94289.jpg', 'truckUnsafe-51-_jpg.rf.b76605bff08c32482c482cab3d355357.jpg', 'truckSafe-10-_jpg.rf.8a9ccf6b54dad10d7d6008315ac72650.jpg', 'truckSafe-40-_jpg.rf.77ec15e052b68d0980bc2e69a14bc27d.jpg', 'truckSafe-3-_jpg.rf.75b6061d720508ec78e98e6ecb6fc39a.jpg', 'truckSafe-43-_jpg.rf.8e4d9cdcb79e28695037e6f147092b1b.jpg', 'truckSafe-144-_jpg.rf.d2c5ddc26f85b6a3e8e7b8f26b97aceb.jpg', 'truckUnsafe-186-_jpg.rf.8a207c733043e64808014528e310d89d.jpg', 'truckUnsafe-122-_jpg.rf.fe6f35b0c37aad5ac962f4189221e0e6.jpg', 'truckSafe-123-_jpg.rf.321f8187a8b607a05a0563f15a93e79f.jpg', 'truckUnsafe-182-_jpg.rf.f07e5fb6163965727248439dc676c083.jpg', 'truckUnsafe-14-_jpg.rf.67b4993e6194f1ef8a2c62bc7bce52f4.jpg', 'truckUnsafe-185-_jpg.rf.6b2b9d024f6a35b55747d508137260a2.jpg', 'truckSafe-120-_jpg.rf.2958ef3918c556ea5f01f9b4c23e0da1.jpg', 'truckSafe-6-_webp.rf.a9c48df2e9f7efcfeb3af3d5fb3122c6.jpg', 'truckSafe-102-_jpg.rf.7753b5b01a12f79096edaaf996ddbc6d.jpg', 'truckUnsafe-75-_jpg.rf.712a44e5bdde0cfdfcc23136f784634d.jpg', 'truckSafe-137-_jpg.rf.9f2180c1d5d9004edfda5aa71550d460.jpg', 'truckSafe-140-_jpg.rf.f9c34aa7675f6de5982107f8edeb86c3.jpg', 'truckSafe-124-_jpg.rf.f720158b37bb2809f3cd9a829ec31d22.jpg', 'truckSafe-62-_jpg.rf.10ab461a5116c816bd0b5c8421eddbb5.jpg', 'truckUnsafe-124-_jpg.rf.4954171d878114814dc9cfff995f6c39.jpg', 'truckSafe-22-_jpg.rf.60c5bc122fa965840765ef1d6d1da2cd.jpg', 'truckSafe-154-_jpg.rf.edc7e1f835cc24ddd2564d8124cc3bb4.jpg', 'truckSafe-67-_jpg.rf.5d0c427be03cb6cdf0f45411361b502e.jpg', 'truckSafe-91-_jpg.rf.a512bb3290043c460cbfae7113c731dd.jpg', 'truckSafe-71-_jpg.rf.3fc30fe5f23685a0dd8cd5ee87886689.jpg', 'truckSafe-42-_jpg.rf.c0ec22b050facbe92d64613902252be2.jpg', 'truckUnsafe-183-_jpg.rf.94580a51961ee1289b34017fc3b3d38b.jpg', 'truckUnsafe-2-_webp.rf.d0dc6a759c3d04ca8d49451c36c021f3.jpg', 'truckUnsafe-35-_jpg.rf.3a140385e21297bebed441337548e8ee.jpg', 'truckSafe-78-_jpg.rf.81d818a87769c8d811532f8f2bd923a2.jpg', 'truckUnsafe-39-_jpg.rf.a9da71bff29f03d2151a906de369388d.jpg', 'truckUnsafe-113-_jpg.rf.c43fd8559af2e24e15eb90a471d17cef.jpg', 'truckSafe-113-_jpg.rf.d0971be4fc07d6bb7bf33d2d3e2d3064.jpg', 'truckSafe-93-_jpg.rf.2ae480e95b8b2cc7fc43609f9cc4a303.jpg', 'truckUnsafe-59-_jpg.rf.b4f8d8cc6de495bacbf4708459963268.jpg', 'truckSafe-187-_jpg.rf.7eb0d289edadf67321837721ea1c6e5d.jpg', 'truckSafe-54-_jpg.rf.c2cbdf3ac479b6c927a15d3edc21e937.jpg', 'truckUnsafe-203-_jpg.rf.6f6dea5a207484867bca181d9966c81d.jpg', 'truckUnsafe-202-_jpg.rf.2b3c827bff086afefc1cca63d7b9b85f.jpg', 'truckUnsafe-101-_jpg.rf.eef663035a8721e04c556ce2ed207eba.jpg', 'truckUnsafe-175-_jpg.rf.49a83ef0f4582ed7b87c59820f9be87b.jpg', 'truckUnsafe-188-_jpg.rf.a84bf42690643feedcdb4319c728e825.jpg', 'truckUnsafe-68-_jpg.rf.372701c8570c60192cff003497247da2.jpg', 'truckUnsafe-150-_jpg.rf.8e208de58316788754df7138624cd61e.jpg', 'truckUnsafe-189-_jpg.rf.8e57c964c07ff8549187692fc9c1b022.jpg', 'truckUnsafe-152-_jpg.rf.67bf16f806a35feb3259467eb213f943.jpg', 'truckSafe-165-_jpg.rf.cad213dc04105fca4c50c3ce1ecba6ca.jpg', 'truckUnsafe-61-_jpg.rf.5c973d77f5a568d776af6f3748548a03.jpg', 'truckUnsafe-138-_jpg.rf.e479f04b6e8a66ff08b8e29013e9265c.jpg', 'truckUnsafe-57-_jpg.rf.191df9af0064b469773cd3a0f70f9229.jpg', 'truckUnsafe-116-_jpg.rf.94578f4ab6415c9fab8865a52798a70c.jpg', 'truckUnsafe-88-_jpg.rf.41c380a1b28aae61d17167da35c4de92.jpg', 'truckUnsafe-69-_jpg.rf.c5e1dacbed4cbc6f1afc31a7e4a551f5.jpg', 'truckSafe-112-_jpg.rf.8e746d7a4b4c801f8a1fc3f4f6477831.jpg', 'truckUnsafe-117-_jpg.rf.1cacd5f61ce9c6f7cec1af7584b6f903.jpg', 'truckSafe-23-_jpg.rf.24301a8f573a7207760e2bf6517c61f4.jpg', 'truckSafe-190-_jpg.rf.2d8164dbd7e5cb51d97b32a13c379271.jpg', 'truckSafe-159-_jpg.rf.81a0274667bc6f3acfa11c4552f819fd.jpg', 'truckUnsafe-190-_jpg.rf.6cb0f5bd29831a7842202d0723627bdf.jpg', 'truckSafe-50-_jpg.rf.e05140a6f2e41a4552cb81c87f22f3e1.jpg', 'truckUnsafe-40-_jpg.rf.cd6b17f9f652fd121f3a6a8e15eb654d.jpg', 'truckSafe-168-_jpg.rf.2c7646b81b3efb9ceed8bf2b997bad03.jpg', 'truckSafe-136-_jpg.rf.48c908bff9c2a624086b514e59b13ffc.jpg', 'truckUnsafe-31-_jpg.rf.4aec26eb17cc019b648d0a92ea9944ad.jpg', 'truckSafe-94-_jpg.rf.5cfdfe8842439763d21e889ba7a6ad61.jpg', 'truckUnsafe-149-_jpg.rf.b126d7b734bb20b91b2bfd099818e2c9.jpg', 'truckSafe-128-_jpg.rf.26dd042ba0eeb0597a577d37d652a5c4.jpg', 'truckUnsafe-155-_jpg.rf.8a89427c494956e2ac55eba8460549ae.jpg', 'truckUnsafe-8-_jpg.rf.a27f93be283dca606b152c96bbdd40e3.jpg', 'truckSafe-31-_jpg.rf.fb1ede627cf672ef46b645be0fc2ece4.jpg', 'truckSafe-72-_jpg.rf.b8faea27058cf7ff0ae7ea0e753ce3f1.jpg', 'truckSafe-47-_jpg.rf.570b68b697d7ecfde919c8749f0e8002.jpg', 'truckUnsafe-20-_jpg.rf.b478bd1204714b4287d62ef7877a45b4.jpg', 'truckUnsafe-22-_jpg.rf.180e2087cfd23ffcb8b04c313232b3ab.jpg', 'truckSafe-108-_jpg.rf.de0ca0561bf258860f5da9855e02e579.jpg', 'truckUnsafe-30-_jpg.rf.942a0c3d8244420914d6f474881897fa.jpg', 'truckSafe-95-_jpg.rf.b32a883f8955c72646c481c4b7e27cb5.jpg', 'truckSafe-1-_png.rf.c7e9cd572c91cb21b8c1d4ac101ad79a.jpg', 'truckUnsafe-200-_jpg.rf.1a13ce61158e8958b223c822802e3080.jpg', 'truckUnsafe-62-_jpg.rf.f60b501172d3815263491816aa762aa3.jpg', 'truckSafe-192-_jpg.rf.3a61ee8db33f0c3c97f0357962031b60.jpg', 'truckSafe-155-_jpg.rf.8c619944eb41eb510aa6220ff4c601d2.jpg', 'truckUnsafe-65-_jpg.rf.fcfd34958d37e457faf6cff33c941b90.jpg', 'truckSafe-98-_jpg.rf.7880d9b1a5a7d5b5473cdba5666fed42.jpg', 'truckSafe-56-_jpg.rf.883d44105fcd428f68f2531965e11162.jpg', 'truckSafe-171-_jpg.rf.070592362ff110c0b78c454193c41984.jpg', 'truckSafe-132-_jpg.rf.82a84a11180f4f71daf9dd68b0b456f7.jpg', 'truckSafe-79-_jpg.rf.abd81ee14a2be9c4006363b5ce38b7af.jpg', 'truckSafe-89-_jpg.rf.67f18184939682f9c5007fbecbdb9dbe.jpg', 'truckSafe-74-_jpg.rf.364e4fab490377d740d1707a08e8081a.jpg', 'truckSafe-126-_jpg.rf.00ba94172a9babfe1285305d9eebcc11.jpg', 'truckSafe-14-_jpg.rf.e0567568b5c62e5978a66a8aed08c3d1.jpg', 'truckUnsafe-73-_jpg.rf.b6f977a418e50b3600b5a97f202844dd.jpg', 'truckSafe-11-_jpg.rf.6c71b100f961c69d222281d938ebc131.jpg', 'truckSafe-77-_jpg.rf.6e11dd3de52a99335b1eccbfde3346f1.jpg', 'truckUnsafe-120-_jpg.rf.a106ba89cd477455d34e6a0f5ca463e8.jpg', 'truckUnsafe-50-_jpg.rf.8a7f0178404d39dc81e94945a9c00771.jpg', 'truckSafe-1-_webp.rf.41ca3c859c10f738185f9c8e603ff11a.jpg', 'truckUnsafe-63-_jpg.rf.1802510f46c25d9f829cd2ba6eeb58d1.jpg', 'truckSafe-115-_jpg.rf.ce07d75065483e165228016805608b58.jpg', 'truckSafe-125-_jpg.rf.19dfe38428480391f3e4222801185034.jpg', 'truckSafe-149-_jpg.rf.aa650f61662aff9a25bd905d171b7154.jpg', 'truckSafe-92-_jpg.rf.8512ce0b4cbe793ab5fc77c7f344e471.jpg', 'truckSafe-18-_jpg.rf.c35a172942205125cdfaf669f878d799.jpg'], 'val': ['truckSafe-85-_jpg.rf.2f4965dbddf3770cbc1021fcbfa735d2.jpg', 'truckUnsafe-205-_jpg.rf.e7ec53759f5b458230488d7e44a49aa1.jpg', 'truckSafe-134-_jpg.rf.bf97520f126239daa9f788726a60bcb5.jpg', 'truckUnsafe-29-_jpg.rf.0136f08138f167488b2b4eeaecacfa99.jpg', 'truckSafe-178-_jpg.rf.4bed5ae6d363dea263d1dbb66cce89f5.jpg', 'truckUnsafe-177-_jpg.rf.5d4f267583529dca089b41e9f7e97883.jpg', 'truckUnsafe-85-_jpg.rf.1979c63ca47ceaa8533ccd773fa63214.jpg', 'truckSafe-130-_jpg.rf.b05ad19ba0d647e19a9553b76e2e598d.jpg', 'truckUnsafe-92-_jpg.rf.c0bc0344d40691da7b49ded98c8614f0.jpg', 'truckSafe-19-_jpg.rf.46fc002423b49f86c30c68bd93b4bea2.jpg', 'truckSafe-26-_jpg.rf.54985421d17fd0a04e187bfabd621ec2.jpg', 'truckUnsafe-179-_jpg.rf.b0f40909cd7092c96c6a7c0bf22f9a2f.jpg', 'truckUnsafe-104-_jpg.rf.050ff76c09644829e14b6eaf10728d0b.jpg', 'truckUnsafe-21-_jpg.rf.3a060c0d3b53704bd922d3d5a465fc51.jpg', 'truckUnsafe-98-_jpg.rf.36d3f90d22be913327d5235c3e197c97.jpg', 'truckUnsafe-114-_jpg.rf.ea89c824deef9111ba53df1e3b358fc6.jpg', 'truckUnsafe-67-_jpg.rf.7ec55fc8a43a9050056ccbaf8be3cca8.jpg', 'truckUnsafe-41-_jpg.rf.24167e65d8001dbcec8995250a395740.jpg', 'truckSafe-106-_jpg.rf.afebeba9f3fa4fcb9573a2c57a524c14.jpg', 'truckUnsafe-43-_jpg.rf.d220226c519c51d7096fdfd4180366fc.jpg', 'truckUnsafe-49-_jpg.rf.8c40763992228923494b3b88975c49f3.jpg', 'truckUnsafe-54-_jpg.rf.e33ad9e7f96d185270f38398fd442c85.jpg', 'truckUnsafe-11-_jpg.rf.f0591c57aff5fbe9d96c17b33776d576.jpg', 'truckSafe-175-_jpg.rf.1a4bed9b5c41c9ed36b8f15e76375fe4.jpg', 'truckUnsafe-84-_jpg.rf.ac6fbdce8437790b644771cdd9268fdf.jpg', 'truckSafe-66-_jpg.rf.7ccbfd700c1761ff0d798562b9baf2fe.jpg', 'truckUnsafe-91-_jpg.rf.c3e0b0affa5fd5ebd79d8ac93d570d29.jpg', 'truckUnsafe-94-_jpg.rf.24071cadc7abd856bbd85a2d237f6379.jpg', 'truckSafe-2-_jpg.rf.c8ceda007375848fe364d9fb95acad70.jpg', 'truckUnsafe-87-_jpg.rf.9580a4daf820d1aba9855dd51536e723.jpg', 'truckUnsafe-90-_jpg.rf.a4a481d609241403cd343a0497ad1c42.jpg', 'truckUnsafe-18-_jpg.rf.a7df91f4a191d49cfb52d239273b9f3b.jpg', 'truckSafe-152-_jpg.rf.850d0eb944f345859e327fe227e2ffd5.jpg', 'truckUnsafe-12-_jpg.rf.8b882577219da7c2a85456de63e31ab1.jpg', 'truckUnsafe-100-_jpg.rf.a34138f18e494e1e2c36308440b028f3.jpg', 'truckUnsafe-201-_jpg.rf.271a90b8de5b6026cde49a6ecb489fa9.jpg', 'truckUnsafe-103-_jpg.rf.081710aa938cdb75b0e684e3a75f8818.jpg', 'truckSafe-183-_jpg.rf.eb2027b28dc77eff4c56e3025cbc2653.jpg', 'truckUnsafe-151-_jpg.rf.97c5beaf7e14f92154ac594ac10d8ff5.jpg', 'truckUnsafe-191-_jpg.rf.95da8d25b2f7d4ba407f2da19072ce62.jpg', 'truckUnsafe-198-_jpg.rf.5b99f84247c9fc028d168c9ad4f66ec8.jpg', 'truckUnsafe-58-_jpg.rf.e3118e0dfb34d6f897354e8661ead737.jpg', 'truckUnsafe-74-_jpg.rf.9460147f5eccc8238ec6794cdcf2c0bd.jpg', 'truckUnsafe-1-_webp.rf.83d14e4ad5e491866b755205e8ed1a84.jpg', 'truckSafe-99-_jpg.rf.d6fdd3a54a647a87cf406cd166efa175.jpg', 'truckUnsafe-133-_jpg.rf.3e694cb4e574c24f9569949ce61b39ba.jpg', 'truckSafe-86-_jpg.rf.4ecc7aac28ace212e96d7556749f8c6a.jpg', 'truckSafe-16-_jpg.rf.dd82e0008d1ca38f19a9015d9c1d904d.jpg', 'truckUnsafe-89-_jpg.rf.a30019a08e5a6bf467932a216fc3f622.jpg', 'truckSafe-118-_jpg.rf.95d277cb7ae1ef4c3680def0797626ae.jpg', 'truckSafe-80-_jpg.rf.983ac820621eb75446c37990b8739001.jpg', 'truckUnsafe-106-_jpg.rf.be4ef15ef9ab2e2ce369596c4ae9ad31.jpg', 'truckSafe-129-_jpg.rf.2cd2112916d88bbdbca81f6c3856b746.jpg', 'truckUnsafe-132-_jpg.rf.6d7b5d74ebbf2a0768884828e594f6b6.jpg', 'truckSafe-49-_jpg.rf.a8aca39572383374c24ba49242b55343.jpg', 'truckUnsafe-5-_jpg.rf.24986876cec09070289cb7de410a4ee2.jpg', 'truckSafe-57-_jpg.rf.884e1204d78fb5ee677f683e8f3d38e0.jpg', 'truckUnsafe-204-_jpg.rf.9493419167eded56b9c2b7872d6e3274.jpg', 'truckUnsafe-72-_jpg.rf.4c92a307d646a5b26e27f41a155dfd00.jpg', 'truckUnsafe-131-_jpg.rf.90e0129fad38b50648f76f78418b3777.jpg', 'truckUnsafe-82-_jpg.rf.151a75d74a7794160fc510beb7bcd783.jpg', 'truckUnsafe-178-_jpg.rf.3ac5c179e6c2c7a5cbea74e3478b2ad9.jpg', 'truckSafe-68-_jpg.rf.78e006c591eb26eb8fcea748bfc9c3c3.jpg', 'truckUnsafe-25-_jpg.rf.cceb12e6df5d0f35bb90c4901596f4e2.jpg', 'truckSafe-9-_jpg.rf.71ca59ea13482b86225a8c1ba67a0429.jpg', 'truckUnsafe-83-_jpg.rf.0f9f64f54581b130c50cf540a820b669.jpg', 'truckUnsafe-168-_jpg.rf.c2f29f4b129a5454bb08d0a3d7b99544.jpg', 'truckUnsafe-76-_jpg.rf.7a77c02aa45758eaa911b02dbaed9e32.jpg', 'truckUnsafe-60-_jpg.rf.1f9281128a662c54690eb6a8688fa98f.jpg', 'truckSafe-138-_jpg.rf.ae0c93eb4ec6b9e06a93eaf873e92b9d.jpg', 'truckSafe-5-_webp.rf.3fe61e0f0022da182f4de2f6bc3b2a58.jpg', 'truckUnsafe-136-_jpg.rf.26c2544af1f9957a430f6aee3ba13c03.jpg', 'truckSafe-143-_jpg.rf.213bd5fb3648e0e1e5df30ad98b9e604.jpg', 'truckSafe-135-_jpg.rf.41737206cdde399534d020bbd47fe49c.jpg', 'truckSafe-151-_jpg.rf.a14cc0a9d69663026441bc5b19feee5e.jpg', 'truckSafe-147-_jpg.rf.7fa6061c05d1c0500eb4c4f8649820eb.jpg', 'truckUnsafe-32-_jpg.rf.3866133a1dc169fb9df501a57309836b.jpg', 'truckSafe-160-_jpg.rf.5af3e09ff37e810b861fc84cbec1117c.jpg', 'truckUnsafe-139-_jpg.rf.98829e0a502800918cf4573505c29747.jpg', 'truckSafe-4-_jpg.rf.ca4ce57a90296c7c370030b00fc6d5f1.jpg', 'truckUnsafe-184-_jpg.rf.860b75d169ff3565604fd17e13ba3a67.jpg']}, 3: {'train': ['truckSafe-139-_jpg.rf.bfbca44af0e528d799053e3092c79a78.jpg', 'truckUnsafe-1-_jpg.rf.66c30a4c2d51092237f994a461307404.jpg', 'truckSafe-85-_jpg.rf.2f4965dbddf3770cbc1021fcbfa735d2.jpg', 'truckUnsafe-160-_jpg.rf.08ee2d61efd03a95fc5e91677cabee5e.jpg', 'truckUnsafe-145-_jpg.rf.41a408aec03313d78e7f8de686329990.jpg', 'truckSafe-59-_jpg.rf.4d6618413c4ca50d80c7d6d80901f8bf.jpg', 'truckUnsafe-10-_jpg.rf.d78d52add836967b3808f43fd6cecb8f.jpg', 'truckUnsafe-45-_jpg.rf.9b951cffe7d878b553b03cd797cfb60e.jpg', 'truckUnsafe-197-_jpg.rf.0b4368e6f43d8d46673149674ec4b528.jpg', 'truckUnsafe-205-_jpg.rf.e7ec53759f5b458230488d7e44a49aa1.jpg', 'truckSafe-150-_jpg.rf.1ed50e629f3d2e845d7cd23d4eb96335.jpg', 'truckSafe-167-_jpg.rf.653ffdb4b979776db1631b64bd6f32f1.jpg', 'truckSafe-9-_webp.rf.ec8194f35ab0607566d472c4be9dd874.jpg', 'truckUnsafe-33-_jpg.rf.de3050699444a5b892e0a8c017e86079.jpg', 'truckUnsafe-159-_jpg.rf.5bb7ecbd26dd51e833a8dfd01ee86d10.jpg', 'truckSafe-34-_jpg.rf.f695860cb2d067e30dce85dfef5018cc.jpg', 'truckUnsafe-176-_jpg.rf.9d134c36da2b927a9a28a3f99a819446.jpg', 'truckUnsafe-66-_jpg.rf.27776d206953c93e79c8934473e8565a.jpg', 'truckUnsafe-81-_jpg.rf.0976c988b44c10754b54d4a058b57904.jpg', 'truckSafe-134-_jpg.rf.bf97520f126239daa9f788726a60bcb5.jpg', 'truckSafe-45-_jpg.rf.f851c3844c8d13234a67d027ee685500.jpg', 'truckUnsafe-196-_jpg.rf.fdf00be085bacd5306c84e14b241ff0e.jpg', 'truckUnsafe-29-_jpg.rf.0136f08138f167488b2b4eeaecacfa99.jpg', 'truckSafe-15-_jpg.rf.010c2c81a5448848cdc9722cd8674185.jpg', 'truckSafe-48-_jpg.rf.abd7a87b72f2147f85c75c553f74a437.jpg', 'truckUnsafe-148-_jpg.rf.5051fa0fa94065ee4142e28d8e71cb8b.jpg', 'truckUnsafe-194-_jpg.rf.07119965499d07a9e671f515db2959a5.jpg', 'truckSafe-69-_jpg.rf.85937ed142b1cc9728bb8c8f2d9785e6.jpg', 'truckSafe-178-_jpg.rf.4bed5ae6d363dea263d1dbb66cce89f5.jpg', 'truckUnsafe-177-_jpg.rf.5d4f267583529dca089b41e9f7e97883.jpg', 'truckSafe-142-_jpg.rf.49aca5c7329af95d1bb6651e8b7c55aa.jpg', 'truckUnsafe-3-_jpg.rf.61b5183f6788797dd9cc609f6dd1d167.jpg', 'truckSafe-96-_jpg.rf.181271efd70cc303a7f94f9e1a01bdb6.jpg', 'truckUnsafe-16-_jpg.rf.da5db16ebd567ecaedf50b54ebedf5e1.jpg', 'truckUnsafe-34-_jpg.rf.6759a7c4f90e6c32f4aadabd71555a3a.jpg', 'truckSafe-30-_jpg.rf.e5ca2dcd251c8c9395023416c350df6a.jpg', 'truckUnsafe-56-_jpg.rf.dced56bae27edc9d6b1c458653cdd2de.jpg', 'truckSafe-28-_jpg.rf.f7d74f66420f4fe61a6f47ed7e8afd04.jpg', 'truckSafe-21-_jpg.rf.b599ac3334760a3f82d388857814f39e.jpg', 'truckUnsafe-115-_jpg.rf.9877e3d97ac61c51d93ec8ea7233c80b.jpg', 'truckSafe-148-_jpg.rf.fb5880a0ed45d36edaf38617112e89bf.jpg', 'truckUnsafe-109-_jpg.rf.a05f8541c4fd8c30f3dab96f798a9603.jpg', 'truckUnsafe-80-_jpg.rf.fe4f1f99d634d1f2103561d39e0edb12.jpg', 'truckSafe-39-_jpg.rf.5b26385e493d70aa4285a404653a88b5.jpg', 'truckSafe-97-_jpg.rf.d0098bd00a7779e4865439140a6380c1.jpg', 'truckSafe-117-_jpg.rf.f903f0bd99d3b9f73b12dda652c88fb8.jpg', 'truckSafe-25-_jpg.rf.cd090f76ee4de2de30ef91b9fe629c57.jpg', 'truckSafe-8-_jpg.rf.d9de58166f3fc2a27cb9fb70a14f5832.jpg', 'truckUnsafe-44-_jpg.rf.e8dcce3e0ce162ab3c6bd8e4f7eaf5d0.jpg', 'truckUnsafe-85-_jpg.rf.1979c63ca47ceaa8533ccd773fa63214.jpg', 'truckSafe-130-_jpg.rf.b05ad19ba0d647e19a9553b76e2e598d.jpg', 'truckUnsafe-92-_jpg.rf.c0bc0344d40691da7b49ded98c8614f0.jpg', 'truckSafe-8-_webp.rf.818239fc2a975dfd17f7733341b86f4b.jpg', 'truckUnsafe-147-_jpg.rf.a245cb4334e0df8c14e6c9d25d9252f0.jpg', 'truckUnsafe-127-_jpg.rf.2ad3044b802aed3efcc8b7bed371d70b.jpg', 'truckSafe-186-_jpg.rf.3029317de36f7e0a368b2f6d2501b8c3.jpg', 'truckSafe-65-_jpg.rf.a69a4e6a56701f9244892aab2c69aa5e.jpg', 'truckSafe-52-_jpg.rf.c96d653bbff8df77bff82f9c6bb777f1.jpg', 'truckSafe-131-_jpg.rf.0c1f5be05d28edf0b553b4bd8e7158e6.jpg', 'truckUnsafe-3-_webp.rf.c674dff6c062307d014c597698bd36b1.jpg', 'truckSafe-51-_jpg.rf.baf14a2919019fdd13b08ed4168a998a.jpg', 'truckSafe-162-_jpg.rf.1061926c8dbf08afb1a4d7eeab58b4c1.jpg', 'truckUnsafe-77-_jpg.rf.d99926b669c1222f8f3b5668ce9d9e7d.jpg', 'truckSafe-163-_jpg.rf.7ea0aaebcbc3bbdaf27c8c330f9ccff6.jpg', 'truckSafe-75-_jpg.rf.ceaee62937387e471cc55fd76a7da4a7.jpg', 'truckSafe-19-_jpg.rf.46fc002423b49f86c30c68bd93b4bea2.jpg', 'truckUnsafe-28-_jpg.rf.f03baff0ec0f1661ad9b4dba594a741b.jpg', 'truckSafe-105-_jpg.rf.9867433ae340497a1dff7c7799dbbc4f.jpg', 'truckSafe-26-_jpg.rf.54985421d17fd0a04e187bfabd621ec2.jpg', 'truckUnsafe-52-_jpg.rf.3623d6ef83d47e24e61ba611ddb7f227.jpg', 'truckUnsafe-163-_jpg.rf.85d049ab8af8b2c86310d5ee9002537d.jpg', 'truckUnsafe-179-_jpg.rf.b0f40909cd7092c96c6a7c0bf22f9a2f.jpg', 'truckSafe-24-_jpg.rf.aae58a00b277e292aa6ebe6a709720c1.jpg', 'truckSafe-88-_jpg.rf.d2c3cadff6f47d1b726af8259132ac61.jpg', 'truckUnsafe-104-_jpg.rf.050ff76c09644829e14b6eaf10728d0b.jpg', 'truckUnsafe-21-_jpg.rf.3a060c0d3b53704bd922d3d5a465fc51.jpg', 'truckSafe-109-_jpg.rf.769b22963f7c9ef3f89d7b69489d2d44.jpg', 'truckSafe-41-_jpg.rf.e44c8a931548a9d8f7450a43206cc764.jpg', 'truckUnsafe-166-_jpg.rf.54016f8d1644b1eae8a6b6f69d590534.jpg', 'truckSafe-127-_jpg.rf.5ff817b579d8b08ae45a1ad16bdcc9ff.jpg', 'truckUnsafe-23-_jpg.rf.08192ec5d1763d69b44c5560e29f6a0f.jpg', 'truckSafe-145-_jpg.rf.14db44130e285bc48832b4a268154942.jpg', 'truckUnsafe-121-_jpg.rf.ada93505365462e8720693f545f24aa6.jpg', 'truckUnsafe-47-_jpg.rf.4c887da5c757e9d5dfb2dfe245bcb1a1.jpg', 'truckUnsafe-97-_jpg.rf.062184dd5c4304a13ca457c25763a725.jpg', 'truckUnsafe-93-_jpg.rf.5185ffc4e708caf4450d6b50e637f53a.jpg', 'truckSafe-157-_jpg.rf.03771434f2554193b0fbbf476595752d.jpg', 'truckUnsafe-98-_jpg.rf.36d3f90d22be913327d5235c3e197c97.jpg', 'truckUnsafe-24-_jpg.rf.d6a5b64e5c3084a2620a5314a1afa9de.jpg', 'truckSafe-170-_jpg.rf.558b296a74c9d5f63e3161f2af8cdbc4.jpg', 'truckUnsafe-114-_jpg.rf.ea89c824deef9111ba53df1e3b358fc6.jpg', 'truckUnsafe-171-_jpg.rf.133139ceb1d2b99742873b8d2040ef23.jpg', 'truckUnsafe-67-_jpg.rf.7ec55fc8a43a9050056ccbaf8be3cca8.jpg', 'truckSafe-38-_jpg.rf.4c37a79e117bf366f74f4f89292e31bb.jpg', 'truckUnsafe-144-_jpg.rf.b1c7133774e19ce0a8037921cfe39646.jpg', 'truckUnsafe-41-_jpg.rf.24167e65d8001dbcec8995250a395740.jpg', 'truckSafe-193-_jpg.rf.0cc0ef8c2b93513d8de1f5a9bbffc43c.jpg', 'truckSafe-106-_jpg.rf.afebeba9f3fa4fcb9573a2c57a524c14.jpg', 'truckUnsafe-43-_jpg.rf.d220226c519c51d7096fdfd4180366fc.jpg', 'truckUnsafe-174-_jpg.rf.64a1c55d4e43522b493ad19a3cdb5aef.jpg', 'truckUnsafe-49-_jpg.rf.8c40763992228923494b3b88975c49f3.jpg', 'truckUnsafe-17-_jpg.rf.18bfbfc041068b2c06bcebd48947e43d.jpg', 'truckUnsafe-54-_jpg.rf.e33ad9e7f96d185270f38398fd442c85.jpg', 'truckUnsafe-146-_jpg.rf.1b1dcb3b4b35776ab87be6bbfecfa8d8.jpg', 'truckSafe-100-_jpg.rf.9e19d4331f246a02ab81257bd84e783f.jpg', 'truckUnsafe-70-_jpg.rf.89c6768acd46c5e4e9993f162711c003.jpg', 'truckUnsafe-27-_jpg.rf.b67e4bd8a18953b44880689e93a9f75d.jpg', 'truckUnsafe-11-_jpg.rf.f0591c57aff5fbe9d96c17b33776d576.jpg', 'truckUnsafe-6-_jpg.rf.6f20ea65517ee4f1903be31c540a051d.jpg', 'truckSafe-161-_jpg.rf.828c830a5836bf8462a29b72e3efafa9.jpg', 'truckSafe-182-_jpg.rf.3f86479d7c598edb7be0688883e6112c.jpg', 'truckUnsafe-156-_jpg.rf.a523b36fd59be5a14650b9e277c29ed8.jpg', 'truckSafe-175-_jpg.rf.1a4bed9b5c41c9ed36b8f15e76375fe4.jpg', 'truckUnsafe-84-_jpg.rf.ac6fbdce8437790b644771cdd9268fdf.jpg', 'truckSafe-55-_jpg.rf.19368f9adcd1e5572a0e87f661319e3c.jpg', 'truckSafe-66-_jpg.rf.7ccbfd700c1761ff0d798562b9baf2fe.jpg', 'truckUnsafe-91-_jpg.rf.c3e0b0affa5fd5ebd79d8ac93d570d29.jpg', 'truckSafe-7-_webp.rf.316dd3dfb61fcd5424dc6d4ef6ff1d9d.jpg', 'truckUnsafe-37-_jpg.rf.0e48a6d560ca471a29e53aea79a73062.jpg', 'truckSafe-122-_jpg.rf.f1327dc90994c69507adcca64208b256.jpg', 'truckSafe-61-_jpg.rf.e31c686802cfbac5d44558ee1365a2e0.jpg', 'truckUnsafe-64-_jpg.rf.9af27645e83ae4264c19d830674615c4.jpg', 'truckSafe-121-_jpg.rf.72059f36368fbde4d53c677f44065d30.jpg', 'truckUnsafe-94-_jpg.rf.24071cadc7abd856bbd85a2d237f6379.jpg', 'truckSafe-6-_jpg.rf.39935e83fc92779a8e98b096aa62e92e.jpg', 'truckUnsafe-142-_jpg.rf.135d9a65569b31f916bf316e7f151995.jpg', 'truckSafe-169-_jpg.rf.87a8c50d791b94ad907ba8afe7e9d8df.jpg', 'truckUnsafe-118-_jpg.rf.b71532e64d2ae7e38092c83aaedc44ea.jpg', 'truckUnsafe-4-_jpg.rf.ea6a0a852f9a431b0e6b9b74cbf2cf99.jpg', 'truckSafe-2-_jpg.rf.c8ceda007375848fe364d9fb95acad70.jpg', 'truckUnsafe-87-_jpg.rf.9580a4daf820d1aba9855dd51536e723.jpg', 'truckSafe-46-_jpg.rf.a38f8664325ecbc40a7e0a4729822caf.jpg', 'truckUnsafe-167-_jpg.rf.326f6ceac9547b80d1cf6513b2e299d8.jpg', 'truckUnsafe-90-_jpg.rf.a4a481d609241403cd343a0497ad1c42.jpg', 'truckSafe-36-_jpg.rf.72afcabf92688c96fcfc2e27b93d99b0.jpg', 'truckSafe-141-_jpg.rf.1d1663f3393709d1ac68a7bab2a5dcc5.jpg', 'truckSafe-101-_jpg.rf.40dc566e05fce20d06038b0ce3fe0ecf.jpg', 'truckUnsafe-187-_jpg.rf.0092685b8e65fdc2418250d631105749.jpg', 'truckSafe-4-_webp.rf.1398d3d92f30abc2102efab6d45f8e7d.jpg', 'truckSafe-7-_jpg.rf.9f79036ac23a53c55a71bbb0dbf8e110.jpg', 'truckUnsafe-157-_jpg.rf.ff8c910ef1bce7bf7eb7695e7e7e2395.jpg', 'truckUnsafe-46-_jpg.rf.ec77d5373365cb6d9cece324197f39b5.jpg', 'truckUnsafe-18-_jpg.rf.a7df91f4a191d49cfb52d239273b9f3b.jpg', 'truckSafe-194-_jpg.rf.c1b179c6fcd4c0375fd6982e2f1e5a35.jpg', 'truckUnsafe-79-_jpg.rf.388712ca31e3c2d9d8fc7134301a9eb5.jpg', 'truckSafe-152-_jpg.rf.850d0eb944f345859e327fe227e2ffd5.jpg', 'truckUnsafe-12-_jpg.rf.8b882577219da7c2a85456de63e31ab1.jpg', 'truckSafe-180-_jpg.rf.30727e4a07fe21babf5174c1ece296d7.jpg', 'truckUnsafe-100-_jpg.rf.a34138f18e494e1e2c36308440b028f3.jpg', 'truckSafe-166-_jpg.rf.10f57ff7822fac232f026f0d1997be81.jpg', 'truckSafe-114-_jpg.rf.9467fcc3e3d833851f7afc2acd05adca.jpg', 'truckUnsafe-173-_jpg.rf.35096135cdaadfea73dc3dd3a145fc08.jpg', 'truckUnsafe-42-_jpg.rf.6b6e9da1327d7357a6ecd0a70191a597.jpg', 'truckSafe-164-_jpg.rf.fa9aa8d5bb4229652b71f190d1f4d25f.jpg', 'truckSafe-179-_jpg.rf.828db7378c3802d44a575e1c44c7fa44.jpg', 'truckUnsafe-201-_jpg.rf.271a90b8de5b6026cde49a6ecb489fa9.jpg', 'truckUnsafe-105-_jpg.rf.d15a089e8cce2eb10cc9215e3b6223c0.jpg', 'truckSafe-63-_jpg.rf.b031c09836d48b8946504ba68c000c20.jpg', 'truckUnsafe-99-_jpg.rf.5e7d063e8fc2fac56cf4b98005c2a03f.jpg', 'truckUnsafe-103-_jpg.rf.081710aa938cdb75b0e684e3a75f8818.jpg', 'truckSafe-156-_jpg.rf.73c9dd229c5811afa503f92c0ff20424.jpg', 'truckSafe-183-_jpg.rf.eb2027b28dc77eff4c56e3025cbc2653.jpg', 'truckUnsafe-108-_jpg.rf.2ccd21321b603b761d1d0ff0882eaa90.jpg', 'truckUnsafe-151-_jpg.rf.97c5beaf7e14f92154ac594ac10d8ff5.jpg', 'truckUnsafe-191-_jpg.rf.95da8d25b2f7d4ba407f2da19072ce62.jpg', 'truckUnsafe-198-_jpg.rf.5b99f84247c9fc028d168c9ad4f66ec8.jpg', 'truckUnsafe-26-_jpg.rf.7320e4935f5ddfb7ff8f313bc620b5a8.jpg', 'truckUnsafe-86-_jpg.rf.d08bb0606fa8a30be8f46a43c11bdc4a.jpg', 'truckUnsafe-195-_jpg.rf.be3e1027ed833259568cb52c78835a86.jpg', 'truckSafe-184-_jpg.rf.43245b2aa995677c14325289e8fd1c5b.jpg', 'truckUnsafe-2-_jpg.rf.d8a033434f0bb74a6c3cbadfb339258b.jpg', 'truckSafe-82-_jpg.rf.0c32f3892190cf4557c83d707b16e74b.jpg', 'truckUnsafe-172-_jpg.rf.9205866b23df7f774bf50554f3a669d4.jpg', 'truckSafe-173-_jpg.rf.014f1884ca80ad770ca0bfba19fdca49.jpg', 'truckSafe-83-_jpg.rf.0eae6bde531e04b9a4a3bc3e07edd74c.jpg', 'truckSafe-153-_jpg.rf.84535b234f54b4a811a9fc3630edd9d8.jpg', 'truckUnsafe-140-_jpg.rf.701dad58d868e3c83399023aa1a00765.jpg', 'truckUnsafe-58-_jpg.rf.e3118e0dfb34d6f897354e8661ead737.jpg', 'truckSafe-87-_jpg.rf.fa900c0d3f7e4c9a7667da0a03362765.jpg', 'truckUnsafe-74-_jpg.rf.9460147f5eccc8238ec6794cdcf2c0bd.jpg', 'truckSafe-172-_jpg.rf.c5e19482d2a3f12c8fbde13b42143ea0.jpg', 'truckUnsafe-130-_jpg.rf.ec114dd5101a292503a6b96cb7918d3d.jpg', 'truckSafe-90-_jpg.rf.59afc6a6554cbdbff546165bbd1d74a2.jpg', 'truckUnsafe-9-_jpg.rf.2f24b55c2436a8c830f604176db8e7bf.jpg', 'truckUnsafe-154-_jpg.rf.572c40fd0390108e8d7cdbe0f94abaca.jpg', 'truckUnsafe-1-_webp.rf.83d14e4ad5e491866b755205e8ed1a84.jpg', 'truckSafe-99-_jpg.rf.d6fdd3a54a647a87cf406cd166efa175.jpg', 'truckUnsafe-134-_jpg.rf.2b64fd235fee765bffcabc349adcb811.jpg', 'truckSafe-17-_jpg.rf.3b732e87d7e125fc8b227f6578adf59e.jpg', 'truckUnsafe-161-_jpg.rf.f6946afcf62d422f2f859b13ceda1e33.jpg', 'truckSafe-81-_jpg.rf.9d94f916ffef79645f1d090afd393983.jpg', 'truckUnsafe-133-_jpg.rf.3e694cb4e574c24f9569949ce61b39ba.jpg', 'truckUnsafe-135-_jpg.rf.ebb702987d717323caf9023bc74dc21e.jpg', 'truckUnsafe-141-_jpg.rf.825ba1db5e1614f0a7a1f30845d02b5a.jpg', 'truckUnsafe-128-_jpg.rf.c2068342dd5985126c6d1c29251fd7b2.jpg', 'truckSafe-53-_jpg.rf.debc72c4c389011f41a94a44f69b8c42.jpg', 'truckSafe-103-_jpg.rf.565796b94aa3b9a650551dad5638dd62.jpg', 'truckUnsafe-180-_jpg.rf.bfe074781a458e41ca7bb78ba589d84a.jpg', 'truckSafe-86-_jpg.rf.4ecc7aac28ace212e96d7556749f8c6a.jpg', 'truckSafe-76-_jpg.rf.3982d37199d66499fad89e44876df5a8.jpg', 'truckUnsafe-13-_jpg.rf.e315ed30edd8462b3103071d935b5636.jpg', 'truckUnsafe-102-_jpg.rf.ba3b895eec73c56576c420107c71fc30.jpg', 'truckUnsafe-181-_jpg.rf.7b902744c06191b3af371ed6c22b956c.jpg', 'truckUnsafe-78-_jpg.rf.18ef332f3ab814b6cc78f0dc1a94fc55.jpg', 'truckSafe-1-_jpg.rf.c0c82f13fe564a63107241f74f0a9e3d.jpg', 'truckSafe-64-_jpg.rf.133330c5158fc2e55a7dc2d7feca10b5.jpg', 'truckSafe-146-_jpg.rf.650600abdaf8b7e0a4cf3c3d6d0482f7.jpg', 'truckSafe-16-_jpg.rf.dd82e0008d1ca38f19a9015d9c1d904d.jpg', 'truckUnsafe-164-_jpg.rf.4bf99daa9fbd4b994d966286be2f7b78.jpg', 'truckSafe-116-_jpg.rf.4eec97da079f01e7f44e22b2203fee89.jpg', 'truckUnsafe-110-_jpg.rf.d0de5edb5dc0dbe52c7bfe1c0908fa0f.jpg', 'truckUnsafe-15-_jpg.rf.cc5e44fad855c19f9a65b507108efb7f.jpg', 'truckSafe-20-_jpg.rf.63a052b5b2e3bdff34aea844bcf94289.jpg', 'truckUnsafe-51-_jpg.rf.b76605bff08c32482c482cab3d355357.jpg', 'truckSafe-10-_jpg.rf.8a9ccf6b54dad10d7d6008315ac72650.jpg', 'truckUnsafe-89-_jpg.rf.a30019a08e5a6bf467932a216fc3f622.jpg', 'truckSafe-40-_jpg.rf.77ec15e052b68d0980bc2e69a14bc27d.jpg', 'truckSafe-3-_jpg.rf.75b6061d720508ec78e98e6ecb6fc39a.jpg', 'truckSafe-118-_jpg.rf.95d277cb7ae1ef4c3680def0797626ae.jpg', 'truckSafe-80-_jpg.rf.983ac820621eb75446c37990b8739001.jpg', 'truckSafe-43-_jpg.rf.8e4d9cdcb79e28695037e6f147092b1b.jpg', 'truckSafe-144-_jpg.rf.d2c5ddc26f85b6a3e8e7b8f26b97aceb.jpg', 'truckUnsafe-186-_jpg.rf.8a207c733043e64808014528e310d89d.jpg', 'truckSafe-123-_jpg.rf.321f8187a8b607a05a0563f15a93e79f.jpg', 'truckUnsafe-106-_jpg.rf.be4ef15ef9ab2e2ce369596c4ae9ad31.jpg', 'truckUnsafe-14-_jpg.rf.67b4993e6194f1ef8a2c62bc7bce52f4.jpg', 'truckSafe-129-_jpg.rf.2cd2112916d88bbdbca81f6c3856b746.jpg', 'truckUnsafe-185-_jpg.rf.6b2b9d024f6a35b55747d508137260a2.jpg', 'truckSafe-120-_jpg.rf.2958ef3918c556ea5f01f9b4c23e0da1.jpg', 'truckUnsafe-132-_jpg.rf.6d7b5d74ebbf2a0768884828e594f6b6.jpg', 'truckSafe-49-_jpg.rf.a8aca39572383374c24ba49242b55343.jpg', 'truckSafe-102-_jpg.rf.7753b5b01a12f79096edaaf996ddbc6d.jpg', 'truckUnsafe-5-_jpg.rf.24986876cec09070289cb7de410a4ee2.jpg', 'truckSafe-137-_jpg.rf.9f2180c1d5d9004edfda5aa71550d460.jpg', 'truckSafe-57-_jpg.rf.884e1204d78fb5ee677f683e8f3d38e0.jpg', 'truckSafe-140-_jpg.rf.f9c34aa7675f6de5982107f8edeb86c3.jpg', 'truckSafe-62-_jpg.rf.10ab461a5116c816bd0b5c8421eddbb5.jpg', 'truckUnsafe-124-_jpg.rf.4954171d878114814dc9cfff995f6c39.jpg', 'truckSafe-22-_jpg.rf.60c5bc122fa965840765ef1d6d1da2cd.jpg', 'truckSafe-67-_jpg.rf.5d0c427be03cb6cdf0f45411361b502e.jpg', 'truckSafe-71-_jpg.rf.3fc30fe5f23685a0dd8cd5ee87886689.jpg', 'truckUnsafe-183-_jpg.rf.94580a51961ee1289b34017fc3b3d38b.jpg', 'truckUnsafe-2-_webp.rf.d0dc6a759c3d04ca8d49451c36c021f3.jpg', 'truckUnsafe-35-_jpg.rf.3a140385e21297bebed441337548e8ee.jpg', 'truckSafe-78-_jpg.rf.81d818a87769c8d811532f8f2bd923a2.jpg', 'truckUnsafe-39-_jpg.rf.a9da71bff29f03d2151a906de369388d.jpg', 'truckSafe-113-_jpg.rf.d0971be4fc07d6bb7bf33d2d3e2d3064.jpg', 'truckUnsafe-204-_jpg.rf.9493419167eded56b9c2b7872d6e3274.jpg', 'truckSafe-93-_jpg.rf.2ae480e95b8b2cc7fc43609f9cc4a303.jpg', 'truckUnsafe-59-_jpg.rf.b4f8d8cc6de495bacbf4708459963268.jpg', 'truckSafe-187-_jpg.rf.7eb0d289edadf67321837721ea1c6e5d.jpg', 'truckUnsafe-203-_jpg.rf.6f6dea5a207484867bca181d9966c81d.jpg', 'truckUnsafe-101-_jpg.rf.eef663035a8721e04c556ce2ed207eba.jpg', 'truckUnsafe-175-_jpg.rf.49a83ef0f4582ed7b87c59820f9be87b.jpg', 'truckUnsafe-72-_jpg.rf.4c92a307d646a5b26e27f41a155dfd00.jpg', 'truckUnsafe-188-_jpg.rf.a84bf42690643feedcdb4319c728e825.jpg', 'truckUnsafe-131-_jpg.rf.90e0129fad38b50648f76f78418b3777.jpg', 'truckUnsafe-82-_jpg.rf.151a75d74a7794160fc510beb7bcd783.jpg', 'truckUnsafe-150-_jpg.rf.8e208de58316788754df7138624cd61e.jpg', 'truckUnsafe-178-_jpg.rf.3ac5c179e6c2c7a5cbea74e3478b2ad9.jpg', 'truckUnsafe-189-_jpg.rf.8e57c964c07ff8549187692fc9c1b022.jpg', 'truckSafe-165-_jpg.rf.cad213dc04105fca4c50c3ce1ecba6ca.jpg', 'truckUnsafe-61-_jpg.rf.5c973d77f5a568d776af6f3748548a03.jpg', 'truckUnsafe-138-_jpg.rf.e479f04b6e8a66ff08b8e29013e9265c.jpg', 'truckUnsafe-57-_jpg.rf.191df9af0064b469773cd3a0f70f9229.jpg', 'truckUnsafe-116-_jpg.rf.94578f4ab6415c9fab8865a52798a70c.jpg', 'truckUnsafe-88-_jpg.rf.41c380a1b28aae61d17167da35c4de92.jpg', 'truckUnsafe-117-_jpg.rf.1cacd5f61ce9c6f7cec1af7584b6f903.jpg', 'truckSafe-23-_jpg.rf.24301a8f573a7207760e2bf6517c61f4.jpg', 'truckSafe-68-_jpg.rf.78e006c591eb26eb8fcea748bfc9c3c3.jpg', 'truckSafe-190-_jpg.rf.2d8164dbd7e5cb51d97b32a13c379271.jpg', 'truckUnsafe-25-_jpg.rf.cceb12e6df5d0f35bb90c4901596f4e2.jpg', 'truckSafe-159-_jpg.rf.81a0274667bc6f3acfa11c4552f819fd.jpg', 'truckUnsafe-190-_jpg.rf.6cb0f5bd29831a7842202d0723627bdf.jpg', 'truckSafe-50-_jpg.rf.e05140a6f2e41a4552cb81c87f22f3e1.jpg', 'truckSafe-9-_jpg.rf.71ca59ea13482b86225a8c1ba67a0429.jpg', 'truckSafe-168-_jpg.rf.2c7646b81b3efb9ceed8bf2b997bad03.jpg', 'truckUnsafe-83-_jpg.rf.0f9f64f54581b130c50cf540a820b669.jpg', 'truckUnsafe-168-_jpg.rf.c2f29f4b129a5454bb08d0a3d7b99544.jpg', 'truckUnsafe-76-_jpg.rf.7a77c02aa45758eaa911b02dbaed9e32.jpg', 'truckUnsafe-149-_jpg.rf.b126d7b734bb20b91b2bfd099818e2c9.jpg', 'truckSafe-128-_jpg.rf.26dd042ba0eeb0597a577d37d652a5c4.jpg', 'truckUnsafe-155-_jpg.rf.8a89427c494956e2ac55eba8460549ae.jpg', 'truckUnsafe-60-_jpg.rf.1f9281128a662c54690eb6a8688fa98f.jpg', 'truckUnsafe-8-_jpg.rf.a27f93be283dca606b152c96bbdd40e3.jpg', 'truckSafe-72-_jpg.rf.b8faea27058cf7ff0ae7ea0e753ce3f1.jpg', 'truckUnsafe-20-_jpg.rf.b478bd1204714b4287d62ef7877a45b4.jpg', 'truckSafe-108-_jpg.rf.de0ca0561bf258860f5da9855e02e579.jpg', 'truckSafe-138-_jpg.rf.ae0c93eb4ec6b9e06a93eaf873e92b9d.jpg', 'truckUnsafe-30-_jpg.rf.942a0c3d8244420914d6f474881897fa.jpg', 'truckSafe-5-_webp.rf.3fe61e0f0022da182f4de2f6bc3b2a58.jpg', 'truckUnsafe-136-_jpg.rf.26c2544af1f9957a430f6aee3ba13c03.jpg', 'truckSafe-95-_jpg.rf.b32a883f8955c72646c481c4b7e27cb5.jpg', 'truckSafe-1-_png.rf.c7e9cd572c91cb21b8c1d4ac101ad79a.jpg', 'truckUnsafe-200-_jpg.rf.1a13ce61158e8958b223c822802e3080.jpg', 'truckSafe-143-_jpg.rf.213bd5fb3648e0e1e5df30ad98b9e604.jpg', 'truckSafe-155-_jpg.rf.8c619944eb41eb510aa6220ff4c601d2.jpg', 'truckUnsafe-65-_jpg.rf.fcfd34958d37e457faf6cff33c941b90.jpg', 'truckSafe-135-_jpg.rf.41737206cdde399534d020bbd47fe49c.jpg', 'truckSafe-98-_jpg.rf.7880d9b1a5a7d5b5473cdba5666fed42.jpg', 'truckSafe-56-_jpg.rf.883d44105fcd428f68f2531965e11162.jpg', 'truckSafe-171-_jpg.rf.070592362ff110c0b78c454193c41984.jpg', 'truckSafe-132-_jpg.rf.82a84a11180f4f71daf9dd68b0b456f7.jpg', 'truckSafe-89-_jpg.rf.67f18184939682f9c5007fbecbdb9dbe.jpg', 'truckSafe-74-_jpg.rf.364e4fab490377d740d1707a08e8081a.jpg', 'truckSafe-151-_jpg.rf.a14cc0a9d69663026441bc5b19feee5e.jpg', 'truckSafe-147-_jpg.rf.7fa6061c05d1c0500eb4c4f8649820eb.jpg', 'truckUnsafe-32-_jpg.rf.3866133a1dc169fb9df501a57309836b.jpg', 'truckSafe-126-_jpg.rf.00ba94172a9babfe1285305d9eebcc11.jpg', 'truckSafe-14-_jpg.rf.e0567568b5c62e5978a66a8aed08c3d1.jpg', 'truckSafe-160-_jpg.rf.5af3e09ff37e810b861fc84cbec1117c.jpg', 'truckUnsafe-73-_jpg.rf.b6f977a418e50b3600b5a97f202844dd.jpg', 'truckSafe-11-_jpg.rf.6c71b100f961c69d222281d938ebc131.jpg', 'truckSafe-77-_jpg.rf.6e11dd3de52a99335b1eccbfde3346f1.jpg', 'truckUnsafe-120-_jpg.rf.a106ba89cd477455d34e6a0f5ca463e8.jpg', 'truckUnsafe-50-_jpg.rf.8a7f0178404d39dc81e94945a9c00771.jpg', 'truckSafe-1-_webp.rf.41ca3c859c10f738185f9c8e603ff11a.jpg', 'truckUnsafe-63-_jpg.rf.1802510f46c25d9f829cd2ba6eeb58d1.jpg', 'truckUnsafe-139-_jpg.rf.98829e0a502800918cf4573505c29747.jpg', 'truckSafe-115-_jpg.rf.ce07d75065483e165228016805608b58.jpg', 'truckSafe-4-_jpg.rf.ca4ce57a90296c7c370030b00fc6d5f1.jpg', 'truckSafe-125-_jpg.rf.19dfe38428480391f3e4222801185034.jpg', 'truckSafe-149-_jpg.rf.aa650f61662aff9a25bd905d171b7154.jpg', 'truckSafe-92-_jpg.rf.8512ce0b4cbe793ab5fc77c7f344e471.jpg', 'truckUnsafe-184-_jpg.rf.860b75d169ff3565604fd17e13ba3a67.jpg', 'truckSafe-18-_jpg.rf.c35a172942205125cdfaf669f878d799.jpg'], 'val': ['truckUnsafe-7-_jpg.rf.c0d7b79f7935ed57afd0ecb8cdedefb6.jpg', 'truckUnsafe-193-_jpg.rf.6c0c40e33ffa9be8acab53248760516b.jpg', 'truckSafe-110-_jpg.rf.f5db6b1c6a2a533ee002291a5f10aeff.jpg', 'truckSafe-2-_webp.rf.e828b2375655806bb92294d8525dfa63.jpg', 'truckUnsafe-95-_jpg.rf.112c640125660cb2b77411962d0abad4.jpg', 'truckSafe-44-_jpg.rf.8a970f8bf631260f3c96a82cc097d6e8.jpg', 'truckSafe-177-_jpg.rf.7e6424dc96bc2b50c3b05cb72b65ce8b.jpg', 'truckSafe-32-_jpg.rf.de9045b2500ea54d720f3e76a9305433.jpg', 'truckUnsafe-143-_jpg.rf.79d14da101782672022e049e1bdeee89.jpg', 'truckUnsafe-38-_jpg.rf.48e1108253f0374e5a089513a90e3ff0.jpg', 'truckSafe-35-_jpg.rf.6210c1789743001e32363c0ea38a15e5.jpg', 'truckSafe-60-_jpg.rf.5bf0d644a9329f054b125c78f6db78de.jpg', 'truckUnsafe-129-_jpg.rf.dd4a5fe7b54d4ac7825291e4611313fd.jpg', 'truckSafe-12-_jpg.rf.39eb53587cbba23d7368073ae290f8ad.jpg', 'truckUnsafe-119-_jpg.rf.672aa2017377b983d01ca9b670ca0e91.jpg', 'truckSafe-181-_jpg.rf.ae0f5f346b1f48b2b28ee67320750f15.jpg', 'truckUnsafe-36-_jpg.rf.5ed78ccd6dbdaea3187841d0607dfa21.jpg', 'truckSafe-27-_jpg.rf.f6bb32d9ec7a45b384a973ed17090b73.jpg', 'truckUnsafe-153-_jpg.rf.7019a8000dd041fc1d6f8d7fe1d8c3c5.jpg', 'truckUnsafe-158-_jpg.rf.2b4978d0d7c31782247ebe044b20260a.jpg', 'truckSafe-29-_jpg.rf.5160f33be04265d12ced4efc8c13d202.jpg', 'truckUnsafe-126-_jpg.rf.0963a81c618f910e514f53fe9626da38.jpg', 'truckSafe-73-_jpg.rf.ea654fb67610c7379028bc356f6eaddb.jpg', 'truckSafe-188-_jpg.rf.2be4f01e41c66cbe250bb1f83a45d77c.jpg', 'truckUnsafe-55-_jpg.rf.b79f03ebb0feb12365e3be86a65b1ea8.jpg', 'truckSafe-5-_jpg.rf.aff0ea198b86b963b316a564abab4c7e.jpg', 'truckUnsafe-48-_jpg.rf.628681efd2a41b93d3ead8d21cb04a3e.jpg', 'truckSafe-185-_jpg.rf.d48bb5c402f48d46dd5ba2b0645bb54f.jpg', 'truckUnsafe-199-_jpg.rf.e611104b8652ea2ae107c2f05e7742a2.jpg', 'truckSafe-3-_webp.rf.ae5b6a2e4cb447cb3eab97d973e07cd4.jpg', 'truckSafe-133-_jpg.rf.b20fbe959839064c74b03254da73feb4.jpg', 'truckUnsafe-192-_jpg.rf.95ff1338941d488d2a2bdd0ff7fa338e.jpg', 'truckUnsafe-165-_jpg.rf.af4191e9509fc76ef6e4298f268a8e8c.jpg', 'truckUnsafe-96-_jpg.rf.b9afb7a259a409e4df70272513e5ec19.jpg', 'truckSafe-84-_jpg.rf.f12222161bc89e93ba1841fe28e6f916.jpg', 'truckSafe-174-_jpg.rf.1f16f54d94437219b942177e4bc0db67.jpg', 'truckSafe-158-_jpg.rf.71b34d0312441967683f0c2c1080ac55.jpg', 'truckUnsafe-111-_jpg.rf.818a3eca4a4cfe36948dcd81c135feb3.jpg', 'truckUnsafe-71-_jpg.rf.a604f89995f09072cd98b2bf5eff7664.jpg', 'truckSafe-176-_jpg.rf.9193d80068af2848538b375a9ab821e0.jpg', 'truckUnsafe-162-_jpg.rf.1480f80ee9853b314e72c38864c81a3a.jpg', 'truckUnsafe-19-_jpg.rf.b579de1709760947387a5d0a192f1801.jpg', 'truckUnsafe-53-_jpg.rf.4176b3f2f3881205d285678f6ea71a17.jpg', 'truckUnsafe-137-_jpg.rf.c66321af191a7352b5bd5c07633058dd.jpg', 'truckSafe-104-_jpg.rf.d1c05a12238769503eb3d4d131fcbc9e.jpg', 'truckSafe-13-_jpg.rf.05c4b7d4f35bec861860469c6a4897cd.jpg', 'truckSafe-37-_jpg.rf.ad92d1aeb3b1c3f08910c1822ce15259.jpg', 'truckUnsafe-123-_jpg.rf.e5151da7b48173b50cbd5fdcf270460b.jpg', 'truckSafe-189-_jpg.rf.32e5ded963fdd6234812af2d3cc0f32a.jpg', 'truckSafe-191-_jpg.rf.6af016a6449226cdc9e500cc560b58b7.jpg', 'truckUnsafe-125-_jpg.rf.36ce28ef8b9d9f0ce74335e7797e30ea.jpg', 'truckSafe-58-_jpg.rf.6d7c4081902acae3464a2f527ce1a2b9.jpg', 'truckSafe-33-_jpg.rf.a577ed75fe979e9284eb7c9bf6382e70.jpg', 'truckSafe-107-_jpg.rf.f2d11dc26ef91ab7a39a1b5493c38d30.jpg', 'truckUnsafe-107-_jpg.rf.babd965d5bc5ae3bffe9c00b0251d2b3.jpg', 'truckUnsafe-112-_jpg.rf.36928ae59d723b0cd298b437976aceac.jpg', 'truckUnsafe-122-_jpg.rf.fe6f35b0c37aad5ac962f4189221e0e6.jpg', 'truckUnsafe-182-_jpg.rf.f07e5fb6163965727248439dc676c083.jpg', 'truckSafe-6-_webp.rf.a9c48df2e9f7efcfeb3af3d5fb3122c6.jpg', 'truckUnsafe-75-_jpg.rf.712a44e5bdde0cfdfcc23136f784634d.jpg', 'truckSafe-124-_jpg.rf.f720158b37bb2809f3cd9a829ec31d22.jpg', 'truckSafe-154-_jpg.rf.edc7e1f835cc24ddd2564d8124cc3bb4.jpg', 'truckSafe-91-_jpg.rf.a512bb3290043c460cbfae7113c731dd.jpg', 'truckSafe-42-_jpg.rf.c0ec22b050facbe92d64613902252be2.jpg', 'truckUnsafe-113-_jpg.rf.c43fd8559af2e24e15eb90a471d17cef.jpg', 'truckSafe-54-_jpg.rf.c2cbdf3ac479b6c927a15d3edc21e937.jpg', 'truckUnsafe-202-_jpg.rf.2b3c827bff086afefc1cca63d7b9b85f.jpg', 'truckUnsafe-68-_jpg.rf.372701c8570c60192cff003497247da2.jpg', 'truckUnsafe-152-_jpg.rf.67bf16f806a35feb3259467eb213f943.jpg', 'truckUnsafe-69-_jpg.rf.c5e1dacbed4cbc6f1afc31a7e4a551f5.jpg', 'truckSafe-112-_jpg.rf.8e746d7a4b4c801f8a1fc3f4f6477831.jpg', 'truckUnsafe-40-_jpg.rf.cd6b17f9f652fd121f3a6a8e15eb654d.jpg', 'truckSafe-136-_jpg.rf.48c908bff9c2a624086b514e59b13ffc.jpg', 'truckUnsafe-31-_jpg.rf.4aec26eb17cc019b648d0a92ea9944ad.jpg', 'truckSafe-94-_jpg.rf.5cfdfe8842439763d21e889ba7a6ad61.jpg', 'truckSafe-31-_jpg.rf.fb1ede627cf672ef46b645be0fc2ece4.jpg', 'truckSafe-47-_jpg.rf.570b68b697d7ecfde919c8749f0e8002.jpg', 'truckUnsafe-22-_jpg.rf.180e2087cfd23ffcb8b04c313232b3ab.jpg', 'truckUnsafe-62-_jpg.rf.f60b501172d3815263491816aa762aa3.jpg', 'truckSafe-192-_jpg.rf.3a61ee8db33f0c3c97f0357962031b60.jpg', 'truckSafe-79-_jpg.rf.abd81ee14a2be9c4006363b5ce38b7af.jpg']}, 4: {'train': ['truckSafe-139-_jpg.rf.bfbca44af0e528d799053e3092c79a78.jpg', 'truckSafe-85-_jpg.rf.2f4965dbddf3770cbc1021fcbfa735d2.jpg', 'truckUnsafe-160-_jpg.rf.08ee2d61efd03a95fc5e91677cabee5e.jpg', 'truckUnsafe-7-_jpg.rf.c0d7b79f7935ed57afd0ecb8cdedefb6.jpg', 'truckUnsafe-145-_jpg.rf.41a408aec03313d78e7f8de686329990.jpg', 'truckSafe-59-_jpg.rf.4d6618413c4ca50d80c7d6d80901f8bf.jpg', 'truckUnsafe-10-_jpg.rf.d78d52add836967b3808f43fd6cecb8f.jpg', 'truckUnsafe-193-_jpg.rf.6c0c40e33ffa9be8acab53248760516b.jpg', 'truckUnsafe-45-_jpg.rf.9b951cffe7d878b553b03cd797cfb60e.jpg', 'truckUnsafe-197-_jpg.rf.0b4368e6f43d8d46673149674ec4b528.jpg', 'truckUnsafe-205-_jpg.rf.e7ec53759f5b458230488d7e44a49aa1.jpg', 'truckSafe-110-_jpg.rf.f5db6b1c6a2a533ee002291a5f10aeff.jpg', 'truckSafe-2-_webp.rf.e828b2375655806bb92294d8525dfa63.jpg', 'truckSafe-167-_jpg.rf.653ffdb4b979776db1631b64bd6f32f1.jpg', 'truckSafe-9-_webp.rf.ec8194f35ab0607566d472c4be9dd874.jpg', 'truckUnsafe-33-_jpg.rf.de3050699444a5b892e0a8c017e86079.jpg', 'truckUnsafe-159-_jpg.rf.5bb7ecbd26dd51e833a8dfd01ee86d10.jpg', 'truckSafe-34-_jpg.rf.f695860cb2d067e30dce85dfef5018cc.jpg', 'truckUnsafe-81-_jpg.rf.0976c988b44c10754b54d4a058b57904.jpg', 'truckSafe-134-_jpg.rf.bf97520f126239daa9f788726a60bcb5.jpg', 'truckSafe-45-_jpg.rf.f851c3844c8d13234a67d027ee685500.jpg', 'truckUnsafe-196-_jpg.rf.fdf00be085bacd5306c84e14b241ff0e.jpg', 'truckUnsafe-29-_jpg.rf.0136f08138f167488b2b4eeaecacfa99.jpg', 'truckUnsafe-95-_jpg.rf.112c640125660cb2b77411962d0abad4.jpg', 'truckSafe-44-_jpg.rf.8a970f8bf631260f3c96a82cc097d6e8.jpg', 'truckSafe-177-_jpg.rf.7e6424dc96bc2b50c3b05cb72b65ce8b.jpg', 'truckSafe-15-_jpg.rf.010c2c81a5448848cdc9722cd8674185.jpg', 'truckSafe-48-_jpg.rf.abd7a87b72f2147f85c75c553f74a437.jpg', 'truckSafe-32-_jpg.rf.de9045b2500ea54d720f3e76a9305433.jpg', 'truckUnsafe-148-_jpg.rf.5051fa0fa94065ee4142e28d8e71cb8b.jpg', 'truckUnsafe-143-_jpg.rf.79d14da101782672022e049e1bdeee89.jpg', 'truckSafe-69-_jpg.rf.85937ed142b1cc9728bb8c8f2d9785e6.jpg', 'truckSafe-178-_jpg.rf.4bed5ae6d363dea263d1dbb66cce89f5.jpg', 'truckUnsafe-177-_jpg.rf.5d4f267583529dca089b41e9f7e97883.jpg', 'truckSafe-142-_jpg.rf.49aca5c7329af95d1bb6651e8b7c55aa.jpg', 'truckUnsafe-38-_jpg.rf.48e1108253f0374e5a089513a90e3ff0.jpg', 'truckSafe-35-_jpg.rf.6210c1789743001e32363c0ea38a15e5.jpg', 'truckUnsafe-3-_jpg.rf.61b5183f6788797dd9cc609f6dd1d167.jpg', 'truckSafe-60-_jpg.rf.5bf0d644a9329f054b125c78f6db78de.jpg', 'truckUnsafe-16-_jpg.rf.da5db16ebd567ecaedf50b54ebedf5e1.jpg', 'truckUnsafe-34-_jpg.rf.6759a7c4f90e6c32f4aadabd71555a3a.jpg', 'truckUnsafe-129-_jpg.rf.dd4a5fe7b54d4ac7825291e4611313fd.jpg', 'truckSafe-12-_jpg.rf.39eb53587cbba23d7368073ae290f8ad.jpg', 'truckUnsafe-109-_jpg.rf.a05f8541c4fd8c30f3dab96f798a9603.jpg', 'truckUnsafe-80-_jpg.rf.fe4f1f99d634d1f2103561d39e0edb12.jpg', 'truckSafe-39-_jpg.rf.5b26385e493d70aa4285a404653a88b5.jpg', 'truckSafe-117-_jpg.rf.f903f0bd99d3b9f73b12dda652c88fb8.jpg', 'truckSafe-25-_jpg.rf.cd090f76ee4de2de30ef91b9fe629c57.jpg', 'truckUnsafe-119-_jpg.rf.672aa2017377b983d01ca9b670ca0e91.jpg', 'truckSafe-181-_jpg.rf.ae0f5f346b1f48b2b28ee67320750f15.jpg', 'truckSafe-8-_jpg.rf.d9de58166f3fc2a27cb9fb70a14f5832.jpg', 'truckUnsafe-36-_jpg.rf.5ed78ccd6dbdaea3187841d0607dfa21.jpg', 'truckSafe-27-_jpg.rf.f6bb32d9ec7a45b384a973ed17090b73.jpg', 'truckUnsafe-44-_jpg.rf.e8dcce3e0ce162ab3c6bd8e4f7eaf5d0.jpg', 'truckUnsafe-85-_jpg.rf.1979c63ca47ceaa8533ccd773fa63214.jpg', 'truckSafe-130-_jpg.rf.b05ad19ba0d647e19a9553b76e2e598d.jpg', 'truckUnsafe-92-_jpg.rf.c0bc0344d40691da7b49ded98c8614f0.jpg', 'truckSafe-8-_webp.rf.818239fc2a975dfd17f7733341b86f4b.jpg', 'truckUnsafe-127-_jpg.rf.2ad3044b802aed3efcc8b7bed371d70b.jpg', 'truckSafe-186-_jpg.rf.3029317de36f7e0a368b2f6d2501b8c3.jpg', 'truckUnsafe-153-_jpg.rf.7019a8000dd041fc1d6f8d7fe1d8c3c5.jpg', 'truckSafe-65-_jpg.rf.a69a4e6a56701f9244892aab2c69aa5e.jpg', 'truckSafe-52-_jpg.rf.c96d653bbff8df77bff82f9c6bb777f1.jpg', 'truckSafe-131-_jpg.rf.0c1f5be05d28edf0b553b4bd8e7158e6.jpg', 'truckUnsafe-3-_webp.rf.c674dff6c062307d014c597698bd36b1.jpg', 'truckSafe-51-_jpg.rf.baf14a2919019fdd13b08ed4168a998a.jpg', 'truckUnsafe-77-_jpg.rf.d99926b669c1222f8f3b5668ce9d9e7d.jpg', 'truckSafe-163-_jpg.rf.7ea0aaebcbc3bbdaf27c8c330f9ccff6.jpg', 'truckUnsafe-158-_jpg.rf.2b4978d0d7c31782247ebe044b20260a.jpg', 'truckSafe-75-_jpg.rf.ceaee62937387e471cc55fd76a7da4a7.jpg', 'truckSafe-29-_jpg.rf.5160f33be04265d12ced4efc8c13d202.jpg', 'truckSafe-19-_jpg.rf.46fc002423b49f86c30c68bd93b4bea2.jpg', 'truckSafe-26-_jpg.rf.54985421d17fd0a04e187bfabd621ec2.jpg', 'truckUnsafe-52-_jpg.rf.3623d6ef83d47e24e61ba611ddb7f227.jpg', 'truckUnsafe-179-_jpg.rf.b0f40909cd7092c96c6a7c0bf22f9a2f.jpg', 'truckSafe-24-_jpg.rf.aae58a00b277e292aa6ebe6a709720c1.jpg', 'truckSafe-88-_jpg.rf.d2c3cadff6f47d1b726af8259132ac61.jpg', 'truckUnsafe-126-_jpg.rf.0963a81c618f910e514f53fe9626da38.jpg', 'truckUnsafe-104-_jpg.rf.050ff76c09644829e14b6eaf10728d0b.jpg', 'truckUnsafe-21-_jpg.rf.3a060c0d3b53704bd922d3d5a465fc51.jpg', 'truckSafe-73-_jpg.rf.ea654fb67610c7379028bc356f6eaddb.jpg', 'truckSafe-188-_jpg.rf.2be4f01e41c66cbe250bb1f83a45d77c.jpg', 'truckSafe-41-_jpg.rf.e44c8a931548a9d8f7450a43206cc764.jpg', 'truckSafe-127-_jpg.rf.5ff817b579d8b08ae45a1ad16bdcc9ff.jpg', 'truckUnsafe-23-_jpg.rf.08192ec5d1763d69b44c5560e29f6a0f.jpg', 'truckUnsafe-55-_jpg.rf.b79f03ebb0feb12365e3be86a65b1ea8.jpg', 'truckUnsafe-47-_jpg.rf.4c887da5c757e9d5dfb2dfe245bcb1a1.jpg', 'truckUnsafe-97-_jpg.rf.062184dd5c4304a13ca457c25763a725.jpg', 'truckUnsafe-93-_jpg.rf.5185ffc4e708caf4450d6b50e637f53a.jpg', 'truckSafe-157-_jpg.rf.03771434f2554193b0fbbf476595752d.jpg', 'truckUnsafe-98-_jpg.rf.36d3f90d22be913327d5235c3e197c97.jpg', 'truckUnsafe-24-_jpg.rf.d6a5b64e5c3084a2620a5314a1afa9de.jpg', 'truckSafe-170-_jpg.rf.558b296a74c9d5f63e3161f2af8cdbc4.jpg', 'truckUnsafe-114-_jpg.rf.ea89c824deef9111ba53df1e3b358fc6.jpg', 'truckUnsafe-171-_jpg.rf.133139ceb1d2b99742873b8d2040ef23.jpg', 'truckUnsafe-67-_jpg.rf.7ec55fc8a43a9050056ccbaf8be3cca8.jpg', 'truckSafe-38-_jpg.rf.4c37a79e117bf366f74f4f89292e31bb.jpg', 'truckUnsafe-144-_jpg.rf.b1c7133774e19ce0a8037921cfe39646.jpg', 'truckUnsafe-41-_jpg.rf.24167e65d8001dbcec8995250a395740.jpg', 'truckSafe-106-_jpg.rf.afebeba9f3fa4fcb9573a2c57a524c14.jpg', 'truckUnsafe-43-_jpg.rf.d220226c519c51d7096fdfd4180366fc.jpg', 'truckUnsafe-174-_jpg.rf.64a1c55d4e43522b493ad19a3cdb5aef.jpg', 'truckUnsafe-49-_jpg.rf.8c40763992228923494b3b88975c49f3.jpg', 'truckUnsafe-17-_jpg.rf.18bfbfc041068b2c06bcebd48947e43d.jpg', 'truckSafe-5-_jpg.rf.aff0ea198b86b963b316a564abab4c7e.jpg', 'truckUnsafe-48-_jpg.rf.628681efd2a41b93d3ead8d21cb04a3e.jpg', 'truckUnsafe-54-_jpg.rf.e33ad9e7f96d185270f38398fd442c85.jpg', 'truckSafe-100-_jpg.rf.9e19d4331f246a02ab81257bd84e783f.jpg', 'truckUnsafe-70-_jpg.rf.89c6768acd46c5e4e9993f162711c003.jpg', 'truckSafe-185-_jpg.rf.d48bb5c402f48d46dd5ba2b0645bb54f.jpg', 'truckUnsafe-199-_jpg.rf.e611104b8652ea2ae107c2f05e7742a2.jpg', 'truckUnsafe-11-_jpg.rf.f0591c57aff5fbe9d96c17b33776d576.jpg', 'truckUnsafe-6-_jpg.rf.6f20ea65517ee4f1903be31c540a051d.jpg', 'truckSafe-3-_webp.rf.ae5b6a2e4cb447cb3eab97d973e07cd4.jpg', 'truckSafe-161-_jpg.rf.828c830a5836bf8462a29b72e3efafa9.jpg', 'truckSafe-182-_jpg.rf.3f86479d7c598edb7be0688883e6112c.jpg', 'truckUnsafe-156-_jpg.rf.a523b36fd59be5a14650b9e277c29ed8.jpg', 'truckSafe-133-_jpg.rf.b20fbe959839064c74b03254da73feb4.jpg', 'truckSafe-175-_jpg.rf.1a4bed9b5c41c9ed36b8f15e76375fe4.jpg', 'truckUnsafe-84-_jpg.rf.ac6fbdce8437790b644771cdd9268fdf.jpg', 'truckSafe-55-_jpg.rf.19368f9adcd1e5572a0e87f661319e3c.jpg', 'truckSafe-66-_jpg.rf.7ccbfd700c1761ff0d798562b9baf2fe.jpg', 'truckUnsafe-91-_jpg.rf.c3e0b0affa5fd5ebd79d8ac93d570d29.jpg', 'truckSafe-7-_webp.rf.316dd3dfb61fcd5424dc6d4ef6ff1d9d.jpg', 'truckSafe-122-_jpg.rf.f1327dc90994c69507adcca64208b256.jpg', 'truckUnsafe-64-_jpg.rf.9af27645e83ae4264c19d830674615c4.jpg', 'truckSafe-121-_jpg.rf.72059f36368fbde4d53c677f44065d30.jpg', 'truckUnsafe-94-_jpg.rf.24071cadc7abd856bbd85a2d237f6379.jpg', 'truckSafe-6-_jpg.rf.39935e83fc92779a8e98b096aa62e92e.jpg', 'truckUnsafe-192-_jpg.rf.95ff1338941d488d2a2bdd0ff7fa338e.jpg', 'truckUnsafe-142-_jpg.rf.135d9a65569b31f916bf316e7f151995.jpg', 'truckSafe-169-_jpg.rf.87a8c50d791b94ad907ba8afe7e9d8df.jpg', 'truckUnsafe-165-_jpg.rf.af4191e9509fc76ef6e4298f268a8e8c.jpg', 'truckUnsafe-96-_jpg.rf.b9afb7a259a409e4df70272513e5ec19.jpg', 'truckSafe-2-_jpg.rf.c8ceda007375848fe364d9fb95acad70.jpg', 'truckUnsafe-87-_jpg.rf.9580a4daf820d1aba9855dd51536e723.jpg', 'truckSafe-46-_jpg.rf.a38f8664325ecbc40a7e0a4729822caf.jpg', 'truckUnsafe-90-_jpg.rf.a4a481d609241403cd343a0497ad1c42.jpg', 'truckSafe-36-_jpg.rf.72afcabf92688c96fcfc2e27b93d99b0.jpg', 'truckSafe-84-_jpg.rf.f12222161bc89e93ba1841fe28e6f916.jpg', 'truckSafe-174-_jpg.rf.1f16f54d94437219b942177e4bc0db67.jpg', 'truckSafe-101-_jpg.rf.40dc566e05fce20d06038b0ce3fe0ecf.jpg', 'truckUnsafe-187-_jpg.rf.0092685b8e65fdc2418250d631105749.jpg', 'truckSafe-7-_jpg.rf.9f79036ac23a53c55a71bbb0dbf8e110.jpg', 'truckUnsafe-157-_jpg.rf.ff8c910ef1bce7bf7eb7695e7e7e2395.jpg', 'truckUnsafe-46-_jpg.rf.ec77d5373365cb6d9cece324197f39b5.jpg', 'truckSafe-158-_jpg.rf.71b34d0312441967683f0c2c1080ac55.jpg', 'truckUnsafe-18-_jpg.rf.a7df91f4a191d49cfb52d239273b9f3b.jpg', 'truckSafe-194-_jpg.rf.c1b179c6fcd4c0375fd6982e2f1e5a35.jpg', 'truckUnsafe-79-_jpg.rf.388712ca31e3c2d9d8fc7134301a9eb5.jpg', 'truckSafe-152-_jpg.rf.850d0eb944f345859e327fe227e2ffd5.jpg', 'truckUnsafe-12-_jpg.rf.8b882577219da7c2a85456de63e31ab1.jpg', 'truckSafe-180-_jpg.rf.30727e4a07fe21babf5174c1ece296d7.jpg', 'truckUnsafe-100-_jpg.rf.a34138f18e494e1e2c36308440b028f3.jpg', 'truckUnsafe-111-_jpg.rf.818a3eca4a4cfe36948dcd81c135feb3.jpg', 'truckSafe-179-_jpg.rf.828db7378c3802d44a575e1c44c7fa44.jpg', 'truckUnsafe-201-_jpg.rf.271a90b8de5b6026cde49a6ecb489fa9.jpg', 'truckUnsafe-105-_jpg.rf.d15a089e8cce2eb10cc9215e3b6223c0.jpg', 'truckSafe-63-_jpg.rf.b031c09836d48b8946504ba68c000c20.jpg', 'truckUnsafe-99-_jpg.rf.5e7d063e8fc2fac56cf4b98005c2a03f.jpg', 'truckUnsafe-103-_jpg.rf.081710aa938cdb75b0e684e3a75f8818.jpg', 'truckSafe-156-_jpg.rf.73c9dd229c5811afa503f92c0ff20424.jpg', 'truckSafe-183-_jpg.rf.eb2027b28dc77eff4c56e3025cbc2653.jpg', 'truckUnsafe-71-_jpg.rf.a604f89995f09072cd98b2bf5eff7664.jpg', 'truckUnsafe-151-_jpg.rf.97c5beaf7e14f92154ac594ac10d8ff5.jpg', 'truckUnsafe-191-_jpg.rf.95da8d25b2f7d4ba407f2da19072ce62.jpg', 'truckUnsafe-198-_jpg.rf.5b99f84247c9fc028d168c9ad4f66ec8.jpg', 'truckSafe-176-_jpg.rf.9193d80068af2848538b375a9ab821e0.jpg', 'truckUnsafe-162-_jpg.rf.1480f80ee9853b314e72c38864c81a3a.jpg', 'truckUnsafe-86-_jpg.rf.d08bb0606fa8a30be8f46a43c11bdc4a.jpg', 'truckUnsafe-195-_jpg.rf.be3e1027ed833259568cb52c78835a86.jpg', 'truckSafe-184-_jpg.rf.43245b2aa995677c14325289e8fd1c5b.jpg', 'truckUnsafe-2-_jpg.rf.d8a033434f0bb74a6c3cbadfb339258b.jpg', 'truckUnsafe-19-_jpg.rf.b579de1709760947387a5d0a192f1801.jpg', 'truckUnsafe-53-_jpg.rf.4176b3f2f3881205d285678f6ea71a17.jpg', 'truckUnsafe-137-_jpg.rf.c66321af191a7352b5bd5c07633058dd.jpg', 'truckSafe-83-_jpg.rf.0eae6bde531e04b9a4a3bc3e07edd74c.jpg', 'truckSafe-153-_jpg.rf.84535b234f54b4a811a9fc3630edd9d8.jpg', 'truckUnsafe-140-_jpg.rf.701dad58d868e3c83399023aa1a00765.jpg', 'truckSafe-104-_jpg.rf.d1c05a12238769503eb3d4d131fcbc9e.jpg', 'truckUnsafe-58-_jpg.rf.e3118e0dfb34d6f897354e8661ead737.jpg', 'truckSafe-87-_jpg.rf.fa900c0d3f7e4c9a7667da0a03362765.jpg', 'truckUnsafe-74-_jpg.rf.9460147f5eccc8238ec6794cdcf2c0bd.jpg', 'truckSafe-172-_jpg.rf.c5e19482d2a3f12c8fbde13b42143ea0.jpg', 'truckSafe-13-_jpg.rf.05c4b7d4f35bec861860469c6a4897cd.jpg', 'truckUnsafe-130-_jpg.rf.ec114dd5101a292503a6b96cb7918d3d.jpg', 'truckSafe-90-_jpg.rf.59afc6a6554cbdbff546165bbd1d74a2.jpg', 'truckUnsafe-9-_jpg.rf.2f24b55c2436a8c830f604176db8e7bf.jpg', 'truckSafe-37-_jpg.rf.ad92d1aeb3b1c3f08910c1822ce15259.jpg', 'truckUnsafe-154-_jpg.rf.572c40fd0390108e8d7cdbe0f94abaca.jpg', 'truckUnsafe-1-_webp.rf.83d14e4ad5e491866b755205e8ed1a84.jpg', 'truckSafe-99-_jpg.rf.d6fdd3a54a647a87cf406cd166efa175.jpg', 'truckUnsafe-134-_jpg.rf.2b64fd235fee765bffcabc349adcb811.jpg', 'truckUnsafe-123-_jpg.rf.e5151da7b48173b50cbd5fdcf270460b.jpg', 'truckUnsafe-161-_jpg.rf.f6946afcf62d422f2f859b13ceda1e33.jpg', 'truckSafe-81-_jpg.rf.9d94f916ffef79645f1d090afd393983.jpg', 'truckUnsafe-133-_jpg.rf.3e694cb4e574c24f9569949ce61b39ba.jpg', 'truckSafe-189-_jpg.rf.32e5ded963fdd6234812af2d3cc0f32a.jpg', 'truckSafe-191-_jpg.rf.6af016a6449226cdc9e500cc560b58b7.jpg', 'truckUnsafe-128-_jpg.rf.c2068342dd5985126c6d1c29251fd7b2.jpg', 'truckSafe-53-_jpg.rf.debc72c4c389011f41a94a44f69b8c42.jpg', 'truckUnsafe-125-_jpg.rf.36ce28ef8b9d9f0ce74335e7797e30ea.jpg', 'truckSafe-103-_jpg.rf.565796b94aa3b9a650551dad5638dd62.jpg', 'truckUnsafe-180-_jpg.rf.bfe074781a458e41ca7bb78ba589d84a.jpg', 'truckSafe-86-_jpg.rf.4ecc7aac28ace212e96d7556749f8c6a.jpg', 'truckSafe-76-_jpg.rf.3982d37199d66499fad89e44876df5a8.jpg', 'truckUnsafe-181-_jpg.rf.7b902744c06191b3af371ed6c22b956c.jpg', 'truckSafe-58-_jpg.rf.6d7c4081902acae3464a2f527ce1a2b9.jpg', 'truckSafe-33-_jpg.rf.a577ed75fe979e9284eb7c9bf6382e70.jpg', 'truckSafe-107-_jpg.rf.f2d11dc26ef91ab7a39a1b5493c38d30.jpg', 'truckSafe-1-_jpg.rf.c0c82f13fe564a63107241f74f0a9e3d.jpg', 'truckUnsafe-107-_jpg.rf.babd965d5bc5ae3bffe9c00b0251d2b3.jpg', 'truckSafe-146-_jpg.rf.650600abdaf8b7e0a4cf3c3d6d0482f7.jpg', 'truckSafe-16-_jpg.rf.dd82e0008d1ca38f19a9015d9c1d904d.jpg', 'truckUnsafe-110-_jpg.rf.d0de5edb5dc0dbe52c7bfe1c0908fa0f.jpg', 'truckUnsafe-15-_jpg.rf.cc5e44fad855c19f9a65b507108efb7f.jpg', 'truckUnsafe-112-_jpg.rf.36928ae59d723b0cd298b437976aceac.jpg', 'truckSafe-20-_jpg.rf.63a052b5b2e3bdff34aea844bcf94289.jpg', 'truckUnsafe-89-_jpg.rf.a30019a08e5a6bf467932a216fc3f622.jpg', 'truckSafe-40-_jpg.rf.77ec15e052b68d0980bc2e69a14bc27d.jpg', 'truckSafe-118-_jpg.rf.95d277cb7ae1ef4c3680def0797626ae.jpg', 'truckSafe-80-_jpg.rf.983ac820621eb75446c37990b8739001.jpg', 'truckSafe-144-_jpg.rf.d2c5ddc26f85b6a3e8e7b8f26b97aceb.jpg', 'truckUnsafe-186-_jpg.rf.8a207c733043e64808014528e310d89d.jpg', 'truckUnsafe-122-_jpg.rf.fe6f35b0c37aad5ac962f4189221e0e6.jpg', 'truckSafe-123-_jpg.rf.321f8187a8b607a05a0563f15a93e79f.jpg', 'truckUnsafe-106-_jpg.rf.be4ef15ef9ab2e2ce369596c4ae9ad31.jpg', 'truckUnsafe-182-_jpg.rf.f07e5fb6163965727248439dc676c083.jpg', 'truckUnsafe-14-_jpg.rf.67b4993e6194f1ef8a2c62bc7bce52f4.jpg', 'truckSafe-129-_jpg.rf.2cd2112916d88bbdbca81f6c3856b746.jpg', 'truckUnsafe-185-_jpg.rf.6b2b9d024f6a35b55747d508137260a2.jpg', 'truckSafe-120-_jpg.rf.2958ef3918c556ea5f01f9b4c23e0da1.jpg', 'truckUnsafe-132-_jpg.rf.6d7b5d74ebbf2a0768884828e594f6b6.jpg', 'truckSafe-6-_webp.rf.a9c48df2e9f7efcfeb3af3d5fb3122c6.jpg', 'truckSafe-49-_jpg.rf.a8aca39572383374c24ba49242b55343.jpg', 'truckSafe-102-_jpg.rf.7753b5b01a12f79096edaaf996ddbc6d.jpg', 'truckUnsafe-5-_jpg.rf.24986876cec09070289cb7de410a4ee2.jpg', 'truckUnsafe-75-_jpg.rf.712a44e5bdde0cfdfcc23136f784634d.jpg', 'truckSafe-57-_jpg.rf.884e1204d78fb5ee677f683e8f3d38e0.jpg', 'truckSafe-124-_jpg.rf.f720158b37bb2809f3cd9a829ec31d22.jpg', 'truckSafe-62-_jpg.rf.10ab461a5116c816bd0b5c8421eddbb5.jpg', 'truckUnsafe-124-_jpg.rf.4954171d878114814dc9cfff995f6c39.jpg', 'truckSafe-22-_jpg.rf.60c5bc122fa965840765ef1d6d1da2cd.jpg', 'truckSafe-154-_jpg.rf.edc7e1f835cc24ddd2564d8124cc3bb4.jpg', 'truckSafe-67-_jpg.rf.5d0c427be03cb6cdf0f45411361b502e.jpg', 'truckSafe-91-_jpg.rf.a512bb3290043c460cbfae7113c731dd.jpg', 'truckSafe-42-_jpg.rf.c0ec22b050facbe92d64613902252be2.jpg', 'truckUnsafe-183-_jpg.rf.94580a51961ee1289b34017fc3b3d38b.jpg', 'truckUnsafe-35-_jpg.rf.3a140385e21297bebed441337548e8ee.jpg', 'truckUnsafe-113-_jpg.rf.c43fd8559af2e24e15eb90a471d17cef.jpg', 'truckSafe-113-_jpg.rf.d0971be4fc07d6bb7bf33d2d3e2d3064.jpg', 'truckUnsafe-204-_jpg.rf.9493419167eded56b9c2b7872d6e3274.jpg', 'truckUnsafe-59-_jpg.rf.b4f8d8cc6de495bacbf4708459963268.jpg', 'truckSafe-54-_jpg.rf.c2cbdf3ac479b6c927a15d3edc21e937.jpg', 'truckUnsafe-203-_jpg.rf.6f6dea5a207484867bca181d9966c81d.jpg', 'truckUnsafe-202-_jpg.rf.2b3c827bff086afefc1cca63d7b9b85f.jpg', 'truckUnsafe-175-_jpg.rf.49a83ef0f4582ed7b87c59820f9be87b.jpg', 'truckUnsafe-72-_jpg.rf.4c92a307d646a5b26e27f41a155dfd00.jpg', 'truckUnsafe-188-_jpg.rf.a84bf42690643feedcdb4319c728e825.jpg', 'truckUnsafe-131-_jpg.rf.90e0129fad38b50648f76f78418b3777.jpg', 'truckUnsafe-68-_jpg.rf.372701c8570c60192cff003497247da2.jpg', 'truckUnsafe-82-_jpg.rf.151a75d74a7794160fc510beb7bcd783.jpg', 'truckUnsafe-150-_jpg.rf.8e208de58316788754df7138624cd61e.jpg', 'truckUnsafe-178-_jpg.rf.3ac5c179e6c2c7a5cbea74e3478b2ad9.jpg', 'truckUnsafe-152-_jpg.rf.67bf16f806a35feb3259467eb213f943.jpg', 'truckUnsafe-61-_jpg.rf.5c973d77f5a568d776af6f3748548a03.jpg', 'truckUnsafe-138-_jpg.rf.e479f04b6e8a66ff08b8e29013e9265c.jpg', 'truckUnsafe-57-_jpg.rf.191df9af0064b469773cd3a0f70f9229.jpg', 'truckUnsafe-116-_jpg.rf.94578f4ab6415c9fab8865a52798a70c.jpg', 'truckUnsafe-88-_jpg.rf.41c380a1b28aae61d17167da35c4de92.jpg', 'truckUnsafe-69-_jpg.rf.c5e1dacbed4cbc6f1afc31a7e4a551f5.jpg', 'truckSafe-112-_jpg.rf.8e746d7a4b4c801f8a1fc3f4f6477831.jpg', 'truckUnsafe-117-_jpg.rf.1cacd5f61ce9c6f7cec1af7584b6f903.jpg', 'truckSafe-68-_jpg.rf.78e006c591eb26eb8fcea748bfc9c3c3.jpg', 'truckSafe-190-_jpg.rf.2d8164dbd7e5cb51d97b32a13c379271.jpg', 'truckUnsafe-25-_jpg.rf.cceb12e6df5d0f35bb90c4901596f4e2.jpg', 'truckSafe-9-_jpg.rf.71ca59ea13482b86225a8c1ba67a0429.jpg', 'truckUnsafe-40-_jpg.rf.cd6b17f9f652fd121f3a6a8e15eb654d.jpg', 'truckSafe-136-_jpg.rf.48c908bff9c2a624086b514e59b13ffc.jpg', 'truckUnsafe-83-_jpg.rf.0f9f64f54581b130c50cf540a820b669.jpg', 'truckUnsafe-31-_jpg.rf.4aec26eb17cc019b648d0a92ea9944ad.jpg', 'truckSafe-94-_jpg.rf.5cfdfe8842439763d21e889ba7a6ad61.jpg', 'truckUnsafe-168-_jpg.rf.c2f29f4b129a5454bb08d0a3d7b99544.jpg', 'truckUnsafe-76-_jpg.rf.7a77c02aa45758eaa911b02dbaed9e32.jpg', 'truckUnsafe-149-_jpg.rf.b126d7b734bb20b91b2bfd099818e2c9.jpg', 'truckSafe-128-_jpg.rf.26dd042ba0eeb0597a577d37d652a5c4.jpg', 'truckUnsafe-155-_jpg.rf.8a89427c494956e2ac55eba8460549ae.jpg', 'truckUnsafe-60-_jpg.rf.1f9281128a662c54690eb6a8688fa98f.jpg', 'truckSafe-31-_jpg.rf.fb1ede627cf672ef46b645be0fc2ece4.jpg', 'truckSafe-72-_jpg.rf.b8faea27058cf7ff0ae7ea0e753ce3f1.jpg', 'truckSafe-47-_jpg.rf.570b68b697d7ecfde919c8749f0e8002.jpg', 'truckUnsafe-22-_jpg.rf.180e2087cfd23ffcb8b04c313232b3ab.jpg', 'truckSafe-108-_jpg.rf.de0ca0561bf258860f5da9855e02e579.jpg', 'truckSafe-138-_jpg.rf.ae0c93eb4ec6b9e06a93eaf873e92b9d.jpg', 'truckUnsafe-30-_jpg.rf.942a0c3d8244420914d6f474881897fa.jpg', 'truckSafe-5-_webp.rf.3fe61e0f0022da182f4de2f6bc3b2a58.jpg', 'truckUnsafe-136-_jpg.rf.26c2544af1f9957a430f6aee3ba13c03.jpg', 'truckSafe-95-_jpg.rf.b32a883f8955c72646c481c4b7e27cb5.jpg', 'truckUnsafe-62-_jpg.rf.f60b501172d3815263491816aa762aa3.jpg', 'truckSafe-192-_jpg.rf.3a61ee8db33f0c3c97f0357962031b60.jpg', 'truckSafe-143-_jpg.rf.213bd5fb3648e0e1e5df30ad98b9e604.jpg', 'truckSafe-155-_jpg.rf.8c619944eb41eb510aa6220ff4c601d2.jpg', 'truckUnsafe-65-_jpg.rf.fcfd34958d37e457faf6cff33c941b90.jpg', 'truckSafe-135-_jpg.rf.41737206cdde399534d020bbd47fe49c.jpg', 'truckSafe-98-_jpg.rf.7880d9b1a5a7d5b5473cdba5666fed42.jpg', 'truckSafe-56-_jpg.rf.883d44105fcd428f68f2531965e11162.jpg', 'truckSafe-171-_jpg.rf.070592362ff110c0b78c454193c41984.jpg', 'truckSafe-132-_jpg.rf.82a84a11180f4f71daf9dd68b0b456f7.jpg', 'truckSafe-79-_jpg.rf.abd81ee14a2be9c4006363b5ce38b7af.jpg', 'truckSafe-89-_jpg.rf.67f18184939682f9c5007fbecbdb9dbe.jpg', 'truckSafe-74-_jpg.rf.364e4fab490377d740d1707a08e8081a.jpg', 'truckSafe-151-_jpg.rf.a14cc0a9d69663026441bc5b19feee5e.jpg', 'truckSafe-147-_jpg.rf.7fa6061c05d1c0500eb4c4f8649820eb.jpg', 'truckUnsafe-32-_jpg.rf.3866133a1dc169fb9df501a57309836b.jpg', 'truckSafe-126-_jpg.rf.00ba94172a9babfe1285305d9eebcc11.jpg', 'truckSafe-160-_jpg.rf.5af3e09ff37e810b861fc84cbec1117c.jpg', 'truckUnsafe-73-_jpg.rf.b6f977a418e50b3600b5a97f202844dd.jpg', 'truckSafe-11-_jpg.rf.6c71b100f961c69d222281d938ebc131.jpg', 'truckUnsafe-120-_jpg.rf.a106ba89cd477455d34e6a0f5ca463e8.jpg', 'truckUnsafe-50-_jpg.rf.8a7f0178404d39dc81e94945a9c00771.jpg', 'truckUnsafe-63-_jpg.rf.1802510f46c25d9f829cd2ba6eeb58d1.jpg', 'truckUnsafe-139-_jpg.rf.98829e0a502800918cf4573505c29747.jpg', 'truckSafe-4-_jpg.rf.ca4ce57a90296c7c370030b00fc6d5f1.jpg', 'truckSafe-125-_jpg.rf.19dfe38428480391f3e4222801185034.jpg', 'truckSafe-149-_jpg.rf.aa650f61662aff9a25bd905d171b7154.jpg', 'truckUnsafe-184-_jpg.rf.860b75d169ff3565604fd17e13ba3a67.jpg'], 'val': ['truckUnsafe-1-_jpg.rf.66c30a4c2d51092237f994a461307404.jpg', 'truckSafe-150-_jpg.rf.1ed50e629f3d2e845d7cd23d4eb96335.jpg', 'truckUnsafe-176-_jpg.rf.9d134c36da2b927a9a28a3f99a819446.jpg', 'truckUnsafe-66-_jpg.rf.27776d206953c93e79c8934473e8565a.jpg', 'truckUnsafe-194-_jpg.rf.07119965499d07a9e671f515db2959a5.jpg', 'truckSafe-96-_jpg.rf.181271efd70cc303a7f94f9e1a01bdb6.jpg', 'truckSafe-30-_jpg.rf.e5ca2dcd251c8c9395023416c350df6a.jpg', 'truckUnsafe-56-_jpg.rf.dced56bae27edc9d6b1c458653cdd2de.jpg', 'truckSafe-28-_jpg.rf.f7d74f66420f4fe61a6f47ed7e8afd04.jpg', 'truckSafe-21-_jpg.rf.b599ac3334760a3f82d388857814f39e.jpg', 'truckUnsafe-115-_jpg.rf.9877e3d97ac61c51d93ec8ea7233c80b.jpg', 'truckSafe-148-_jpg.rf.fb5880a0ed45d36edaf38617112e89bf.jpg', 'truckSafe-97-_jpg.rf.d0098bd00a7779e4865439140a6380c1.jpg', 'truckUnsafe-147-_jpg.rf.a245cb4334e0df8c14e6c9d25d9252f0.jpg', 'truckSafe-162-_jpg.rf.1061926c8dbf08afb1a4d7eeab58b4c1.jpg', 'truckUnsafe-28-_jpg.rf.f03baff0ec0f1661ad9b4dba594a741b.jpg', 'truckSafe-105-_jpg.rf.9867433ae340497a1dff7c7799dbbc4f.jpg', 'truckUnsafe-163-_jpg.rf.85d049ab8af8b2c86310d5ee9002537d.jpg', 'truckSafe-109-_jpg.rf.769b22963f7c9ef3f89d7b69489d2d44.jpg', 'truckUnsafe-166-_jpg.rf.54016f8d1644b1eae8a6b6f69d590534.jpg', 'truckSafe-145-_jpg.rf.14db44130e285bc48832b4a268154942.jpg', 'truckUnsafe-121-_jpg.rf.ada93505365462e8720693f545f24aa6.jpg', 'truckSafe-193-_jpg.rf.0cc0ef8c2b93513d8de1f5a9bbffc43c.jpg', 'truckUnsafe-146-_jpg.rf.1b1dcb3b4b35776ab87be6bbfecfa8d8.jpg', 'truckUnsafe-27-_jpg.rf.b67e4bd8a18953b44880689e93a9f75d.jpg', 'truckUnsafe-37-_jpg.rf.0e48a6d560ca471a29e53aea79a73062.jpg', 'truckSafe-61-_jpg.rf.e31c686802cfbac5d44558ee1365a2e0.jpg', 'truckUnsafe-118-_jpg.rf.b71532e64d2ae7e38092c83aaedc44ea.jpg', 'truckUnsafe-4-_jpg.rf.ea6a0a852f9a431b0e6b9b74cbf2cf99.jpg', 'truckUnsafe-167-_jpg.rf.326f6ceac9547b80d1cf6513b2e299d8.jpg', 'truckSafe-141-_jpg.rf.1d1663f3393709d1ac68a7bab2a5dcc5.jpg', 'truckSafe-4-_webp.rf.1398d3d92f30abc2102efab6d45f8e7d.jpg', 'truckSafe-166-_jpg.rf.10f57ff7822fac232f026f0d1997be81.jpg', 'truckSafe-114-_jpg.rf.9467fcc3e3d833851f7afc2acd05adca.jpg', 'truckUnsafe-173-_jpg.rf.35096135cdaadfea73dc3dd3a145fc08.jpg', 'truckUnsafe-42-_jpg.rf.6b6e9da1327d7357a6ecd0a70191a597.jpg', 'truckSafe-164-_jpg.rf.fa9aa8d5bb4229652b71f190d1f4d25f.jpg', 'truckUnsafe-108-_jpg.rf.2ccd21321b603b761d1d0ff0882eaa90.jpg', 'truckUnsafe-26-_jpg.rf.7320e4935f5ddfb7ff8f313bc620b5a8.jpg', 'truckSafe-82-_jpg.rf.0c32f3892190cf4557c83d707b16e74b.jpg', 'truckUnsafe-172-_jpg.rf.9205866b23df7f774bf50554f3a669d4.jpg', 'truckSafe-173-_jpg.rf.014f1884ca80ad770ca0bfba19fdca49.jpg', 'truckSafe-17-_jpg.rf.3b732e87d7e125fc8b227f6578adf59e.jpg', 'truckUnsafe-135-_jpg.rf.ebb702987d717323caf9023bc74dc21e.jpg', 'truckUnsafe-141-_jpg.rf.825ba1db5e1614f0a7a1f30845d02b5a.jpg', 'truckUnsafe-13-_jpg.rf.e315ed30edd8462b3103071d935b5636.jpg', 'truckUnsafe-102-_jpg.rf.ba3b895eec73c56576c420107c71fc30.jpg', 'truckUnsafe-78-_jpg.rf.18ef332f3ab814b6cc78f0dc1a94fc55.jpg', 'truckSafe-64-_jpg.rf.133330c5158fc2e55a7dc2d7feca10b5.jpg', 'truckUnsafe-164-_jpg.rf.4bf99daa9fbd4b994d966286be2f7b78.jpg', 'truckSafe-116-_jpg.rf.4eec97da079f01e7f44e22b2203fee89.jpg', 'truckUnsafe-51-_jpg.rf.b76605bff08c32482c482cab3d355357.jpg', 'truckSafe-10-_jpg.rf.8a9ccf6b54dad10d7d6008315ac72650.jpg', 'truckSafe-3-_jpg.rf.75b6061d720508ec78e98e6ecb6fc39a.jpg', 'truckSafe-43-_jpg.rf.8e4d9cdcb79e28695037e6f147092b1b.jpg', 'truckSafe-137-_jpg.rf.9f2180c1d5d9004edfda5aa71550d460.jpg', 'truckSafe-140-_jpg.rf.f9c34aa7675f6de5982107f8edeb86c3.jpg', 'truckSafe-71-_jpg.rf.3fc30fe5f23685a0dd8cd5ee87886689.jpg', 'truckUnsafe-2-_webp.rf.d0dc6a759c3d04ca8d49451c36c021f3.jpg', 'truckSafe-78-_jpg.rf.81d818a87769c8d811532f8f2bd923a2.jpg', 'truckUnsafe-39-_jpg.rf.a9da71bff29f03d2151a906de369388d.jpg', 'truckSafe-93-_jpg.rf.2ae480e95b8b2cc7fc43609f9cc4a303.jpg', 'truckSafe-187-_jpg.rf.7eb0d289edadf67321837721ea1c6e5d.jpg', 'truckUnsafe-101-_jpg.rf.eef663035a8721e04c556ce2ed207eba.jpg', 'truckUnsafe-189-_jpg.rf.8e57c964c07ff8549187692fc9c1b022.jpg', 'truckSafe-165-_jpg.rf.cad213dc04105fca4c50c3ce1ecba6ca.jpg', 'truckSafe-23-_jpg.rf.24301a8f573a7207760e2bf6517c61f4.jpg', 'truckSafe-159-_jpg.rf.81a0274667bc6f3acfa11c4552f819fd.jpg', 'truckUnsafe-190-_jpg.rf.6cb0f5bd29831a7842202d0723627bdf.jpg', 'truckSafe-50-_jpg.rf.e05140a6f2e41a4552cb81c87f22f3e1.jpg', 'truckSafe-168-_jpg.rf.2c7646b81b3efb9ceed8bf2b997bad03.jpg', 'truckUnsafe-8-_jpg.rf.a27f93be283dca606b152c96bbdd40e3.jpg', 'truckUnsafe-20-_jpg.rf.b478bd1204714b4287d62ef7877a45b4.jpg', 'truckSafe-1-_png.rf.c7e9cd572c91cb21b8c1d4ac101ad79a.jpg', 'truckUnsafe-200-_jpg.rf.1a13ce61158e8958b223c822802e3080.jpg', 'truckSafe-14-_jpg.rf.e0567568b5c62e5978a66a8aed08c3d1.jpg', 'truckSafe-77-_jpg.rf.6e11dd3de52a99335b1eccbfde3346f1.jpg', 'truckSafe-1-_webp.rf.41ca3c859c10f738185f9c8e603ff11a.jpg', 'truckSafe-115-_jpg.rf.ce07d75065483e165228016805608b58.jpg', 'truckSafe-92-_jpg.rf.8512ce0b4cbe793ab5fc77c7f344e471.jpg', 'truckSafe-18-_jpg.rf.c35a172942205125cdfaf669f878d799.jpg']}}

print("Dictionary split K-Fold berhasil dimuat.")

In [ ]:
# @title 5. Fungsi Helper Persiapan Dataset & Konversi ke format YOLO
def move_files(source_path: str, destination_path: str, extension: list):
    if os.path.isdir(source_path):
        os.makedirs(destination_path, exist_ok=True)
        print(f"Memindahkan file dari folder {source_path} ke {destination_path}...")
        for filename in os.listdir(source_path):
            if filename.lower().endswith(tuple(extension)):
                shutil.move(os.path.join(source_path, filename), os.path.join(destination_path, filename))
        print("Pemindahan file selesai.")
    elif os.path.isfile(source_path):
        destination_dir = os.path.dirname(destination_path)
        os.makedirs(destination_dir, exist_ok=True)
        print(f"Memindahkan file {source_path} ke {destination_path}...")
        if source_path.lower().endswith(tuple(extension)):
            shutil.move(source_path, destination_path)
            print("Pemindahan file selesai.")

def create_labels(json_path, img_dir, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    with open(json_path, 'r') as f:
        data = json.load(f)

    images_info = {img['id']: img for img in data['images']}
    cat_mapping = {1: 1, 2: 0}  # 1: HazardousTrucks -> class 1, 2: NoHazardous -> class 0

    for img_id, img_info in images_info.items():
        base_name = os.path.splitext(img_info['file_name'])[0]
        open(os.path.join(save_dir, f"{base_name}.txt"), 'w').close()

    for ann in data['annotations']:
        img_id = ann['image_id']
        if img_id not in images_info:
            continue
        if ann['category_id'] not in cat_mapping:
            continue

        img_info = images_info[img_id]
        w_img, h_img = img_info['width'], img_info['height']

        x_min, y_min, w_bbox, h_bbox = ann['bbox']
        x_center = (x_min + w_bbox / 2) / w_img
        y_center = (y_min + h_bbox / 2) / h_img
        w_yolo = w_bbox / w_img
        h_yolo = h_bbox / h_img

        class_id = cat_mapping[ann['category_id']]
        base_name = os.path.splitext(img_info['file_name'])[0]
        with open(os.path.join(save_dir, f"{base_name}.txt"), 'a') as f_out:
            f_out.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w_yolo:.6f} {h_yolo:.6f}\n")
    print(f"Label format YOLO berhasil dibuat di {save_dir}")

def prepareKFold(image_dir, fold=5):
    # Tulis train_fold_X.txt dan val_fold_X.txt berdasarkan list nama file
    for f in range(fold):
        train_file_path = f'train_fold_{f}.txt'
        val_file_path = f'val_fold_{f}.txt'

        with open(train_file_path, 'w') as tf:
            for fn in RECONSTRUCTED_FOLDS[f]['train']:
                tf.write(os.path.abspath(os.path.join(image_dir, fn)) + '\n')

        with open(val_file_path, 'w') as vf:
            for fn in RECONSTRUCTED_FOLDS[f]['val']:
                vf.write(os.path.abspath(os.path.join(image_dir, fn)) + '\n')
    print("File data split fold berhasil dibuat.")

def createYAMLKFOLD(destination_dir: str, fold=5):
    for f in range(fold):
        data_yaml = {
            "path": os.path.abspath(destination_dir),
            "train": os.path.abspath(f"train_fold_{f}.txt"),
            "val": os.path.abspath(f"val_fold_{f}.txt"),
            "nc": 2,
            "names": {
                0: "NoHazardous",
                1: "HazardousTrucks"
            }
        }
        file_name = f"data_fold_{f}.yaml"
        with open(file_name, "w") as yaml_f:
            yaml.dump(data_yaml, yaml_f, sort_keys=False, default_flow_style=False)
        print(f"Berhasil membuat {file_name}")

def run_datasets_preparation(dataset_dir: str):
    move_files(f"{dataset_dir}/train", f"{dataset_dir}/train/images", [".jpg", ".png", ".avif", ".jpeg"])
    move_files(f"{dataset_dir}/train/_annotations.coco.json", f"{dataset_dir}/train/labels/_annotations.coco.json", [".json"])
    prepareKFold(image_dir=f"{dataset_dir}/train/images")
    createYAMLKFOLD(destination_dir=dataset_dir, fold=5)
    create_labels(f"{dataset_dir}/train/labels/_annotations.coco.json", f"{dataset_dir}/train/images", f"{dataset_dir}/train/labels")

run_datasets_preparation(dataset_dir)

In [ ]:
# @title 6. Visualisasi Confusion Matrix & Muat Baseline Referensi
def get_best_model_path(model, algo_save_dir, fold):
    import glob
    # 1. Coba dari model.trainer.best jika ada
    if hasattr(model, 'trainer') and model.trainer is not None and hasattr(model.trainer, 'best') and model.trainer.best:
        path = str(model.trainer.best)
        if os.path.exists(path):
            return path

    # 2. Coba manual path fold
    manual_path = os.path.join(algo_save_dir, f'fold_{fold}', 'weights', 'best.pt')
    if os.path.exists(manual_path):
        return manual_path

    # 3. Coba last.pt di fold
    last_path = os.path.join(algo_save_dir, f'fold_{fold}', 'weights', 'last.pt')
    if os.path.exists(last_path):
        print(f'Warning: best.pt tidak ditemukan, menggunakan last.pt: {last_path}')
        return last_path

    # 4. Cari file .pt apa saja di direktori fold
    fold_dir = os.path.join(algo_save_dir, f'fold_{fold}')
    pt_files = glob.glob(f'{fold_dir}/**/*.pt', recursive=True)
    if pt_files:
        print(f'Warning: Menggunakan file bobot alternatif: {pt_files[0]}')
        return pt_files[0]

    # 5. Cari di direktori runs/
    runs_files = glob.glob('runs/**/*.pt', recursive=True)
    if runs_files:
        print(f'Warning: Menggunakan file bobot dari runs/: {runs_files[0]}')
        return runs_files[0]

    # Jika tidak ada sama sekali, print isi direktori untuk debug
    print(f'Error: Tidak ada file .pt ditemukan di {fold_dir} atau runs/')
    if os.path.exists(fold_dir):
        print(f'Isi folder {fold_dir}:', os.listdir(fold_dir))
    raise FileNotFoundError(f'Tidak dapat menemukan file bobot model untuk fold {fold}')

def plot_confusion_matrix(model_path, data_yaml, save_dir, fold):
    if "rtdetr" in MODEL_NAME.lower():
        model = RTDETR(model_path)
    else:
        model = YOLO(model_path)

    metrics = model.val(data=data_yaml, save_json=True)
    cm = metrics.confusion_matrix.matrix
    class_names = list(model.names.values()) + ["background"]

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt=".0f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix Fold {fold}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"confusion_matrix_fold_{fold}.png"), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    return metrics

# Data Baseline Rata-rata dari Eksperimen Referensi (Fold 0-4)
REFERENCE_BASELINES = {
    "yolov8n.pt":  {"P": 0.8510, "R": 0.8504, "mAP50": 0.8962, "mAP50-95": 0.7800},
    "yolov9t.pt":  {"P": 0.8266, "R": 0.8450, "mAP50": 0.8890, "mAP50-95": 0.7912},
    "yolov10n.pt": {"P": 0.7916, "R": 0.7742, "mAP50": 0.8248, "mAP50-95": 0.7192},
    "yolo11n.pt":  {"P": 0.8648, "R": 0.8146, "mAP50": 0.8816, "mAP50-95": 0.7688},
    "rtdetr-l.pt": {"P": 0.8742, "R": 0.8118, "mAP50": 0.8830, "mAP50-95": 0.7974},
    "yolo26n.pt":  {"P": 0.0, "R": 0.0, "mAP50": 0.0, "mAP50-95": 0.0} # Membutuhkan training baseline
}

baseline_results = {}
if MODEL_NAME == "yolo26n.pt":
    print("Running Baseline Training for YOLOv26 since it was interrupted in baseline run...")
    baseline_folds = []
    os.makedirs(f"{OUTPUT_DIR}/yolo26_baseline", exist_ok=True)

    for fold in range(5):
        model = YOLO(MODEL_NAME)
        results = model.train(
            data=f"data_fold_{fold}.yaml",
            epochs=FINAL_EPOCHS,
            imgsz=640,
            batch=4,
            name=f"fold_{fold}",
            project=f"{OUTPUT_DIR}/yolo26_baseline",
            exist_ok=True,
            patience=15,
            seed=42,
            device=DEVICE
        )
        # Plot Confusion Matrix & Ambil metrik evaluasi secara bersamaan
        model_path = get_best_model_path(model, f"{OUTPUT_DIR}/yolo26_baseline", fold)
        val_metrics = plot_confusion_matrix(model_path, f"data_fold_{fold}.yaml", f"{OUTPUT_DIR}/yolo26_baseline", fold)
        baseline_folds.append({
            "P": val_metrics.results_dict["metrics/precision(B)"],
            "R": val_metrics.results_dict["metrics/recall(B)"],
            "mAP50": val_metrics.results_dict["metrics/mAP50(B)"],
            "mAP50-95": val_metrics.results_dict["metrics/mAP50-95(B)"]
        })

    # Hitung rata-rata
    baseline_results = {
        "P": np.mean([f["P"] for f in baseline_folds]),
        "R": np.mean([f["R"] for f in baseline_folds]),
        "mAP50": np.mean([f["mAP50"] for f in baseline_folds]),
        "mAP50-95": np.mean([f["mAP50-95"] for f in baseline_folds])
    }
    print(f"YOLOv26 Baseline Average: {baseline_results}")
else:
    baseline_results = REFERENCE_BASELINES.get(MODEL_NAME, {"P": 0.0, "R": 0.0, "mAP50": 0.0, "mAP50-95": 0.0})
    print(f"Reused Baseline Metrics for {MODEL_NAME}: {baseline_results}")

In [ ]:
# @title 7. Fungsi Evaluasi HPO & Pemetaan Hyperparameter
def vector_to_hyperparams(x):
    return {
        "lr0": float(x[0]),
        "lrf": float(x[1]),
        "momentum": float(x[2]),
        "weight_decay": float(x[3]),
        "mosaic": float(x[4]),
        "box": float(x[5])
    }

def train_and_evaluate(hyperparams, folds=None, epochs=None):
    if folds is None:
        folds = HPO_FOLDS
    if epochs is None:
        epochs = HPO_EPOCHS

    # Simpan status acak numpy dan python sebelum training agar proses HPO tidak terpengaruh oleh reset seed internal YOLO/RT-DETR
    numpy_state = np.random.get_state()
    python_state = random.getstate()

    fitness_scores = []
    temp_project = "./hpo_temp"
    temp_name = "eval"

    for fold in folds:
        if "rtdetr" in MODEL_NAME.lower():
            model = RTDETR(MODEL_NAME)
        else:
            model = YOLO(MODEL_NAME)

        try:
            results = model.train(
                data=f"data_fold_{fold}.yaml",
                epochs=epochs,
                imgsz=640,
                batch=4,
                patience=15,
                seed=42,
                device=DEVICE,
                project=temp_project,
                name=temp_name,
                exist_ok=True,
                verbose=False,
                plots=False,
                save=False,        # Tidak menyimpan file bobot untuk menghemat memori disk
                # Masukkan hyperparameter hasil optimasi
                lr0=hyperparams["lr0"],
                lrf=hyperparams["lrf"],
                momentum=hyperparams["momentum"],
                weight_decay=hyperparams["weight_decay"],
                mosaic=hyperparams["mosaic"],
                box=hyperparams["box"]
            )

            # Ambil nilai mAP50 sebagai fitness value
            fitness = results.results_dict.get('metrics/mAP50(B)', 0.0)
            fitness_scores.append(fitness)

        except Exception as e:
            print(f"[ERROR] Error selama training Fold {fold} dengan parameter {hyperparams}: {e}")
            fitness_scores.append(0.0)

        # Bersihkan folder sementara agar disk penyimpanan tidak penuh
        if os.path.exists(temp_project):
            shutil.rmtree(temp_project, ignore_errors=True)

    # Pulihkan status acak numpy dan python
    np.random.set_state(numpy_state)
    random.setstate(python_state)

    return float(np.mean(fitness_scores))

In [ ]:
# @title 8. Inisialisasi Batas Parameter (Bounds)
# Batas bawah (lb) dan batas atas (ub) untuk [lr0, lrf, momentum, weight_decay, mosaic, box]
lb = np.array([0.005, 0.005, 0.85, 0.0001, 0.0, 5.0])
ub = np.array([0.025, 0.05, 0.98, 0.01, 0.0, 10.0])
bounds = (lb, ub)

In [ ]:
# @title 15. Implementasi Ant Colony Optimization (ACO)
def run_aco(num_agents, num_iterations, bounds):
    lb, ub = bounds
    num_dim = len(lb)

    positions = np.random.uniform(lb, ub, (num_agents, num_dim))
    fitness = np.zeros(num_agents)

    print("Initializing ACO Colony...")
    for i in range(num_agents):
        params = vector_to_hyperparams(positions[i])
        fitness[i] = train_and_evaluate(params)
        print(f"Ant {i+1}/{num_agents} Initial Fitness: {fitness[i]:.4f}")

    best_idx = np.argmax(fitness)
    best_pos = np.copy(positions[best_idx])
    best_fit = fitness[best_idx]

    pheromones = np.copy(fitness)
    history = []

    for it in range(num_iterations):
        print(f"\n--- ACO Iteration {it+1}/{num_iterations} ---")
        pheromones *= 0.9
        prob_sum = np.sum(pheromones)
        if prob_sum > 0:
            prob = pheromones / prob_sum
        else:
            prob = np.ones(num_agents) / num_agents

        new_positions = np.zeros_like(positions)
        for i in range(num_agents):
            parent_idx = np.random.choice(num_agents, p=prob)
            parent_pos = positions[parent_idx]

            mutation_scale = 0.1 * (ub - lb) * (1 - it / num_iterations)
            new_positions[i] = parent_pos + np.random.normal(0, mutation_scale)
            new_positions[i] = np.clip(new_positions[i], lb, ub)

            params = vector_to_hyperparams(new_positions[i])
            fit = train_and_evaluate(params)
            fitness[i] = fit
            pheromones[i] += fit

            if fit > best_fit:
                best_fit = fit
                best_pos = np.copy(new_positions[i])

        positions = np.copy(new_positions)
        print(f"\n--- ACO Iteration {it+1} Global Best mAP50: {best_fit:.4f} ---")
        history.append({"iteration": it+1, "best_fitness": best_fit})

    return best_pos, best_fit, history

In [ ]:
# @title 16. Eksekusi HPO ACO
print("==================== RUNNING ACO ====================")
aco_pos, aco_fit, aco_hist = run_aco(NUM_AGENTS, NUM_ITERATIONS, bounds)

In [ ]:
# @title 17. Inisialisasi Parameter Model Final Terbaik (ACO)
best_params_dict = {
    "ACO": vector_to_hyperparams(aco_pos)
}
final_results = {}

In [ ]:
# @title 21. Training Final Model ACO (5-Fold K-Fold x 50 Epochs)
algo_name = "ACO"
params = best_params_dict[algo_name]
print(f"\n==================== TRAINING FINAL MODEL: {algo_name} ====================")
print(f"Optimized Hyperparameters: {params}")
algo_folds = []

algo_save_dir = f"{OUTPUT_DIR}/{algo_name}"
os.makedirs(algo_save_dir, exist_ok=True)

for fold in range(5):
    if "rtdetr" in MODEL_NAME.lower():
        model = RTDETR(MODEL_NAME)
    else:
        model = YOLO(MODEL_NAME)

    results = model.train(
        data=f"data_fold_{fold}.yaml",
        epochs=FINAL_EPOCHS,
        imgsz=640,
        batch=4,
        name=f"fold_{fold}",
        project=algo_save_dir,
        exist_ok=True,
        patience=15,
        seed=42,
        device=DEVICE,
        # Parameter optimasi
        lr0=params["lr0"],
        lrf=params["lrf"],
        momentum=params["momentum"],
        weight_decay=params["weight_decay"],
        mosaic=params["mosaic"],
        box=params["box"]
    )
    # Matriks Konfusi & Validasi akhir
    model_path = get_best_model_path(model, algo_save_dir, fold)
    val_metrics = plot_confusion_matrix(model_path, f"data_fold_{fold}.yaml", algo_save_dir, fold)
    algo_folds.append({
        "P": val_metrics.results_dict["metrics/precision(B)"],
        "R": val_metrics.results_dict["metrics/recall(B)"],
        "mAP50": val_metrics.results_dict["metrics/mAP50(B)"],
        "mAP50-95": val_metrics.results_dict["metrics/mAP50-95(B)"]
    })

final_results[algo_name] = {
    "P": np.mean([f["P"] for f in algo_folds]),
    "R": np.mean([f["R"] for f in algo_folds]),
    "mAP50": np.mean([f["mAP50"] for f in algo_folds]),
    "mAP50-95": np.mean([f["mAP50-95"] for f in algo_folds]),
    "folds": algo_folds
}
print(f"Summary {algo_name}: P={final_results[algo_name]['P']:.4f}, R={final_results[algo_name]['R']:.4f}, mAP50={final_results[algo_name]['mAP50']:.4f}, mAP50-95={final_results[algo_name]['mAP50-95']:.4f}")

In [ ]:
# @title 22. Kompilasi Hasil, CSV & Grafik Analisis (ACO)
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Simpan Riwayat Konvergensi HPO ke CSV
history_rows = []
for it in range(NUM_ITERATIONS):
    history_rows.append({
        "Iteration": it+1,
        "ACO": aco_hist[it]["best_fitness"]
    })
df_hist = pd.DataFrame(history_rows)
df_hist.to_csv(f"{OUTPUT_DIR}/convergence_history.csv", index=False)
print("[INFO] convergence_history.csv disimpan.")

# 2. Simpan Metrik Akhir Perbandingan ke CSV
metrics_rows = [
    {
        "Optimizer": "Baseline",
        "lr0": 0.01, "lrf": 0.01, "momentum": 0.937, "weight_decay": 0.0005, "mosaic": 1.0, "box": 7.5,
        "Precision": baseline_results["P"],
        "Recall": baseline_results["R"],
        "mAP50": baseline_results["mAP50"],
        "mAP50-95": baseline_results["mAP50-95"]
    }
]

p = best_params_dict["ACO"]
res = final_results["ACO"]
metrics_rows.append({
    "Optimizer": "ACO",
    "lr0": p["lr0"], "lrf": p["lrf"], "momentum": p["momentum"], "weight_decay": p["weight_decay"], "mosaic": p["mosaic"], "box": p["box"],
    "Precision": res["P"],
    "Recall": res["R"],
    "mAP50": res["mAP50"],
    "mAP50-95": res["mAP50-95"]
})

df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv(f"{OUTPUT_DIR}/final_metrics_comparison.csv", index=False)
print("[INFO] final_metrics_comparison.csv disimpan.")

# 3. Plot Kurva Konvergensi HPO
plt.figure(figsize=(8, 5))
plt.plot(df_hist["Iteration"], df_hist["ACO"], marker='o', label="ACO")
plt.title(f"HPO Convergence Curve - {MODEL_NAME} (ACO)")
plt.xlabel("Iteration")
plt.ylabel("Fitness (mAP50 on Fold 0)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/convergence_plot.png", dpi=300, bbox_inches='tight')
plt.show()

# 4. Boxplot Distribusi mAP50 pada 5 Fold
box_data = {}
box_data["ACO"] = [f["mAP50"] for f in final_results["ACO"]["folds"]]
df_box = pd.DataFrame(box_data)
plt.figure(figsize=(5, 5))
sns.boxplot(data=df_box)
plt.axhline(y=baseline_results["mAP50"], color='r', linestyle='--', label=f"Baseline Mean ({baseline_results['mAP50']:.4f})")
plt.title(f"mAP50 Distribution over 5 Folds - {MODEL_NAME} (ACO)")
plt.ylabel("mAP50")
plt.xlabel("HPO Optimizer")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/mAP50_folds_boxplot.png", dpi=300, bbox_inches='tight')
plt.show()

# 5. Diagram Batang Perbandingan mAP50 Rata-rata
methods = ["Baseline", "ACO"]
map50_vals = [baseline_results["mAP50"], final_results["ACO"]["mAP50"]]
plt.figure(figsize=(5, 5))
colors = ['#cccccc', '#1f77b4']
bars = plt.bar(methods, map50_vals, color=colors, edgecolor='black', alpha=0.8)
plt.title(f"Mean mAP50 Comparison - {MODEL_NAME} (ACO)")
plt.ylabel("Mean mAP50 (5 Folds)")
plt.ylim(0.0, 1.0)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.01, f"{height:.4f}", ha='center', va='bottom', fontsize=9)
plt.grid(True, linestyle='--', alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/mean_mAP50_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print("Eksperimen Selesai! Semua file tersimpan di direktori: ", OUTPUT_DIR)